# imports

In [1]:
import pandas as pd
import json
import requests
import numpy as np
from bs4 import BeautifulSoup
from bs4 import XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
import re
import io, os
import csv
import boto3
from botocore.exceptions import ClientError
import openai
from openai import OpenAI
import base64
from PIL import Image
import anthropic
from anthropic import Anthropic

In [2]:
data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover'
ctd_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/CTD_data'
pubtator_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data'
input_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockInput'
output_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockOutput'
figures_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockFigures'
supplement_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockSupplement'
pubtatorFileName = '0-400pub_med_ids.json'

In [3]:
# save data file
pm_pmc_dict = {'22228119': '3349910', '26714306': '4695093', '28285367': '5387039', '28593498': '5773665', '15154616': '2275409', '17118201': '1676006', '18299717': '2243244', '19114014': '2631511', '25859307': '4387225', '15953866': '2782200', '22110480': '3205718', '22571318': '3485091', '22808131': '3392256', '23724058': '3665791', '24169358': '3833203', '24403256': '3892387', '24413757': '3907846', '24664296': '3963982', '26039251': '4454644', '9118903': '1469763', '17524151': '1890552', '22747577': '3489685', '25351418': '4255228', '25361045': '4245613', '25923732': '4414544'}
pmids = list(pm_pmc_dict.keys())

#remove from data pmids:  36864359

url = "https://www.ncbi.nlm.nih.gov/research/pubtator3-api/publications/export/biocjson?pmids="

for id in range (0,len(pmids)):
    temp = str(pmids[id])
    url+=(temp+',')

url = url[0:-1] + "&full=true"
print(url)
# Save to a file only once, uncomment for replication
# response = requests.get(url)

# response.raise_for_status()

# data = response.json()

# with open(f"{pubtator_data_path}/{pubtatorFileName}", "w") as f:
#     json.dump(data, f, indent=4)

# print(f"Saved {pubtatorFileName}")

https://www.ncbi.nlm.nih.gov/research/pubtator3-api/publications/export/biocjson?pmids=22228119,26714306,28285367,28593498,15154616,17118201,18299717,19114014,25859307,15953866,22110480,22571318,22808131,23724058,24169358,24403256,24413757,24664296,26039251,9118903,17524151,22747577,25351418,25361045,25923732&full=true


# Process pubtator, figures, and supplemental data

## figures

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (PMC figure/table downloader)"}

def download_pmc_figures_and_tables(pmcid, pmid, outdir=figures_path):
    """
    Downloads all figures and tables (images) from a PMC article.
    Extracts correct captions for figures and tables.
    Saves metadata CSV with pmid, number, caption, filename, and type.
    """
    os.makedirs(outdir, exist_ok=True)
    csv_path = os.path.join(outdir, f"{pmcid}_figures_metadata.csv")

    url = f"https://www.ncbi.nlm.nih.gov/pmc/articles/{pmcid}/"
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    with open(csv_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=["pmid", "number", "caption", "filename", "orginal_filename", "type"])
        writer.writeheader()

        count = 1

        # --- Figures ---
        fig_containers = soup.select("div.fig-wrap, div.fig, figure")
        for i, fig in enumerate(fig_containers, 1):
            caption_tag = fig.find("div", class_="fig-caption") or fig.find("figcaption")
            caption_text = " ".join(caption_tag.stripped_strings) if caption_tag else ""

            imgs = fig.find_all("img")
            if not imgs:
                continue

            for j, img in enumerate(imgs, 1):
                img_url = img.get("src")
                if not img_url:
                    continue
                if img_url.startswith("/"):
                    img_url = "https://www.ncbi.nlm.nih.gov" + img_url

                base_filename = os.path.basename(img_url)
                filename = f"{pmid}_figure{i}_img{j}_{base_filename}"
                filepath = os.path.join(outdir, filename)

                try:
                    r2 = requests.get(img_url, headers=HEADERS)
                    r2.raise_for_status()
                    with open(filepath, "wb") as f:
                        f.write(r2.content)
                    print(f"Downloaded figure: {filename}")
                except requests.HTTPError:
                    print(f"❌ Failed to download {img_url}")
                    continue

                writer.writerow({
                    "pmid": pmid,
                    "number": count,
                    "caption": caption_text,
                    "filename": filename,
                    "orginal_filename": base_filename,
                    "type": "figure"
                })
                count += 1

        # --- Tables (including images) ---
        table_sections = soup.select("section.tw.xbox")
        for section in table_sections:
            table_id = section.get("id", "")
            if not table_id.startswith("T"):  # Only tables
                continue

            # Extract caption from div.caption.p
            caption_div = section.find("div", class_="caption")
            caption_text = ""
            if caption_div:
                caption_text = " ".join(caption_div.stripped_strings)

            # Find table images
            img_box = section.select_one("p.img-box img")
            if not img_box:
                continue

            img_url = img_box.get("src")
            if not img_url:
                continue
            if img_url.startswith("/"):
                img_url = "https://www.ncbi.nlm.nih.gov" + img_url

            base_filename = os.path.basename(img_url)
            filename = f"{pmid}_table_{count}_{base_filename}"
            filepath = os.path.join(outdir, filename)

            try:
                r2 = requests.get(img_url, headers=HEADERS)
                r2.raise_for_status()
                with open(filepath, "wb") as f:
                    f.write(r2.content)
                print(f"Downloaded table image: {filename}")
            except requests.HTTPError:
                print(f"❌ Failed to download {img_url}")
                continue

            writer.writerow({
                "pmid": pmid,
                "number": count,
                "caption": caption_text,
                "filename": filename,
                "orginal_filename": base_filename,
                "type": "table"
            })
            count += 1

    print(f"✅ Done. Total items downloaded: {count-1}")

def get_figuredata(pmcid, pmid, figures_path):
    download_pmc_figures_and_tables(pmcid, pmid, figures_path)

In [ ]:
PREFIX_LEN = 15

def normalize_table_caption(text):
    if not text or pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"\s+", "", text)
    text = re.sub(r"[^a-z0-9]", "", text)
    return text

def match_pubtator_fig_annotations(csv_dir, pubtator_file):
    """
    Match PMC FIGURES by filename and TABLES by caption prefix.
    Does NOT add helper columns to the output DataFrame.
    """

    # 1️⃣ Load CSVs
    all_csvs = [os.path.join(csv_dir, f) for f in os.listdir(csv_dir) if f.endswith(".csv")]
    df = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
    print(f"Loaded {len(df)} rows from {len(all_csvs)} CSVs")

    # 2️⃣ Load PubTator JSON
    with open(pubtator_file, "r", encoding="utf-8") as f:
        pubtator_data = json.load(f)

    # 3️⃣ Build mappings
    fig_map = {}    # pmid -> filename -> annotations
    table_map = {}  # pmid -> caption_prefix -> annotations

    for entry in pubtator_data.get("PubTator3", []):
        pmid = entry["_id"].split("|")[0]

        fig_map.setdefault(pmid, {})
        table_map.setdefault(pmid, {})

        for passage in entry.get("passages", []):
            infons = passage.get("infons", {})
            section_type = infons.get("section_type", "")
            ptype = infons.get("type", "").lower()
            annotations = passage.get("annotations", [])

            # FIGURES
            if section_type == "FIG" and ptype in {"fig_caption", "fig_title_caption"}:
                file_name = infons.get("file")
                if file_name:
                    fig_map[pmid][file_name] = annotations

            # TABLES
            elif section_type == "TABLE" and ptype in {"table_caption", "table_title_caption"}:
                caption_text = passage.get("text", "")
                norm = normalize_table_caption(caption_text)
                if norm:
                    table_map[pmid][norm[:PREFIX_LEN]] = annotations

    # 4️⃣ Extract entities
    def extract_entities(row):
        pmid = str(row["pmid"])
        orginal_filename = row.get("orginal_filename", "")
        caption = row.get("caption", "")

        genes = {}
        chemicals = {}

        annotations = []

        # FIG match
        if pmid in fig_map and orginal_filename in fig_map[pmid]:
            annotations = fig_map[pmid][orginal_filename]

        # TABLE match
        else:
            cap_norm = normalize_table_caption(caption)
            cap_prefix = cap_norm[:PREFIX_LEN]
            if pmid in table_map and cap_prefix in table_map[pmid]:
                annotations = table_map[pmid][cap_prefix]

        for ann in annotations:
            infons = ann.get("infons", {})
            if not infons.get("valid", True):
                continue

            ann_type = infons.get("type", "").lower()
            name = ann.get("text") or infons.get("name")
            identifier = (
                infons.get("normalized_id")
                or infons.get("identifier")
                or infons.get("accession")
            )

            if not name or not identifier:
                continue

            identifier = str(identifier).upper()

            if ann_type == "gene":
                genes[name] = identifier
            elif ann_type == "chemical":
                chemicals[name] = identifier

        genes_str = "\n".join(f"{k}: {v}" for k, v in genes.items()) if genes else None
        chemicals_str = "\n".join(f"{k}: {v}" for k, v in chemicals.items()) if chemicals else None

        return pd.Series([genes_str, chemicals_str])

    df[["gene_map", "chemical_map"]] = df.apply(extract_entities, axis=1)

    print("✅ FIG (filename) and TABLE (caption prefix) annotations matched successfully.")
    return df


## supplement

In [1]:
xls = pd.ExcelFile(f'{supplement_path}/22747577_S2.xls')

supp_xls_2_a = pd.read_excel(f'{supplement_path}/22747577_S2.xls', sheet_name=xls.sheet_names[0])
supp_xls_2_a_name = xls.sheet_names[0]

supp_xls_2_b = pd.read_excel(f'{supplement_path}/22747577_S2.xls', sheet_name=xls.sheet_names[1])
supp_xls_2_b_name = xls.sheet_names[1]

supp_cap_2 = "Genes with a time-dependent, continuous increase or decrease in mRNA expression level during masitinib treatment."

def filter_supp(df):
    # keep columns that have "gene" case insensitive
    # keep columns that have "fold change" or "fold-change" case insensitive
    # if multiple fold change cols exist, keep those that have only numerical values

    gene_cols = [
        c for c in df.columns
        if re.search(r"gene", c, re.IGNORECASE)
    ]

    fc_candidates = [
        c for c in df.columns
        if re.search(r"fold[\s\-]?change", c, re.IGNORECASE)
    ]

    # --- 4. subset dataframe ---
    keep_cols = gene_cols + fc_candidates
    df = df.loc[:, keep_cols]
    return df

supp_xls_2_a = filter_supp(supp_xls_2_a)
supp_xls_2_b = filter_supp(supp_xls_2_b)
supp_xls_2_a.head()

NameError: name 'pd' is not defined

## pubtator

In [7]:
def process_pubtator_json(json_path, download_missing_tables=True):
    """Process PubTator JSON and return a DataFrame with deduplicated chem/gene maps."""
    with open(json_path, "r") as f:
        data = json.load(f)

    rows = []
    counter = 0

    for entry in data["PubTator3"]:
        pmid, pmcid = entry["_id"].split("|")

        # Step 1: Group TABLE passages
        group_passages = {}
        for passage in entry.get("passages", []):
            section_type = passage["infons"].get("section_type", "")
            ptype = passage["infons"].get("type", "")
            pid = passage["infons"].get("id", "")
            text = passage.get("text", "")
            annotations = passage.get("annotations", [])
            xml_data = passage["infons"].get("xml", "") if section_type=="TABLE" and ptype=="table" else ""

            if section_type in ["TABLE"] and pid:
                if pid not in group_passages:
                    group_passages[pid] = {"caption": "", "body": "", "annotations": [], "type": section_type, "xml": ""}

                if ptype.endswith("caption"):
                    group_passages[pid]["caption"] = text
                elif ptype in ["table", "fig"]:
                    group_passages[pid]["body"] = text
                    if xml_data:
                        group_passages[pid]["xml"] = xml_data

                group_passages[pid]["annotations"].extend(annotations)

        # Step 2: Process grouped TABLE
        for pid, pdata in group_passages.items():
            chem_map = {}
            gene_map = {}

            for ann in pdata["annotations"]:
                info = ann["infons"]
                ann_type = info.get("type", "").lower()
                ann_text = ann.get("text")
                ann_id = info.get("normalized_id") or ("E0000" if ann_type=="chemical" else "0000")
                ann_id_str = str(ann_id)
                ann_id_norm = ann_id_str.upper()

                if not ann_text:
                    continue

                # Deduplicate based on normalized ID (case-insensitive)
                if ann_type == "chemical":
                    if ann_id_norm not in {str(v).upper() for v in chem_map.values()}:
                        chem_map[ann_text] = ann_id
                elif ann_type == "gene":
                    if ann_id_norm not in {str(v).upper() for v in gene_map.values()}:
                        gene_map[ann_text] = ann_id

            if chem_map or gene_map:
                row = {
                    "id": f"{pubtatorFileName}{counter}",
                    "pmid": pmid,
                    "pmcid": pmcid,
                    "section_type": pdata["type"],
                    "text": "",
                    "chem_map": json.dumps(chem_map),
                    "gene_map": json.dumps(gene_map)
                }

                if pdata["type"] == "TABLE":
                    row["table_caption"] = pdata["caption"]
                    row["table_text"] = pdata["body"]
                    row["table_xml"] = pdata["xml"]

                    # # download missing tables as figure
                    # if download_missing_tables and not row["table_text"]:
                    #     table_file = download_table_as_figure(pmcid, pid)
                    #     row["table_text"] = f"Downloaded as figure: {table_file}" if table_file else "none"

                # else:  # FIG
                    # row["fig_caption"] = pdata["caption"]
                    # row["fig_text"] = pdata["body"]

                rows.append(row)
                counter += 1

        # Step 3: Process normal passages (non-TABLE/)
        for passage in entry.get("passages", []):
            section_type = passage["infons"].get("section_type", "")
            if section_type in ["TABLE", "FIG"]:
                continue

            text = passage.get("text", "")
            annotations = passage.get("annotations", [])

            chem_map = {}
            gene_map = {}

            for ann in annotations:
                info = ann["infons"]
                ann_type = info.get("type", "").lower()
                ann_text = ann.get("text")
                ann_id = info.get("normalized_id") or ("E0000" if ann_type=="chemical" else "0000")
                ann_id_str = str(ann_id)
                ann_id_norm = ann_id_str.upper()

                if not ann_text:
                    continue

                if ann_type == "chemical":
                    if ann_id_norm not in {str(v).upper() for v in chem_map.values()}:
                        chem_map[ann_text] = ann_id
                elif ann_type == "gene":
                    if ann_id_norm not in {str(v).upper() for v in gene_map.values()}:
                        gene_map[ann_text] = ann_id

            if chem_map and gene_map:
                rows.append({
                    "id": f"{pubtatorFileName}{counter}",
                    "pmid": pmid,
                    "pmcid": pmcid,
                    "section_type": section_type,
                    "text": text,
                    "table_caption": "",
                    "table_text": "",
                    "table_xml": "",
                    "chem_map": json.dumps(chem_map),
                    "gene_map": json.dumps(gene_map)
                })
                counter += 1

    df = pd.DataFrame(rows)
    df = df.sort_values(by="pmid").reset_index(drop=True)
    return df


## calls

In [55]:
# download figures
for pmid, pmcid in pm_pmc_dict.items():
    try:
        print(f"Processing PMID: {pmid}, PMCID: {pmcid}")
        get_figuredata(pmcid, pmid, figures_path)
    except Exception as e:
        print(f"❌ Failed for PMID {pmid}, PMCID {pmcid}: {e}")

df = match_pubtator_fig_annotations(figures_path, f'{pubtator_data_path}/{pubtatorFileName}')
df.to_csv(f'{figures_path}/figures.csv')

Processing PMID: 22228119, PMCID: 3349910
Downloaded figure: 22228119_figure1_img1_emm-44-281-g001.jpg
Downloaded figure: 22228119_figure2_img1_emm-44-281-g002.jpg
Downloaded figure: 22228119_figure3_img1_emm-44-281-g003.jpg
Downloaded figure: 22228119_figure4_img1_emm-44-281-g004.jpg
Downloaded table image: 22228119_table_5_emm-44-281-i001.jpg
Downloaded table image: 22228119_table_6_emm-44-281-i002.jpg
✅ Done. Total items downloaded: 6
Processing PMID: 26714306, PMCID: 4695093
Downloaded figure: 26714306_figure1_img1_pone.0145363.g001.jpg
✅ Done. Total items downloaded: 1
Processing PMID: 28285367, PMCID: 5387039
Downloaded figure: 28285367_figure1_img1_403_2017_1729_Fig1_HTML.jpg
Downloaded figure: 28285367_figure2_img1_403_2017_1729_Fig2_HTML.jpg
Downloaded figure: 28285367_figure3_img1_403_2017_1729_Fig3_HTML.jpg
Downloaded figure: 28285367_figure4_img1_403_2017_1729_Fig4_HTML.jpg
Downloaded figure: 28285367_figure5_img1_403_2017_1729_Fig5_HTML.jpg
Downloaded figure: 28285367_figu

,pmid,number,caption,filename,orginal_filename,type,gene_map,chemical_map
0,17524151,1,Cytotoxicity of SM on SAE and BTE cells . Cell...,17524151_figure1_img1_1471-2121-8-17-1.jpg,1471-2121-8-17-1.jpg,figure,NaN,SM: D009151
1,17524151,2,Effect of roxithromycin on SM cytotoxicity in ...,17524151_figure2_img1_1471-2121-8-17-2.jpg,1471-2121-8-17-2.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151
2,17524151,3,Anti-cytotoxicity effect of roxithromycin on S...,17524151_figure3_img1_1471-2121-8-17-3.jpg,1471-2121-8-17-3.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151\nCalcein A...
3,17524151,4,Inhibition by roxithromycin of cytokine releas...,17524151_figure4_img1_1471-2121-8-17-4.jpg,1471-2121-8-17-4.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575
4,17524151,5,Inhibition by roxithromycin of cytokine mRNA e...,17524151_figure5_img1_1471-2121-8-17-5.jpg,1471-2121-8-17-5.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575


In [8]:
figs = pd.read_csv(f'{figures_path}/figures.csv').drop(columns=['Unnamed: 0'], axis=1)
figs

,pmid,number,caption,filename,orginal_filename,type,gene_map,chemical_map
0,17524151,1,Cytotoxicity of SM on SAE and BTE cells . Cell...,17524151_figure1_img1_1471-2121-8-17-1.jpg,1471-2121-8-17-1.jpg,figure,NaN,SM: D009151
1,17524151,2,Effect of roxithromycin on SM cytotoxicity in ...,17524151_figure2_img1_1471-2121-8-17-2.jpg,1471-2121-8-17-2.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151
2,17524151,3,Anti-cytotoxicity effect of roxithromycin on S...,17524151_figure3_img1_1471-2121-8-17-3.jpg,1471-2121-8-17-3.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151\nCalcein A...
3,17524151,4,Inhibition by roxithromycin of cytokine releas...,17524151_figure4_img1_1471-2121-8-17-4.jpg,1471-2121-8-17-4.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575
4,17524151,5,Inhibition by roxithromycin of cytokine mRNA e...,17524151_figure5_img1_1471-2121-8-17-5.jpg,1471-2121-8-17-5.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575
...,...,...,...,...,...,...,...,...
145,22808131,3,Pituitaries were recovered from females aged f...,22808131_figure3_img1_pone.0040306.g003.jpg,pone.0040306.g003.jpg,figure,Hprt: 24465,NaN
146,22808131,4,"Testes were recovered from males of 5, 28 and ...",22808131_figure4_img1_pone.0040306.g004.jpg,pone.0040306.g004.jpg,figure,Hprt: 24465,NaN
147,22808131,5,Ovaries were recovered from females of 3 to 25...,22808131_figure5_img1_pone.0040306.g005.jpg,pone.0040306.g005.jpg,figure,Hprt: 24465,NaN
148,22808131,6,Pituitaries were recovered from females of 3 t...,22808131_figure6_img1_pone.0040306.g006.jpg,pone.0040306.g006.jpg,figure,Hprt: 24465,NaN


In [9]:
text = process_pubtator_json(f"{pubtator_data_path}/{pubtatorFileName}")
text

,id,pmid,pmcid,section_type,text,table_caption,table_text,table_xml,chem_map,gene_map
0,0-400pub_med_ids.json1,15154616,PMC2275409,ABSTRACT,"Monokine Induced by Interferon- (MIG), a CXC c...",,,,"{""LPS"": ""D008070"", ""CpG"": ""C015772"", ""zymosan""...","{""MIG"": 17329, ""Toll-Like Receptor-4"": 21898, ..."
1,0-400pub_med_ids.json12,15953866,PMC2782200,DISCUSS,Treatment with BMP-2 increases the ALP activit...,,,,"{""simvastatin"": ""D019821""}","{""BMP-2"": 650, ""ALP"": 250, ""osteocalcin"": 632}"
2,0-400pub_med_ids.json14,15953866,PMC2782200,DISCUSS,"In conclusion, this study found that the HMG-C...",,,,"{""simvastatin"": ""D019821""}","{""HMG-CoA reductase"": 3156, ""alkaline phosphat..."
3,0-400pub_med_ids.json13,15953866,PMC2782200,DISCUSS,This study also examined whether or not simvas...,,,,"{""ethanol"": ""D000431"", ""simvastatin"": ""D019821""}","{""BMP-2"": 650}"
4,0-400pub_med_ids.json11,15953866,PMC2782200,DISCUSS,Simvastatin was chosen from various HMG-CoA re...,,,,"{""simvastatin"": ""D019821"", ""cerivastatin"": ""C0...","{""HMG-CoA reductase"": 3156}"
...,...,...,...,...,...,...,...,...,...,...
423,0-400pub_med_ids.json417,28593498,PMC5773665,ABSTRACT,Benzo[a]pyrene is a known human carcinogen. As...,,,,"{""BPDE"": ""E0000"", ""Benzo[a]pyrene"": ""D001564"",...","{""p53"": 7157, ""AP-1"": 3725, ""PIG-A"": 5277}"
424,0-400pub_med_ids.json416,28593498,PMC5773665,TITLE,BPDE-induced genotoxicity: relationship betwee...,,,,"{""BPDE"": ""E0000""}","{""PIG-A"": 5277}"
425,0-400pub_med_ids.json426,28593498,PMC5773665,DISCUSS,"In this study, mutations induced by BPDE were ...",,,,"{""GPI"": ""D017261"", ""NQO"": ""E0000"", ""cyclohexim...","{""CD55"": 1604, ""CD59"": 966, ""PIG-A"": 5277, ""PI..."
426,0-400pub_med_ids.json420,28593498,PMC5773665,METHODS,"Cell treatment, sample preparation, and analys...",,,,"{""7-AAD"": ""E0000"", ""formaldehyde"": ""D005557"", ...","{""CD19"": 930, ""CD55"": 1604, ""CD59"": 966}"


# prompts

In [10]:
def fig_prompt(caption):
    systemPrompt = (
        "You are an AI assistant specialized in extracting relationships between chemicals "
        "and genes from images in scientific papers. Be precise, helpful, and honest. "
        "If a relationship is unclear or not present, state that clearly. Only use data from "
        "the image and caption; do not assume anything.\n"
        "Pay close attention to visual elements such as arrows, color coding, bar graphs, "
        "line graphs, labels, annotations, fold-change indicators, and highlighted elements, "
        "as these often indicate key relationships.\n"
        "Do not extract relationships from negative or non-significant results."
    )
    mainPrompt = f"""### Extract Specific Relationships Between Chemicals and Genes from image

#### Image Caption:
{caption}

---

### Provided Relationship List
The following relationships are predefined and must be used to describe chemical-gene interactions:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Instructions
For each extracted relationship, generate a CSV object with the following fields:
- `chemicalName`: Name of the chemical.
- `relationship`: This must be chosen from the provided list of relationships. The relationship may be prefixed by `increase`, `decrease`, or `affect`.
- `geneName`: Gene name if provided, otherwise leave blank. Gene names are longer than gene symbol.
- `geneSymbol`: Gene symbol if provided, otherwise leave blank. Gene symbols are typically all in uppercase.

#### Guidelines:
1. Extract specific relationships between these chemicals and genes using only the predefined terms from the provided relationship list. Do not use any terms or descriptions of interactions from the source text unless they exactly match terms from the list.
2. Provide only the CSV output—no additional explanations or comments.
3. Ensure that your CSV output is well-formed and follows CSV format.
4. Recheck to ensure the relationship is in the provided relationship list.
5. Use your best judgement to extract chemical-gene relationships, in some cases there might not be any. If there are no relationships, leave the csv empty.

---

### Example Output

#### Example Image Caption:
Gene regulation by Genistein.

#### Example Image:
Assume the image is passed in and you parsed the image internally.

#### Example CSV Output:
chemicalName,relationship,geneName,geneSymbol
"Genistein","Decreases expression","-","DEF2"
"Genistein","Increases expression","-","ABC1"
"Genistein","Decreases activity","tyrosine 3-monooxygenase","-"
"""
    return systemPrompt, mainPrompt


In [11]:
def fig_table_prompt(caption):
    """
    Generates a system prompt and main prompt for a table in a figure.
    Returns systemPrompt, mainPrompt.
    """
    systemPrompt = (
        "You are an AI assistant specialized in extracting relationships between chemicals "
        "and genes from XML tables in scientific papers. Be precise, helpful, and honest. "
        "If a relationship is unclear or not present, state that clearly. Only use the data "
        "from the provided table; do not assume anything.\n"
        "Do not extract relationships from negative or non-significant results."
    )
    mainPrompt = f"""### Extract Specific Relationships Between Chemicals and Genes from Table

#### Table Caption:
{caption}

---

### Provided Relationship List
The following relationships are predefined and must be used to describe chemical-gene interactions:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Instructions
For each extracted relationship, generate a CSV object with the following fields:
- `chemicalName`: Name of the chemical.
- `relationship`: This must be chosen from the provided list of relationships. The relationship may be prefixed by `increase`, `decrease`, or `affect`.
- `geneName`: Gene name if provided, otherwise leave blank. Gene names are longer than gene symbol.
- `geneSymbol`: Gene symbol if provided, otherwise leave blank. Gene symbols are typically all in uppercase.

#### Guidelines:
1. Extract specific relationships between these chemicals and genes using only the predefined terms from the provided relationship list. Do not use any terms or descriptions of interactions from the source text unless they exactly match terms from the list.
2. Provide only the CSV output—no additional explanations or comments.
3. Ensure that your CSV output is well-formed and follows CSV format.
4. Recheck to ensure the relationship is in the provided relationship list.
5. In tabular data, relationships of interest are typically increase expression or decrease expression unless otherwise specified.
6. There may not be any relationships; in this case leave the output blank.
7. Do not extract relationships from negative or non-significant results.

---

### Example Output

#### Example Table Caption:
Gene regulation by Genistein. Table contains differentially expressed genes (p<0.05).

#### Example Table:
Assume the Table is passed in as an image, and you parsed the image internally.

#### Example CSV Output:
chemicalName,relationship,geneName,geneSymbol
"Genistein","Decreases expression","-","DEF2"
"Genistein","Increases expression","-","ABC1"
"Genistein","Decreases expression","tyrosine 3-monooxygenase","-"
"""
    return systemPrompt, mainPrompt


In [12]:
def supplement_XLS_prompt(tableCSVstring, caption, sheet_name):
    """
    Returns systemPrompt, mainPrompt.
    """
    systemPrompt = (
        "You are an AI assistant specialized in extracting relationships between chemicals "
        "and genes from tables in scientific papers. Be precise, helpful, and honest. "
        "If a relationship is unclear or not present, state that clearly. Only use the data "
        "from the table and caption; do not assume anything.\n"
        "Do not extract relationships from negative or non-significant results."
    )

    mainPrompt = f"""### Extract Specific Relationships Between Chemicals and Genes from the following table in csv format with a header.

#### Table Caption:
{caption}

#### Table Sheet Name:
{sheet_name if sheet_name else "Not provided"}

#### Full Table:
{tableCSVstring}

---
### Provided Relationship List
The following relationships are predefined and must be used to describe chemical-gene interactions:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---
### Instructions
For each extracted relationship, generate a CSV object with the following fields:
- `chemicalName`: Name of the chemical.
- `relationship`: This must be chosen from the provided list of relationships. The relationship may be prefixed by `increase`, `decrease`, or `affect`.
- `geneName`: Gene name if provided, otherwise leave blank.
- `geneSymbol`: Gene symbol if provided, otherwise leave blank.

#### Guidelines:
1. Extract specific relationships between these chemicals and genes using only the predefined terms from the provided relationship list. Do not use any terms or descriptions of interactions from the source text unless they exactly match terms from the list.
2. Provide only the CSV output—no additional explanations or comments.
3. Ensure that your CSV output is well-formed and follows CSV format.
4. Recheck to ensure the relationship is in the provided relationship list.
5. Determining increase vs. decrease in expression—use ALL available context:
   - If sheet name is provided: Look for keywords like "up", "down", "increase", "decrease", "upregulated", "downregulated" in the sheet name.
   - If sheet name is not provided or unclear: Use the table caption and column headers:
     * Look for phrases like "up-regulated in [chemical] treatment" → chemical INCREASES expression
     * Look for phrases like "down-regulated in [chemical] treatment" → chemical DECREASES expression
     * Look for comparison direction in column headers
   - Fold change interpretation (when present):
     * Positive fold change typically means first condition > second condition
     * Negative fold change (or negative log2) typically means first condition < second condition
     * Always check column headers to understand what is being compared (e.g., "Control vs Treatment" or "Treatment vs Control")
   - If directionality is unclear: Use "affect expression" instead of "increase expression" or "decrease expression"
6. Do not extract relationships from negative or non-significant results. Only extract rows where a clear, significant effect is present.

---

### Example 1: With Sheet Name

#### Table Caption:
Genes with significant changes in mRNA expression levels after masitinib treatment

#### Table Sheet Name:
The control decreased by more than 1.

#### Example Table:
Probe Set ID,Gene Symbol,Description,log2 fold change
D89070cds_s_at,CR1,carbonyl reductase 1,-2.5
X07467_at,G6PD,glucose-6-phosphate dehydrogenase,-2.1

#### Example CSV Output:
chemicalName,relationship,geneName,geneSymbol
"masitinib","increase expression","carbonyl reductase 1","CR1"
"masitinib","increase expression","glucose-6-phosphate dehydrogenase","G6PD"

#### Logic: Sheet name indicates genes DOWN in control, so masitinib INCREASES expression.

---

### Example 2: Without Sheet Name (using caption)

#### Table Caption:
Genes up-regulated at least 1.5-fold in rebamipide treatment in rats

#### Table Sheet Name:
Not provided

#### Example Table:
Probe Set ID,Description,Signal intensity vehicle,Signal intensity Rebamipide,Rebamipide/vehicle
D89070cds_s_at,carbonyl reductase 1,347.5,5844.1,4.00
X07467_at,glucose-6-phosphate dehydrogenase,272.0,1660.4,3.48

#### Example CSV Output:
chemical,relationship,geneName,geneSymbol
"Rebamipide","increase expression","carbonyl reductase 1",""
"Rebamipide","increase expression","glucose-6-phosphate dehydrogenase",""

#### Logic: Caption says "up-regulated in rebamipide treatment" → rebamipide INCREASES expression.

---

### Example 3: Ambiguous Direction

#### Table Caption:
Genes with altered expression after treatment

#### Table Sheet Name:
Not provided

#### Example Table:
Gene,Change
BRCA1,altered
TP53,modified

#### Example CSV Output:
chemical,relationship,geneName,geneSymbol
"treatment compound","affect expression","BRCA1",""
"treatment compound","affect expression","TP53",""

#### Logic: No clear directional information → use "affect expression".
"""

    return systemPrompt, mainPrompt


In [54]:
def createTextPrompt(text, chem_map, gene_map):

    systemPrompt = """You are an AI assistant specialized in extracting relationships between chemicals and genes from scientific text. Be precise, accurate, and honest. Only extract relationships explicitly stated or clearly implied in the source text; if evidence is insufficient, do not extract the relationship.

Do not extract relationships from negative, null, or non-significant findings."""

    if isinstance(chem_map, str):
        chem_map = json.loads(chem_map)
    if isinstance(gene_map, str):
        gene_map = json.loads(gene_map)
    
    # Create a CSV-style list of chemicals: Name, MESH_ID
    chemicals = [f"{chem},{mesh_id}" for chem, mesh_id in chem_map.items()]
    
    # Create a CSV-style list of genes: Name, NCBI_ID
    genes = [f"{gene},{ncbi_id}" for gene, ncbi_id in gene_map.items()]
    
    # Join them into a string for the prompt
    chemicals_str = "\n".join(chemicals)
    genes_str = "\n".join(genes)
    
    mainPrompt = f"""### Task: Extract Chemical-Gene Relationships

Extract specific relationships between chemicals and genes from the given text. Format the output as CSV with one relationship per row.

#### Input Text:
{text}

#### Chemicals (Name, MESH ID):
{chemicals_str}

#### Genes (Name, NCBI ID):
{genes_str}

---
### Predefined Relationship Types

You must use ONLY the following relationship types. Do not use any other terms:

**Base relationships:**
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

**Directional prefixes:**
These relationships can be prefixed with one of the following to indicate direction:
- `increase`—when the text indicates upregulation, enhancement, stimulation, or increase
- `decrease`—when the text indicates downregulation, inhibition, reduction, or decrease
- `affect`—when the text indicates a change or effect but the direction is unclear or mixed

**Examples of valid relationships:**
- "increase expression"
- "decrease activity"
- "affect phosphorylation"
- "binding" (no prefix needed for non-directional relationships)

---
### Output Format

Generate a CSV with the following columns (include the header row):

**explanation, chemicalName, chemicalID, relationship, geneName, geneID, source**

#### Field Descriptions:
1. **explanation**: Brief description of the interaction (1–2 sentences max)
2. **chemicalName**: Exact name of the chemical from the provided list
3. **chemicalID**: MESH ID of the chemical
4. **relationship**: Relationship type from the predefined list (with prefix if applicable)
5. **geneName**: Exact name of the gene from the provided list
6. **geneID**: NCBI ID of the gene
7. **source**: The exact verbatim text snippet from the input that supports this relationship (use quotation marks if the snippet contains commas)

---
### Extraction Guidelines

DO:
1. Extract only relationships explicitly stated or clearly implied in the text
2. Use only the predefined relationship types from the list above
3. Map common verbs to the appropriate relationship:
   - "upregulates", "enhances", "stimulates" → increase
   - "downregulates", "inhibits", "suppresses", "reduces" → decrease
   - "modulates", "alters", "changes", "affects" → affect
4. Extract each distinct chemical-gene pair as a separate row
5. Use the exact chemical and gene names from the provided lists
6. Ensure all CSV fields are properly formatted and escaped

DO NOT:
1. Make assumptions about relationships not stated in the text
2. Use relationship terms not in the predefined list
3. Add interpretations or additional context beyond what is in the text
4. Extract vague or uncertain relationships
5. Include explanatory text outside the CSV format
6. Invent or paraphrase the source text—it must be verbatim
7. Extract negative relationships (e.g., "chemical X does NOT affect gene Y", "no effect on", "did not change")
8. Extract null findings or lack of effect—only extract positive findings where an interaction exists

---
### Example

#### Example Input Text:
"Exposure to cadmium chloride increases expression of MMP13 mRNA and MMP9 mRNA in cartilage cells. Bisphenol A reduces the activity of TOP2A protein and upregulates TP53 expression in chondrocytes. Treatment with lead acetate did not affect COL2A1 expression."

#### Example Output:
explanation,chemicalName,chemicalID,relationship,geneName,geneID,source
"cadmium chloride increases expression of MMP13","cadmium chloride","D019256","increase expression","MMP13","4332","Exposure to cadmium chloride increases expression of MMP13 mRNA and MMP9 mRNA in cartilage cells."
"cadmium chloride increases expression of MMP9","cadmium chloride","D019256","increase expression","MMP9","4318","Exposure to cadmium chloride increases expression of MMP13 mRNA and MMP9 mRNA in cartilage cells."
"bisphenol A decreases activity of TOP2A","bisphenol A","D001778","decrease activity","TOP2A","7153","Bisphenol A reduces the activity of TOP2A protein and upregulates TP53 expression in chondrocytes."
"bisphenol A increases expression of TP53","bisphenol A","D001778","increase expression","TP53","7157","Bisphenol A reduces the activity of TOP2A protein and upregulates TP53 expression in chondrocytes."

Note: The relationship "lead acetate did not affect COL2A1 expression" is NOT extracted because it describes a null finding (no effect).

---
### Your Task

Now extract all chemical-gene relationships from the input text provided above following these guidelines. Output only the CSV format with no additional commentary.
"""

    return systemPrompt, mainPrompt


In [55]:
def createTableXML_prompt_updated(caption, tableText, tableXML, chem_map, gene_map):
    """
    Generates a system prompt and main prompt for a table (possibly from XML),
    including chemicals and genes. Returns systemPrompt, mainPrompt.
    """
    if isinstance(chem_map, str):
        chem_map = json.loads(chem_map)
    if isinstance(gene_map, str):
        gene_map = json.loads(gene_map)

    systemPrompt = (
        "You are an AI assistant specialized in extracting relationships between chemicals "
        "and genes from XML tables in scientific papers. Be precise, helpful, and honest. "
        "If a relationship is unclear or not present, state that clearly. Only use the data "
        "from the provided table; do not assume anything.\n"
        "Do not extract relationships from negative or non-significant results."
    )

    chemicals = [f"{chem},{mesh_id}" for chem, mesh_id in chem_map.items()]
    genes = [f"{gene},{ncbi_id}" for gene, ncbi_id in gene_map.items()]
    chemicals_str = "\n".join(chemicals)
    genes_str = "\n".join(genes)

    mainPrompt = f"""### Extract Specific Relationships Between Chemicals and Genes from the following table in XML format

#### Table Caption:
{caption}

#### Full Table:
{tableXML}

#### Chemicals and their MESH IDs:
{chemicals_str}

#### Genes and their NCBI IDs:
{genes_str}

---
### Provided Relationship List
The following relationships are predefined and must be used to describe chemical-gene interactions:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---
### Instructions
For each extracted relationship, generate a CSV object with the following fields:
- `chemicalName`: Name of the chemical
- `chemicalID`: Chemical ID if provided, otherwise leave blank.
- `relationship`: Must be chosen from the provided relationship list (may be prefixed by increase, decrease, or affect)
- `geneName`: Name of the gene
- `geneID`: Gene ID if provided, otherwise leave blank.

#### Guidelines:
1. Extract specific relationships included in the table.
2. Include ALL chemicals and genes in your output regardless of them being in the provided gene or chemical data. Set the ID field to - if not provided.
3. Provide only the CSV output—no additional explanations or comments.
4. Ensure that your CSV output is well-formed.
5. Recheck that the relationship is in the provided list.
6. In tabular data, relationships of interest are typically increase expression or decrease expression unless otherwise specified.
7. Do not infer relationships not supported by the provided content

---
### Example Output
chemicalName,chemicalID,relationship,geneName,geneID
"Rebamipide","C052785","increase expression","carbonyl reductase 1","29224"
"Rebamipide","C052785","increase expression","glucose-6-phosphate dehydrogenase","855480"
"""
    return systemPrompt, mainPrompt


In [56]:
def validation_prompt_text(input_text, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes using ONLY the information provided below.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the text; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Use affect only when direction cannot be determined from the text. Remove any prefix applied to exempt relationships.

3. **Entity Validation:** Each relationship must involve exactly two entities. Set agreement=0 if entity information is missing or incomplete; do not guess missing entities.

4. **Null Relationships:** Remove interactions where the source text indicates no effect (e.g., "did not change", "no effect on", "failed to alter").

5. **Prior Work:** Remove interactions cited from previous studies or other works (e.g., "previous studies showed", "it has been reported", "as demonstrated by").

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Original Text: {input_text}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: explanation, chemicalName, chemicalID, relationship, geneName, geneID, source, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise specify which rule was violated and what changed

---

### Example
Original Text: "Exposure to cadmium chloride increases levels of MMP13 and MMP9. Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes. Lead acetate did not change COL2A1 expression. Previous studies showed that arsenic increases oxidation of KEAP1."

Input:
explanation,chemicalName,chemicalID,relationship,geneName,geneID,source
"cadmium chloride increases levels of MMP13","cadmium chloride","D019256","increase levels","MMP13","4332","Exposure to cadmium chloride increases levels of MMP13 and MMP9."
"cadmium chloride increases expression of MMP9","cadmium chloride","D019256","response to chemical","MMP9","4318","Exposure to cadmium chloride increases levels of MMP13 and MMP9."
"bisphenol A decreases activity of TOP2A","bisphenol A","D001778","decrease activity","TOP2A","7153","Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes."
"bisphenol A affects expression of TP53","bisphenol A","D001778","affect expression","TP53","7157","Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes."
"lead acetate has no effect on COL2A1","lead acetate","D007854","affect expression","COL2A1","1280","Lead acetate did not change COL2A1 expression."
"arsenic increases oxidation of KEAP1","arsenic","D001151","increase oxidation","KEAP1","9817","Previous studies showed that arsenic increases oxidation of KEAP1."

Output:
explanation,chemicalName,chemicalID,relationship,geneName,geneID,source,agreement,reasoning
"cadmium chloride increases expression of MMP13","cadmium chloride","D019256","increase expression","MMP13","4332","Exposure to cadmium chloride increases levels of MMP13 and MMP9.","0","Changed 'increase levels' to 'increase expression': 'levels' is not in the predefined list"
"cadmium chloride increases expression of MMP9","cadmium chloride","D019256","increase expression","MMP9","4318","Exposure to cadmium chloride increases levels of MMP13 and MMP9.","0","Changed 'response to chemical' to 'increase expression': chemical-only relationship invalid for chemical-gene pair"
"bisphenol A decreases activity of TOP2A","bisphenol A","D001778","decrease activity","TOP2A","7153","Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes.","1",""
"bisphenol A affects expression of TP53","bisphenol A","D001778","affect expression","TP53","7157","Bisphenol A reduces activity of TOP2A and affects expression of TP53 in chondrocytes.","1",""
"lead acetate has no effect on COL2A1","lead acetate","D007854","affect expression","COL2A1","1280","Lead acetate did not change COL2A1 expression.","0","Null relationship: source text states 'did not change'"
"arsenic increases oxidation of KEAP1","arsenic","D001151","increase oxidation","KEAP1","9817","Previous studies showed that arsenic increases oxidation of KEAP1.","0","Relationship from previous studies/other works"

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [16]:
def validation_prompt_table(caption, tableXML, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes extracted from a table.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the table; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Use table context to determine direction: positive values, or fold-change >1.0 indicate increase; negative values, or fold-change <1.0 indicate decrease.

3. **Entity Validation:** Each relationship must involve exactly two entities. Set agreement=0 if entity information is missing or incomplete.

4. **Null Relationships:** Remove interactions marked "NS", "no change", "0.00", empty effect cells, or fold-change = 1.0 with NS notation.

5. **Table Context:** Verify that the relationship type and prefix direction align with column headers, table symbols, and fold-change values.

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Table Caption: {caption}
- Table XML:
{tableXML}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: chemicalName, chemicalID, relationship, geneName, geneID, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise explain what changed and why

---

### Example

Input:
chemicalName,chemicalID,relationship,geneName,geneID
"cadmium chloride","D019256","increase levels","MMP13","4332"
"Lead acetate","D007854","affect expression","COL2A1","1280"

Table shows: Cadmium increases expression 2.3-fold; Lead shows 1.0 (NS).

Output:
chemicalName,chemicalID,relationship,geneName,geneID,agreement,reasoning
"cadmium chloride","D019256","increase expression","MMP13","4332","0","Changed 'increase levels' to 'increase expression': 'levels' not in predefined list"
"Lead acetate","D007854","affect expression","COL2A1","1280","0","Null relationship: table shows 1.0 (NS), no significant change"

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [17]:
def validation_prompt_figure(caption, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes extracted from a figure/image.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the image; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Ensure consistent lowercase capitalization.

3. **Image Direction Indicators:** Use visual cues to determine directionality: taller/darker bars, upward slopes, symbols, and warm colors indicate increase; shorter/lighter bars, downward slopes, symbols, and cool colors indicate decrease.

4. **Entity Validation:** Each relationship must involve exactly two entities. Use "-" for missing gene name or symbol. Set agreement=0 if entity information is incomplete.

5. **Null Relationships:** Remove interactions marked "NS" or "n.s.", showing no visible difference, or indicating no effect.

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Image Caption: {caption}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: chemicalName, relationship, geneName, geneSymbol, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise explain what changed and why

---

### Example

Input:
chemicalName,relationship,geneName,geneSymbol
"Genistein","Decreases expression","-","DEF2"
"Genistein","Increases expression","-","ABC1"
"Genistein","activity","tyrosine 3-monooxygenase","-"

Caption: Gene regulation by Genistein.

Output:
chemicalName,relationship,geneName,geneSymbol,agreement,reasoning
"Genistein","decrease expression","-","DEF2","0","Fixed capitalization: 'Decreases' to 'decrease'"
"Genistein","increase expression","-","ABC1","0","Fixed capitalization: 'Increases' to 'increase'"
"Genistein","affect activity","tyrosine 3-monooxygenase","-","0","Added 'affect' prefix: 'activity' requires a directional prefix"

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [18]:
def validation_prompt_figure_table(caption, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes extracted from a table in a figure/image.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the table; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Ensure consistent lowercase capitalization.

3. **Direction Validation:** Verify prefix direction using all available context:
   - Caption: "up-regulated" → increase; "down-regulated" → decrease.
   - Fold-change: >1.0 or positive log2 = increase; <1.0 or negative log2 = decrease (verify comparison direction from column headers).

4. **Entity Validation:** Each relationship must involve exactly two entities. Use "-" for missing gene name or symbol. Set agreement=0 if entity information is incomplete.

5. **Null Relationships:** Remove interactions marked "NS", "n.s.", fold-change = 1.0, or indicating no effect.

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Table Caption: {caption}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: chemicalName, relationship, geneName, geneSymbol, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise explain what changed and why

---

### Example

Input:
chemicalName,relationship,geneName,geneSymbol
"Genistein","Decreases expression","-","DEF2"
"Genistein","Increases expression","-","ABC1"
"Genistein","Decreases expression","tyrosine 3-monooxygenase","-"

Caption: Gene regulation by Genistein.

Output:
chemicalName,relationship,geneName,geneSymbol,agreement,reasoning
"Genistein","decrease expression","-","DEF2","0","Fixed capitalization: 'Decreases' to 'decrease'"
"Genistein","increase expression","-","ABC1","0","Fixed capitalization: 'Increases' to 'increase'"
"Genistein","decrease expression","tyrosine 3-monooxygenase","-","0","Fixed capitalization: 'Decreases' to 'decrease'"

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [19]:
def validation_prompt_supplement(caption, sheet_name, tableCSVstring, extracted_relationships):
    
    prompt = f"""Validate and correct relationships between chemicals and genes extracted from a supplemental table.

---

### Validation Rules

1. **Relationship Type:** Must match the predefined list. Replace invalid relationships with the closest valid alternative supported by the table; mark for removal (agreement=0) if no valid alternative exists. Chemical-only relationships (abundance, response to chemical, chemical synthesis) are invalid for chemical–gene pairs.

2. **Prefix Rules:** All relationships require a directional prefix (increase, decrease, affect) except cotreatment and binding. Use affect when direction is unclear.

3. **Direction Validation:** Verify prefix direction using all available context:
   - Sheet name: "down" or "decrease" in sheet name → genes DOWN in control → chemical INCREASES expression; "up" or "increase" → chemical DECREASES expression.
   - Caption/headers: "up-regulated in [chemical]" → increase; "down-regulated in [chemical]" → decrease.
   - Fold-change: >1.0 or positive log2 = increase; <1.0 or negative log2 = decrease (verify comparison direction from column headers).

4. **Entity Validation:** Each relationship must involve exactly two entities. Set agreement=0 if entity information is missing or incomplete.

5. **Null Relationships:** Remove interactions marked "NS", "no change", fold-change = 1.0, or ~0 in log scale.

---

### Predefined Relationships:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

---

### Input:
- Table Caption: {caption}
- Sheet Name: {sheet_name if sheet_name else "Not provided"}
- Table CSV:
{tableCSVstring}
- Extracted Relationships:
{extracted_relationships}

---

### Output Format
Return CSV with header: chemicalName, relationship, geneName, geneSymbol, agreement, reasoning
- agreement: 1 (correct as-is) or 0 (needs correction/removal)
- reasoning: blank if agreement=1, otherwise explain what changed and why

---

### Example

Caption: Genes with significant changes after masitinib treatment.
Sheet Name: Control decreased.

Input:
chemicalName,relationship,geneName,geneSymbol
"masitinib","decrease expression","carbonyl reductase 1","CR1"
"masitinib","increase expression","glucose-6-phosphate dehydrogenase","G6PD"

Output:
chemicalName,relationship,geneName,geneSymbol,agreement,reasoning
"masitinib","increase expression","carbonyl reductase 1","CR1","0","Changed 'decrease' to 'increase' sheet name indicates genes down in control"
"masitinib","increase expression","glucose-6-phosphate dehydrogenase","G6PD","1",""

---

Validate each row. Output only the CSV, no additional commentary.
"""
    return prompt


In [20]:
systemPromptValidateText = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided source text. Verify whether each extracted relationship is factually supported by the text and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the text, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- You may ONLY modify: relationship, explanation.
"""

systemPromptValidateTable = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided XML table. Verify whether each extracted relationship is factually supported by the table and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the table, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- You may ONLY modify: chemicalName, chemicalID, relationship, geneName, geneSymbol.
"""

systemPromptValidateFigure = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided figure/image and its caption. Verify whether each extracted relationship is factually supported by the visual data and caption, and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the image/caption, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- You may ONLY modify: chemicalName, relationship, geneName, geneSymbol.
"""

systemPromptValidateFigureTable = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided table embedded in a figure/image and its caption. Verify whether each extracted relationship is factually supported by the table data and caption, and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the table/caption, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- You may ONLY modify: chemicalName, relationship, geneName, geneSymbol.
"""

systemPromptValidateSupplement = """
You are an AI assistant responsible for validating extracted relationships between chemicals and genes based solely on a provided supplemental table (CSV format), its caption, and sheet name. Verify whether each extracted relationship is factually supported by the table data and conforms to the allowed relationship schema.

Constraints:
- Do NOT add new relationships or remove existing relationships.
- Correct a relationship ONLY if it is factually incorrect, unsupported by the table data, or violates the relationship rules.
- Do NOT introduce external knowledge or assumptions.
- Pay special attention to sheet names and captions for directional information (increase/decrease).
- You may ONLY modify: chemicalName, relationship, geneName, geneSymbol.
"""


# add prompts

In [21]:
# create supp prompts
def createBatches(df, batch_size=50):
    """
    Create batches of CSV strings from a DataFrame.
    Each batch includes header, no index.

    Returns: list of CSV strings
    """
    batches = []

    for i in range(0, len(df), batch_size):
        batch_df = df.iloc[i:i + batch_size]
        batches.append(batch_df.to_csv(index=False))

    return batches


def create_supp_prompts(batches, caption, sheet_name, pmid):
    """
    For each CSV batch, create system and main prompts.
    Stores results in a DataFrame and returns it.
    """

    records = []

    for i, batch_csv in enumerate(batches):
        systemPrompt, mainPrompt = supplement_XLS_prompt(batch_csv, caption, sheet_name)

        records.append({
            "pmid": pmid,
            "caption": caption,
            "sheet_name": sheet_name,
            "section_type": "SUPPLEMENT",
            "system_prompt": systemPrompt,
            "main_prompt": mainPrompt
        })

    return pd.DataFrame(records)
    
supp_2_batches_a = createBatches(supp_xls_2_a, batch_size=30)
supp_2_batches_b = createBatches(supp_xls_2_b, batch_size=30)

supp_2_prompts_a = create_supp_prompts(supp_2_batches_a, supp_cap_2, supp_xls_2_a_name, 22747577)
supp_2_prompts_b = create_supp_prompts(supp_2_batches_b, supp_cap_2, supp_xls_2_b_name, 22747577)

supp_2_prompts_b

,pmid,caption,sheet_name,section_type,system_prompt,main_prompt
0,22747577,"Genes with a time-dependent, continuous increa...",ctr down >1.5,SUPPLEMENT,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...
1,22747577,"Genes with a time-dependent, continuous increa...",ctr down >1.5,SUPPLEMENT,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...
2,22747577,"Genes with a time-dependent, continuous increa...",ctr down >1.5,SUPPLEMENT,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...


In [22]:
# create fig prompts
def parse_map(cell):
    """
    Converts a newline-separated 'name: ID' string into a dict.
    Returns None if cell is NaN or empty.
    """
    if pd.isna(cell) or not str(cell).strip():
        return None

    result = {}
    for line in str(cell).split("\n"):
        if ":" in line:
            name, _id = line.split(":", 1)
            result[name.strip()] = _id.strip()
    return result if result else None

def build_prompts(row):
    chem_map = parse_map(row["chemical_map"])
    gene_map = parse_map(row["gene_map"])
    caption = row["caption"]

    if row["type"] == "figure":
        sys_prompt, main_prompt = fig_prompt(
            # chem_map=chem_map,
            # gene_map=gene_map,
            caption=caption
        )
    if row["type"] == "table":
        sys_prompt, main_prompt = fig_table_prompt(
            # chem_map=chem_map,
            # gene_map=gene_map,
            caption=caption
        )
 
    df = pd.Series({
        "system_prompt": sys_prompt,
        "main_prompt": main_prompt
    })

    return df

figs[["system_prompt", "main_prompt"]] = figs.apply(build_prompts, axis=1)
figs['section_type'] = 'FIG'
len(figs)

150

In [23]:
# create text & table prompts
def generate_prompts(row):
    # Skip TABLE or FIG rows
    chem_map = row['chem_map']
    gene_map = row['gene_map']
    
    if row['section_type'] in ['TABLE']:
        if row['section_type'] in ['TABLE']:
            print('True')
            return createTableXML_prompt_updated(row['table_caption'], row['table_text'], row['table_xml'], chem_map, gene_map)
        else:
            return None

    # Convert chem_map and gene_map from JSON strings to dicts if needed
    if isinstance(chem_map, str):
        chem_map = json.loads(chem_map)
    if isinstance(gene_map, str):
        gene_map = json.loads(gene_map)

    # Call your existing function that returns a tuple (system_prompt, main_prompt)
    return createTextPrompt(row['text'], chem_map, gene_map)

# Apply to the DataFrame
prompts = text.apply(generate_prompts, axis=1)

# Split the tuple into two separate columns
text['system_prompt'] = prompts.apply(lambda x: x[0] if x is not None else None)
text['main_prompt'] = prompts.apply(lambda x: x[1] if x is not None else None)

len(text)

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


428

In [24]:
# combine

text = text.rename(columns={"chem_map": "chemical_map"})
text = text.rename(columns={"table_caption": "caption"})

combined_main = pd.concat(
    [figs, text, supp_2_prompts_a, supp_2_prompts_b],
    ignore_index=True
)
combined = combined_main.drop(columns = ['id', 'pmcid', 'table_text', 'number'], axis=1)

In [25]:
combined = combined.drop(
    combined[
        (combined["section_type"] == "TABLE") &
        (combined["table_xml"].isna() | (combined["table_xml"].str.strip() == ""))
    ].index
)
combined
combined.to_csv(f'{input_data_path}/bedrock_input_v2.csv')

# LLM calls - extraction

In [26]:
os.environ["AWS_ACCESS_KEY_ID"] = "X"
os.environ["AWS_SECRET_ACCESS_KEY"] = "X"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
gpt_key = 'X'
deepseek_key = 'X'
anthropic_key = "X"

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

openai_client = OpenAI(api_key=gpt_key)
claude_client = Anthropic(api_key=anthropic_key)
deepseek_client = OpenAI(api_key=deepseek_key, base_url="https://api.deepseek.com")

MODELS = {
    "claude": {
        "provider": "anthropic",
        "model_id": "claude-sonnet-4-5"
    },
    "deepseek": {
        "provider": "deepseek",
        "model_id": "deepseek.v3.2"
    },
    "llama": {
        "provider": "bedrock",
        "style": "instruction",
        "model_id": "meta.llama3-2-90b-instruct-v1:0"
    },
    "gpt": {
        "provider": "openai",
        "model_id": "gpt4.1"
    },
    "gpt4o": {
        "provider": "openai",
        "model_id": "gpt4.1"
    }
}

MODELS=['claude','deepseek', 'gpt', 'gpt4o']

/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [30]:
input = pd.read_csv(f'{input_data_path}/bedrock_input_v2.csv').drop(columns=['Unnamed: 0'], axis=1)

for name in MODELS:
    col = f"{name}_output"
    if col not in input.columns:
        input[col] = None

input.columns

input

,pmid,caption,filename,orginal_filename,type,gene_map,chemical_map,system_prompt,main_prompt,section_type,text,table_xml,sheet_name,claude_output,deepseek_output,gpt_output,gpt4o_output
0,17524151,Cytotoxicity of SM on SAE and BTE cells . Cell...,17524151_figure1_img1_1471-2121-8-17-1.jpg,1471-2121-8-17-1.jpg,figure,NaN,SM: D009151,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,NaN,NaN,NaN,None,None,None,None
1,17524151,Effect of roxithromycin on SM cytotoxicity in ...,17524151_figure2_img1_1471-2121-8-17-2.jpg,1471-2121-8-17-2.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,NaN,NaN,NaN,None,None,None,None
2,17524151,Anti-cytotoxicity effect of roxithromycin on S...,17524151_figure3_img1_1471-2121-8-17-3.jpg,1471-2121-8-17-3.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151\nCalcein A...,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,NaN,NaN,NaN,None,None,None,None
3,17524151,Inhibition by roxithromycin of cytokine releas...,17524151_figure4_img1_1471-2121-8-17-4.jpg,1471-2121-8-17-4.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,NaN,NaN,NaN,None,None,None,None
4,17524151,Inhibition by roxithromycin of cytokine mRNA e...,17524151_figure5_img1_1471-2121-8-17-5.jpg,1471-2121-8-17-5.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,NaN,NaN,NaN,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
576,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,NaN,NaN,ctr up >1.5,None,None,None,None
577,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,NaN,NaN,ctr up >1.5,None,None,None,None
578,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,NaN,NaN,ctr down >1.5,None,None,None,None
579,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,NaN,NaN,ctr down >1.5,None,None,None,None


In [31]:
def image_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


In [32]:
def call_claude(system_prompt, main_prompt, claude_model="claude-sonnet-4-5", image_path=None):
    """
    Calls Claude Sonnet 4.5 via Messages API with optional image support.
    """

    messages = []

    # Add image if provided
    if image_path:
        messages.append({
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",  # your images are jpg
                        "data": image_to_base64(image_path)
                    }
                }
            ]
        })

    # Add main prompt text
    messages.append({
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": main_prompt
            }
        ]
    })

    try:
        response = claude_client.messages.create(
            model=claude_model,
            system=system_prompt,  # top-level system prompt
            messages=messages,
            max_tokens=10000,
            temperature=0
        )

        # Extract text from the response
        response_text = ""
        if hasattr(response, "content") and response.content:
            text_blocks = [
                block.text for block in response.content
                if hasattr(block, "text") and block.text
            ]
            response_text = "\n".join(text_blocks)

        if not response_text.strip():
            response_text = "ERROR: Empty Claude response"

        return response_text

    except Exception as e:
        return f"ERROR: {e}"




def call_deepseek(system_prompt, main_prompt, model="deepseek-chat"):
    """
    Calls DeepSeek for a single row with system_prompt and main_prompt.
    Returns the model output as a string.
    """
    try:
        response = deepseek_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": main_prompt}
            ],
            temperature=0.0,
            seed=42,
            max_tokens=8192,
            stream=False
        )

        # Return the response text
        return response.choices[0].message.content

    except Exception as e:
        print(f"✗ DeepSeek call failed: {e}")
        return None


def call_llama(system_prompt, main_prompt,
               model="meta.llama3-2-90b-instruct-v1:0"):

    body = {
        "prompt": f"<<SYS>>{system_prompt}<</SYS>>\n\n{main_prompt}",
        "max_gen_len": 10000,
        "temperature": 0
    }

    response = bedrock.invoke_model(
        modelId=model,
        body=json.dumps(body)
    )

    payload = json.loads(response["body"].read())
    return payload["generation"]

def call_gpt(system_prompt, main_prompt, image_path=None,
             model="gpt-4.1"):

    content = []

    if image_path:
        content.append({
            "type": "input_image",
            "image_url": f"data:image/png;base64,{image_to_base64(image_path)}"
        })

    content.append({
        "type": "input_text",
        "text": main_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=[{
            "role": "user",
            "content": content
        }],
        max_output_tokens=10000
    )

    return response.output_text

def call_gpt4o(system_prompt, main_prompt, image_path=None,
             model="gpt-4o"):

    content = []

    if image_path:
        content.append({
            "type": "input_image",
            "image_url": f"data:image/png;base64,{image_to_base64(image_path)}"
        })

    content.append({
        "type": "input_text",
        "text": main_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=[{
            "role": "user",
            "content": content
        }],
        max_output_tokens=10000
    )

    return response.output_text


In [33]:
def run_models_on_row(row, idx):
    is_fig = row.get('section_type') in ["FIG", "FIG_TABLE"]

    image_path = (
        f"{figures_path}/{row['filename']}"
        if is_fig and pd.notna(row.get("filename"))
        else None
    )
    print(image_path)

    outputs = {}

    print(f"\n🔹 Row {idx} | section_type={row['section_type']}")

    # ---------- Claude ----------
    if pd.notna(row.get("claude_output")):
        print("  ⏭ Claude skipped (already exists)")
        outputs["claude_output"] = row["claude_output"]
    else:
        print("  ▶ Claude...")
        try:
            outputs["claude_output"] = call_claude(
                row["system_prompt"],
                row["main_prompt"],
                image_path=image_path
            )
            print("    ✓ Claude done")
        except Exception as e:
            outputs["claude_output"] = f"ERROR: {e}"
            print("    ✗ Claude failed")

    # ---------- GPT ----------
    if pd.notna(row.get("gpt_output")):
        print("  ⏭ GPT skipped (already exists)")
        outputs["gpt_output"] = row["gpt_output"]
    else:
        print("  ▶ GPT...")
        try:
            outputs["gpt_output"] = call_gpt(
                row["system_prompt"],
                row["main_prompt"],
                image_path=image_path
            )
            print("    ✓ GPT done")
        except Exception as e:
            outputs["gpt_output"] = f"ERROR: {e}"
            print("    ✗ GPT failed")

    # ---------- GPT ----------
    if pd.notna(row.get("gpt4o_output")):
        print("  ⏭ GPT4o skipped (already exists)")
        outputs["gpt4o_output"] = row["gpt4o_output"]
    else:
        print("  ▶ GPT4o...")
        try:
            outputs["gpt4o_output"] = call_gpt4o(
                row["system_prompt"],
                row["main_prompt"],
                image_path=image_path
            )
            print("    ✓ GPT done")
        except Exception as e:
            outputs["gpt4o_output"] = f"ERROR: {e}"
            print("    ✗ GPT failed")

    # ---------- DeepSeek ----------
    if is_fig:
        print("  ⏭ DeepSeek skipped (FIG)")
        outputs["deepseek_output"] = row.get("deepseek_output")
    elif pd.notna(row.get("deepseek_output")):
        print("  ⏭ DeepSeek skipped (already exists)")
        outputs["deepseek_output"] = row["deepseek_output"]
    else:
        print("  ▶ DeepSeek...")
        try:
            outputs["deepseek_output"] = call_deepseek(
                row["system_prompt"],
                row["main_prompt"]
            )
            print("    ✓ DeepSeek done")
        except Exception as e:
            outputs["deepseek_output"] = f"ERROR: {e}"
            print("    ✗ DeepSeek failed")

    # ---------- LLaMA ----------
    if is_fig:
        print("  ⏭ LLaMA skipped (FIG)")
        outputs["llama_output"] = row.get("llama_output")
    elif pd.notna(row.get("llama_output")):
        print("  ⏭ LLaMA skipped (already exists)")
        outputs["llama_output"] = row["llama_output"]
    else:
        print("  ▶ LLaMA...")
        try:
            outputs["llama_output"] = call_llama(
                row["system_prompt"],
                row["main_prompt"]
            )
            print("    ✓ LLaMA done")
        except Exception as e:
            outputs["llama_output"] = f"ERROR: {e}"
            print("    ✗ LLaMA failed")

    print(f"✅ Row {idx} complete")
    return pd.Series(outputs)


In [35]:
import pandas as pd

# Path to save the CSV checkpoint
output_file = f"{output_data_path}/output_4o.csv"

for idx, row in input.iterrows():
    # Skip if outputs already exist for this row
    if (
        # pd.notna(row.get("claude_output")) or
        # pd.notna(row.get("gpt_output")) or
        # pd.notna(row.get("deepseek_output"))
        # pd.notna(row.get("llama_output"))
        pd.notna(row.get("gpt4o_output"))
    ):
        print(f"⏭ Row {idx} skipped: outputs already exist")
        continue

    # Run models for this row
    outputs = run_models_on_row(row, idx)

    # Write outputs back to the dataframe
    for col, val in outputs.items():
        input.at[idx, col] = val

    # Save checkpoint every 50 rows
    if (idx + 1) % 50 == 0:
        input.to_csv(output_file, index=False)
        print(f"✅ Checkpoint saved at row {idx + 1}")

# Save final output
input.to_csv(output_file, index=False)
print("\n✅ All rows complete and saved to", output_file)


⏭ Row 0 skipped: outputs already exist
⏭ Row 1 skipped: outputs already exist
⏭ Row 2 skipped: outputs already exist
⏭ Row 3 skipped: outputs already exist
⏭ Row 4 skipped: outputs already exist
⏭ Row 5 skipped: outputs already exist
⏭ Row 6 skipped: outputs already exist
⏭ Row 7 skipped: outputs already exist
⏭ Row 8 skipped: outputs already exist
⏭ Row 9 skipped: outputs already exist
⏭ Row 10 skipped: outputs already exist
⏭ Row 11 skipped: outputs already exist
⏭ Row 12 skipped: outputs already exist
⏭ Row 13 skipped: outputs already exist
⏭ Row 14 skipped: outputs already exist
⏭ Row 15 skipped: outputs already exist
⏭ Row 16 skipped: outputs already exist
⏭ Row 17 skipped: outputs already exist
⏭ Row 18 skipped: outputs already exist
⏭ Row 19 skipped: outputs already exist
⏭ Row 20 skipped: outputs already exist
⏭ Row 21 skipped: outputs already exist
⏭ Row 22 skipped: outputs already exist
⏭ Row 23 skipped: outputs already exist
⏭ Row 24 skipped: outputs already exist
⏭ Row 25 s

In [212]:
input['claude_output']

0      ```\nchemicalName,relationship,geneName,geneSy...
1      ```\nchemicalName,relationship,geneName,geneSy...
2      ```\nchemicalName,relationship,geneName,geneSy...
3      ```\nchemicalName,relationship,geneName,geneSy...
4      ```\nchemicalName,relationship,geneName,geneSy...
                             ...                        
576    ```csv\nchemicalName,relationship,geneName,gen...
577    ```csv\nchemicalName,relationship,geneName,gen...
578    ```csv\nchemicalName,relationship,geneName,gen...
579    ```csv\nchemicalName,relationship,geneName,gen...
580    ```csv\nchemicalName,relationship,geneName,gen...
Name: claude_output, Length: 581, dtype: object

# Validation

In [51]:
input = pd.read_csv(f'{output_data_path}/output_v2.csv')
input.columns

Index(['pmid', 'caption', 'filename', 'orginal_filename', 'type', 'gene_map',
       'chemical_map', 'system_prompt', 'main_prompt', 'section_type', 'text',
       'table_xml', 'sheet_name', 'claude_output', 'deepseek_output',
       'gpt_output', 'gpt4o_output', 'llama_output'],
      dtype='object')

In [52]:
def find_csv_blocks(text):
    pattern = r'(?:^[^\n,]+(?:,[^\n,]+)+\n)+(?:^[^\n,]+(?:,[^\n,]+)+$)'
    matches = re.findall(pattern, text, re.MULTILINE)
    return matches

def extract_csv(text):

    # ✅ Handle None / NaN / non-strings safely
    if text is None or pd.isna(text):
        return None

    if not isinstance(text, str):
        text = str(text)

    # STEP 1: Look inside ``` or ```csv blocks
    code_block_pattern = r"```(?:csv)?\s*\n(.*?)```"
    code_blocks = re.findall(code_block_pattern, text, re.DOTALL | re.IGNORECASE)

    csv_blocks = []

    for block in code_blocks:
        if "," in block and len(block.strip().splitlines()) >= 2:
            csv_blocks.append(block.strip())

    if csv_blocks:
        return "\n\n".join(csv_blocks)

    # STEP 2: fallback regex detection
    fallback = find_csv_blocks(text)
    if fallback:
        return "\n\n".join(fallback)

    return None

llm_cols = ['deepseek_output', 'gpt_output', 'claude_output', 'gpt4o_output']

for col in llm_cols:
    new_col = col.replace("_output", "_E_output")
    print(f"Processing {col} → {new_col}")

    input[new_col] = input[col].apply(extract_csv)

input

Processing deepseek_output → deepseek_E_output
Processing gpt_output → gpt_E_output
Processing claude_output → claude_E_output
Processing gpt4o_output → gpt4o_E_output


,pmid,caption,filename,orginal_filename,type,gene_map,chemical_map,system_prompt,main_prompt,section_type,...,sheet_name,claude_output,deepseek_output,gpt_output,gpt4o_output,llama_output,deepseek_E_output,gpt_E_output,claude_E_output,gpt4o_E_output
0,17524151,Cytotoxicity of SM on SAE and BTE cells . Cell...,17524151_figure1_img1_1471-2121-8-17-1.jpg,1471-2121-8-17-1.jpg,figure,NaN,SM: D009151,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,"```\n""chemicalName"",""relationship"",""geneName"",...",NaN,"```\n""chemicalName"",""relationship"",""geneName"",...","```\n""chemicalName"",""relationship"",""geneName"",...",NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene..."
1,17524151,Effect of roxithromycin on SM cytotoxicity in ...,17524151_figure2_img1_1471-2121-8-17-2.jpg,1471-2121-8-17-2.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,"```\n""chemicalName"",""relationship"",""geneName"",...",NaN,"```\n""chemicalName"",""relationship"",""geneName"",...","```\n""chemicalName"",""relationship"",""geneName"",...",NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene..."
2,17524151,Anti-cytotoxicity effect of roxithromycin on S...,17524151_figure3_img1_1471-2121-8-17-3.jpg,1471-2121-8-17-3.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151\nCalcein A...,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,"```\n""chemicalName"",""relationship"",""geneName"",...",NaN,"```\n""chemicalName"",""relationship"",""geneName"",...","```\n""chemicalName"",""relationship"",""geneName"",...",NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene..."
3,17524151,Inhibition by roxithromycin of cytokine releas...,17524151_figure4_img1_1471-2121-8-17-4.jpg,1471-2121-8-17-4.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,"```\n""chemicalName"",""relationship"",""geneName"",...",NaN,"```\n""chemicalName"",""relationship"",""geneName"",...","```\n""chemicalName"",""relationship"",""geneName"",...",NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene..."
4,17524151,Inhibition by roxithromycin of cytokine mRNA e...,17524151_figure5_img1_1471-2121-8-17-5.jpg,1471-2121-8-17-5.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,"```\n""chemicalName"",""relationship"",""geneName"",...",NaN,"```\n""chemicalName"",""relationship"",""geneName"",...","```\n""chemicalName"",""relationship"",""geneName"",...",NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
576,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,ctr up >1.5,"```csv\n""chemicalName"",""relationship"",""geneNam...","""chemicalName"",""relationship"",""geneName"",""gene...","```\n""chemicalName"",""relationship"",""geneName"",...","```csv\n""chemica

In [53]:
# build validation prompts for each
# add system prompts based on section_type
def assign_validation_prompt(section_type):
    if not isinstance(section_type, str):
        return systemPromptValidateText  # safe default

    st = section_type.strip().upper()

    if "SUPPLEMENT" in st:
        return systemPromptValidateSupplement

    if "FIG" in st and "TABLE" in st:
        return systemPromptValidateFigureTable

    if "FIG" in st:
        return systemPromptValidateFigure

    if "TABLE" in st:
        return systemPromptValidateTable

    # default = regular text validation
    return systemPromptValidateText

input["system_prompt_val"] = input["section_type"].apply(assign_validation_prompt)


In [54]:
def extract_full_table_csv(text: str) -> str:
    """
    Extracts the CSV content after '### Full Table:' until the next '---' line.
    Returns the CSV string or None if not found.
    """
    if not isinstance(text, str):
        return None

    lines = text.splitlines()
    table_lines = []
    in_table = False

    for line in lines:
        stripped = line.strip()
        # Start collecting after Full Table:
        if not in_table and stripped.lower().startswith("#### full table:"):
            in_table = True
            continue
        # Stop at section delimiter
        if in_table and stripped.startswith("---"):
            break
        # Collect lines if inside table
        if in_table:
            table_lines.append(line)

    if not table_lines:
        return None

    return "\n".join(table_lines).strip()


In [55]:
def build_main_prompt_val_gpt(row):

    section_type = str(row.get("section_type", "")).strip().upper()

    if pd.isna(row.get("gpt_E_output")):
        return None

    extracted_relationships = row.get("gpt_E_output", "")
    caption = row.get("caption", "")
    text = row.get("text", "")
    table_xml = row.get("table_xml", "")
    sheet_name = row.get("sheet_name", "")

    tableCSVstring = ''
    if(section_type == "SUPPLEMENT"):
        tableCSVstring = extract_full_table_csv(row.get('main_prompt'))

    if "SUPPLEMENT" in section_type:
        return validation_prompt_supplement(caption, sheet_name, tableCSVstring, extracted_relationships)

    if "FIG" in section_type and "TABLE" in section_type:
        return validation_prompt_figure_table(caption, extracted_relationships)

    if "FIG" in section_type:
        return validation_prompt_figure(caption, extracted_relationships)

    if "TABLE" in section_type:
        return validation_prompt_table(caption, table_xml, extracted_relationships)

    # Default → TEXT
    return validation_prompt_text(text, extracted_relationships)

def build_main_prompt_val_gpt4o(row):

    section_type = str(row.get("section_type", "")).strip().upper()

    if pd.isna(row.get("gpt4o_E_output")):
        return None

    extracted_relationships = row.get("gpt4o_E_output", "")
    caption = row.get("caption", "")
    text = row.get("text", "")
    table_xml = row.get("table_xml", "")
    sheet_name = row.get("sheet_name", "")

    tableCSVstring = ''
    if(section_type == "SUPPLEMENT"):
        tableCSVstring = extract_full_table_csv(row.get('main_prompt'))

    if "SUPPLEMENT" in section_type:
        return validation_prompt_supplement(caption, sheet_name, tableCSVstring, extracted_relationships)

    if "FIG" in section_type and "TABLE" in section_type:
        return validation_prompt_figure_table(caption, extracted_relationships)

    if "FIG" in section_type:
        return validation_prompt_figure(caption, extracted_relationships)

    if "TABLE" in section_type:
        return validation_prompt_table(caption, table_xml, extracted_relationships)

    # Default → TEXT
    return validation_prompt_text(text, extracted_relationships)

def build_main_prompt_val_deepseek(row):

    section_type = str(row.get("section_type", "")).strip().upper()

    if pd.isna(row.get("deepseek_E_output")):
        return None

    extracted_relationships = row.get("deepseek_E_output", "")
    caption = row.get("caption", "")
    text = row.get("text", "")
    table_xml = row.get("table_xml", "")
    sheet_name = row.get("sheet_name", "")
    
    tableCSVstring = ''
    if(section_type == "SUPPLEMENT"):
        tableCSVstring = extract_full_table_csv(row.get('main_prompt'))

    if "SUPPLEMENT" in section_type:
        return validation_prompt_supplement(caption, sheet_name, tableCSVstring, extracted_relationships)

    if "FIG" in section_type and "TABLE" in section_type:
        return validation_prompt_figure_table(caption, extracted_relationships)

    if "FIG" in section_type:
        return validation_prompt_figure(caption, extracted_relationships)

    if "TABLE" in section_type:
        return validation_prompt_table(caption, table_xml, extracted_relationships)

    # Default → TEXT
    return validation_prompt_text(text, extracted_relationships)


def build_main_prompt_val_claude(row):

    section_type = str(row.get("section_type", "")).strip().upper()

    if pd.isna(row.get("claude_E_output")):
        return None

    extracted_relationships = row.get("claude_E_output", "")
    caption = row.get("caption", "")
    text = row.get("text", "")
    table_xml = row.get("table_xml", "")
    sheet_name = row.get("sheet_name", "")

    tableCSVstring = ''
    if(section_type == "SUPPLEMENT"):
        tableCSVstring = extract_full_table_csv(row.get('main_prompt'))

    if "SUPPLEMENT" in section_type:
        return validation_prompt_supplement(caption, sheet_name, tableCSVstring, extracted_relationships)

    if "FIG" in section_type and "TABLE" in section_type:
        return validation_prompt_figure_table(caption, extracted_relationships)

    if "FIG" in section_type:
        return validation_prompt_figure(caption, extracted_relationships)

    if "TABLE" in section_type:
        return validation_prompt_table(caption, table_xml, extracted_relationships)

    # Default → TEXT
    return validation_prompt_text(text, extracted_relationships)



In [56]:
input["main_prompt_val_gpt"] = input.apply(build_main_prompt_val_gpt, axis=1)
input["main_prompt_val_deepseek"] = input.apply(build_main_prompt_val_deepseek, axis=1)
input["main_prompt_val_claude"] = input.apply(build_main_prompt_val_claude, axis=1)
input["main_prompt_val_gpt4o"] = input.apply(build_main_prompt_val_gpt4o, axis=1)

input.head(5)

,pmid,caption,filename,orginal_filename,type,gene_map,chemical_map,system_prompt,main_prompt,section_type,...,llama_output,deepseek_E_output,gpt_E_output,claude_E_output,gpt4o_E_output,system_prompt_val,main_prompt_val_gpt,main_prompt_val_deepseek,main_prompt_val_claude,main_prompt_val_gpt4o
0,17524151,Cytotoxicity of SM on SAE and BTE cells . Cell...,17524151_figure1_img1_1471-2121-8-17-1.jpg,1471-2121-8-17-1.jpg,figure,NaN,SM: D009151,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...",\nYou are an AI assistant responsible for vali...,\nValidate and correct relationships between c...,None,\nValidate and correct relationships between c...,\nValidate and correct relationships between c...
1,17524151,Effect of roxithromycin on SM cytotoxicity in ...,17524151_figure2_img1_1471-2121-8-17-2.jpg,1471-2121-8-17-2.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...",\nYou are an AI assistant responsible for vali...,\nValidate and correct relationships between c...,None,\nValidate and correct relationships between c...,\nValidate and correct relationships between c...
2,17524151,Anti-cytotoxicity effect of roxithromycin on S...,17524151_figure3_img1_1471-2121-8-17-3.jpg,1471-2121-8-17-3.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151\nCalcein A...,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...",\nYou are an AI assistant responsible for vali...,\nValidate and correct relationships between c...,None,\nValidate and correct relationships between c...,\nValidate and correct relationships between c...
3,17524151,Inhibition by roxithromycin of cytokine releas...,17524151_figure4_img1_1471-2121-8-17-4.jpg,1471-2121-8-17-4.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...",\nYou are an AI assistant responsible for vali...,\nValidate and correct relationships between c...,None,\nValidate and correct relationships between c...,\nValidate and correct relationships between c...
4,17524151,Inhibition by roxithromycin of cytokine mRNA e...,17524151_figure5_img1_1471-2121-8-17-5.jpg,1471-2121-8-17-5.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,NaN,None,"""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...","""chemicalName"",""relationship"",""geneName"",""gene...",\nYou are an AI assistant responsible for vali...,\nValidate and correct relationships between c...,None,\nValidate and correct relationships between c...,\nValidate and correct relationships between c...


In [57]:
# List of new columns to add
new_cols = [
    'deepseekV_gptE', 'deepseekV_claudeE',
    'gptV_deepseekE', 'gptV_claudeE',
    'claudeV_deepseekE', 'claudeV_gptE',
    'deepseekV_gpt4oE', 'claudeV_gpt4oE',
    'gpt4oV_deepseekE', 'gpt4oV_claudeE'
]

# Add each column with None as default
for col in new_cols:
    if col not in input.columns:
        input[col] = None

# input.to_csv(f"{input_data_path}/inputVal_V2.csv", index=False)

In [58]:
# Verify
input = pd.read_csv(f"{input_data_path}/inputVal_V2.csv")
input.columns

Index(['pmid', 'caption', 'filename', 'orginal_filename', 'type', 'gene_map',
       'chemical_map', 'system_prompt', 'main_prompt', 'section_type', 'text',
       'table_xml', 'sheet_name', 'claude_output', 'deepseek_output',
       'gpt_output', 'gpt4o_output', 'llama_output', 'deepseek_E_output',
       'gpt_E_output', 'claude_E_output', 'gpt4o_E_output',
       'system_prompt_val', 'main_prompt_val_gpt', 'main_prompt_val_deepseek',
       'main_prompt_val_claude', 'main_prompt_val_gpt4o', 'deepseekV_gptE',
       'deepseekV_claudeE', 'gptV_deepseekE', 'gptV_claudeE',
       'claudeV_deepseekE', 'claudeV_gptE', 'deepseekV_gpt4oE',
       'claudeV_gpt4oE', 'gpt4oV_deepseekE', 'gpt4oV_claudeE'],
      dtype='object')

In [59]:
def run_validation_models_deepseek(row, idx):
    """
    Calls DeepSeek validation on extracted prompts for a single row.
    Outputs:
        - deepseekV_claudeE
        - deepseekV_gptE
        - deepseekV_gpt4oE
    Skips FIG/FIG_TABLE rows (DeepSeek can't handle images).
    Skips EACH output individually if it already exists.
    """

    outputs = {}

    section_type = str(row.get("section_type", "")).strip().upper()
    is_fig = section_type in ["FIG", "FIG_TABLE"]

    print(f"\n🔹 Row {idx} | section_type={section_type}")

    # DeepSeek cannot handle images
    if is_fig:
        print("  ⏭ DeepSeek validation skipped (FIG/Figure type)")
        outputs["deepseekV_claudeE"] = row.get("deepseekV_claudeE")
        outputs["deepseekV_gptE"] = row.get("deepseekV_gptE")
        outputs["deepseekV_gpt4oE"] = row.get("deepseekV_gpt4oE")
        return pd.Series(outputs)

    system_prompt_val = row.get("system_prompt_val")

    # -------------------------
    # DeepSeek on Claude extraction
    # -------------------------
    if pd.notna(row.get("deepseekV_claudeE")):
        print("  ⏭ deepseekV_claudeE already exists. Skipping.")
        outputs["deepseekV_claudeE"] = row.get("deepseekV_claudeE")
    else:
        main_prompt_val_claude = row.get("main_prompt_val_claude")
        if main_prompt_val_claude and system_prompt_val:
            print("  ▶ DeepSeek on Claude extracted CSV...")
            try:
                outputs["deepseekV_claudeE"] = call_deepseek(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_claude,
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["deepseekV_claudeE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["deepseekV_claudeE"] = None

    # -------------------------
    # DeepSeek on GPT extraction
    # -------------------------
    if pd.notna(row.get("deepseekV_gptE")):
        print("  ⏭ deepseekV_gptE already exists. Skipping.")
        outputs["deepseekV_gptE"] = row.get("deepseekV_gptE")
    else:
        main_prompt_val_gpt = row.get("main_prompt_val_gpt")
        if main_prompt_val_gpt and system_prompt_val:
            print("  ▶ DeepSeek on GPT extracted CSV...")
            try:
                outputs["deepseekV_gptE"] = call_deepseek(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_gpt,
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["deepseekV_gptE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["deepseekV_gptE"] = None

    # -------------------------
    # DeepSeek on GPT-4o extraction
    # -------------------------
    if pd.notna(row.get("deepseekV_gpt4oE")):
        print("  ⏭ deepseekV_gpt4oE already exists. Skipping.")
        outputs["deepseekV_gpt4oE"] = row.get("deepseekV_gpt4oE")
    else:
        main_prompt_val_gpt4o = row.get("main_prompt_val_gpt4o")
        if main_prompt_val_gpt4o and system_prompt_val:
            print("  ▶ DeepSeek on GPT4o extracted CSV...")
            try:
                outputs["deepseekV_gpt4oE"] = call_deepseek(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_gpt4o,
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["deepseekV_gpt4oE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["deepseekV_gpt4oE"] = None

    print(f"✅ Row {idx} validation complete")
    return pd.Series(outputs)


In [60]:
for idx, row in input.iterrows():
    outputs = run_validation_models_deepseek(row, idx)
    for col, val in outputs.items():
        input.at[idx, col] = val

    # Save every 50 rows
    if idx % 50 == 0:
        input.to_csv(f"{output_data_path}/validation_checkpoint_v2.csv", index=False)

# Final save
input.to_csv(f"{output_data_path}/validation_checkpoint_v2.csv", index=False)


🔹 Row 0 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 1 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 2 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 3 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 4 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 5 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 6 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 7 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 8 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 9 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 10 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 11 | section_type=FIG
  ⏭ DeepSeek validation skipped (FIG/Figure type)

🔹 Row 12 | section_type=FIG
  ⏭ DeepSeek validation skipped (

/var/folders/kc/llqm4_p10bxdm821sz789bmh0000gn/T/ipykernel_56114/745629264.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '"explanation","chemicalName","chemicalID","relationship","geneName","geneID","confidence","source","agreement","reasoning"
"LPS synergizes with IFN-gamma to induce MIG expression in macrophages, direct finding by authors","LPS","D008070","increase expression","MIG","17329","3","the Toll-Like Receptor-4 (TLR-4) ligand LPS synergizes with IFN-gamma to induce MIG expression in macrophages","1",""
"LPS induces NF-kappaB which contributes to synergy on MIG expression, direct finding by authors","LPS","D008070","increase activity","NF-kappaB","18033","3","the marked synergy between LPS and IFN-gamma on MIG mRNA expression in mouse macrophages is a result of LPS-induced NF-kappaB and IFN-gamma-induced STAT","1",""
"zymosan synergizes with IFN-gamma to induce MIG in an NF-kappaB dependen


🔹 Row 151 | section_type=DISCUSS
  ▶ DeepSeek on Claude extracted CSV...
    ✓ Done
  ▶ DeepSeek on GPT extracted CSV...
    ✓ Done
  ▶ DeepSeek on GPT4o extracted CSV...
    ✓ Done
✅ Row 151 validation complete

🔹 Row 152 | section_type=DISCUSS
  ▶ DeepSeek on Claude extracted CSV...
    ✓ Done
  ▶ DeepSeek on GPT extracted CSV...
    ✓ Done
  ▶ DeepSeek on GPT4o extracted CSV...
    ✓ Done
✅ Row 152 validation complete

🔹 Row 153 | section_type=DISCUSS
  ▶ DeepSeek on Claude extracted CSV...
    ✓ Done
  ▶ DeepSeek on GPT extracted CSV...
    ✓ Done
  ▶ DeepSeek on GPT4o extracted CSV...
    ✓ Done
✅ Row 153 validation complete

🔹 Row 154 | section_type=DISCUSS
  ▶ DeepSeek on Claude extracted CSV...
    ✓ Done
  ▶ DeepSeek on GPT extracted CSV...
    ✓ Done
  ▶ DeepSeek on GPT4o extracted CSV...
    ✓ Done
✅ Row 154 validation complete

🔹 Row 155 | section_type=DISCUSS
  ▶ DeepSeek on Claude extracted CSV...
    ✓ Done
  ▶ DeepSeek on GPT extracted CSV...
    ✓ Done
  ▶ DeepSeek on

In [250]:
input.tail()

,pmid,caption,filename,orginal_filename,type,gene_map,chemical_map,system_prompt,main_prompt,section_type,...,claudeV_deepseekE,claudeV_gptE,gpt4o_output,llama_output,gpt4o_E_output,main_prompt_val_gpt4o,deepseekV_gpt4oE,claudeV_gpt4oE,gpt4oV_deepseekE,gpt4oV_claudeE
576,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen...",ERROR: An error occurred (UnrecognizedClientEx...,"chemicalName,relationship,geneName,geneSymbol\...",\nValidate and correct relationships between c...,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN
577,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen...",ERROR: An error occurred (UnrecognizedClientEx...,"chemicalName,relationship,geneName,geneSymbol\...",\nValidate and correct relationships between c...,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN
578,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen...","```\nchemicalName,relationship,geneName,geneSy...",ERROR: An error occurred (UnrecognizedClientEx...,"chemicalName,relationship,geneName,geneSymbol\...",\nValidate and correct relationships between c...,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN
579,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen...",ERROR: An error occurred (UnrecognizedClientEx...,"chemicalName,relationship,geneName,geneSymbol\...",\nValidate and correct relationships between c...,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN
580,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen...",ERROR: An error occurred (UnrecognizedClientEx...,"chemicalName,relationship,geneName,geneSymbol\...",\nValidate and correct relationships between c...,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN


In [62]:
def run_gpt_validation_on_row(row, idx, figures_path):
    """
    Calls GPT on extracted validation prompts for a single row.
    Outputs are saved to new columns: gptV_deepseekE, gptV_claudeE.
    GPT can handle all section types, including FIG/FIG_TABLE.
    If the section is a figure, passes the image path to the model.
    """
    outputs = {}

    section_type = str(row.get("section_type", "")).strip().upper()
    is_fig = section_type in ["FIG", "FIG_TABLE"]

    # Skip if validation outputs already exist
    if pd.notna(row.get("gptV_deepseekE")) and pd.notna(row.get("gptV_claudeE")):
        print(f"Row {idx} ✅ GPT validation outputs already exist. Skipping.")
        return pd.Series(outputs)

    print(f"\n🔹 Row {idx} | section_type={section_type}")

    system_prompt_val = row.get("system_prompt_val")

    # Prepare image path if this is a figure
    image_path = (
        f"{figures_path}/{row['filename']}"
        if is_fig and pd.notna(row.get("filename"))
        else None
    )

    # -------------------------
    # GPT on main_prompt_val_deepseek
    # -------------------------
    main_prompt_val_deepseek = row.get("main_prompt_val_deepseek")
    if main_prompt_val_deepseek and system_prompt_val and not is_fig:
        print("  ▶ GPT on DeepSeek extracted CSV...")
        try:
            outputs["gptV_deepseekE"] = call_gpt(
                system_prompt=system_prompt_val,
                main_prompt=main_prompt_val_deepseek,
                image_path=image_path
            )
            print("    ✓ Done")
        except Exception as e:
            print(f"    ✗ Failed: {e}")
            outputs["gptV_deepseekE"] = None
    else:
        outputs["gptV_deepseekE"] = None
        print("    ⏭ Skipped (no prompt)")

    # -------------------------
    # GPT on main_prompt_val_claude
    # -------------------------
    main_prompt_val_claude = row.get("main_prompt_val_claude")
    if main_prompt_val_claude and system_prompt_val:
        print("  ▶ GPT on Claude extracted CSV...")
        try:
            outputs["gptV_claudeE"] = call_gpt(
                system_prompt=system_prompt_val,
                main_prompt=main_prompt_val_claude,
                image_path=image_path
            )
            print("    ✓ Done")
        except Exception as e:
            print(f"    ✗ Failed: {e}")
            outputs["gptV_claudeE"] = None
    else:
        outputs["gptV_claudeE"] = None
        print("    ⏭ Skipped (no prompt)")

    print(f"✅ Row {idx} GPT validation complete")
    return pd.Series(outputs)


In [63]:
for idx, row in input.iterrows():
    outputs = run_gpt_validation_on_row(row, idx, figures_path)
    for col, val in outputs.items():
        input.at[idx, col] = val

    # Save every 50 rows
    if idx % 50 == 0:
        input.to_csv(f"{output_data_path}/validation_checkpoint_V2.csv", index=False)

# Final save
input.to_csv(f"{output_data_path}/validation_checkpoint_V2.csv", index=False)


🔹 Row 0 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 0 GPT validation complete


/var/folders/kc/llqm4_p10bxdm821sz789bmh0000gn/T/ipykernel_56114/920014255.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '"chemicalName","relationship","geneName","geneSymbol","confidence","agreement","reasoning"
"Sulfur Mustard","decrease abundance","-","-","3","1",""' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  input.at[idx, col] = val



🔹 Row 1 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 1 GPT validation complete

🔹 Row 2 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 2 GPT validation complete

🔹 Row 3 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 3 GPT validation complete

🔹 Row 4 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 4 GPT validation complete

🔹 Row 5 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 5 GPT validation complete

🔹 Row 6 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 6 GPT validation complete

🔹 Row 7 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 7 GPT validation complete

🔹 Row 8 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT on Claude extr

/var/folders/kc/llqm4_p10bxdm821sz789bmh0000gn/T/ipykernel_56114/920014255.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '"explanation","chemicalName","chemicalID","relationship","geneName","geneID","confidence","source","agreement","reasoning"
"LPS synergizes with IFN-gamma to induce MIG expression, cited from other work","LPS","D008070","increase expression","MIG","17329","2","Although it is known that the Toll-Like Receptor-4 (TLR-4) ligand LPS synergizes with IFN-gamma to induce MIG expression in macrophages","0","Relationship should be removed - comes from previous studies/other works"
"LPS induces NF-kappaB, direct finding by authors","LPS","D008070","increase activity","NF-kappaB","18033","3","We determined that the marked synergy between LPS and IFN-gamma on MIG mRNA expression in mouse macrophages is a result of LPS-induced NF-kappaB","1",""
"zymosan synergizes with IFN-gamma to induce MIG 


🔹 Row 151 | section_type=DISCUSS
  ▶ GPT on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 151 GPT validation complete

🔹 Row 152 | section_type=DISCUSS
  ▶ GPT on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 152 GPT validation complete

🔹 Row 153 | section_type=DISCUSS
  ▶ GPT on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 153 GPT validation complete

🔹 Row 154 | section_type=DISCUSS
  ▶ GPT on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 154 GPT validation complete

🔹 Row 155 | section_type=DISCUSS
  ▶ GPT on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 155 GPT validation complete

🔹 Row 156 | section_type=RESULTS
  ▶ GPT on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT on Claude extracted CSV...
    ✓ Done
✅ Row 156 GPT validation complete

🔹 Row 157 | section_type=RE

In [64]:
input.tail()

,pmid,caption,filename,orginal_filename,type,gene_map,chemical_map,system_prompt,main_prompt,section_type,...,deepseekV_gptE,deepseekV_claudeE,gptV_deepseekE,gptV_claudeE,claudeV_deepseekE,claudeV_gptE,deepseekV_gpt4oE,claudeV_gpt4oE,gpt4oV_deepseekE,gpt4oV_claudeE
576,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","```\nchemicalName,relationship,geneName,geneSy...",NaN,NaN,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN
577,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN
578,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN
579,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","```\nchemicalName,relationship,geneName,geneSy...",NaN,NaN,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN
580,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,"chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN,NaN


In [65]:
def run_claude_validation_on_row(row, idx, figures_path):
    """
    Calls Claude on validation prompts for a single row.

    Outputs:
        - claudeV_deepseekE
        - claudeV_gptE
        - claudeV_gpt4oE

    Claude can handle all section types including FIG/FIG_TABLE.
    Skips EACH output individually if it already exists.
    """

    outputs = {}

    section_type = str(row.get("section_type", "")).strip().upper()
    is_fig = section_type in ["FIG", "FIG_TABLE"]

    print(f"\n🔹 Row {idx} | section_type={section_type}")

    system_prompt_val = row.get("system_prompt_val")

    # Prepare image path if figure
    image_path = (
        f"{figures_path}/{row['filename']}"
        if is_fig and pd.notna(row.get("filename"))
        else None
    )

    # -------------------------
    # Claude on DeepSeek extraction
    # -------------------------
    if pd.notna(row.get("claudeV_deepseekE")):
        print("  ⏭ claudeV_deepseekE already exists. Skipping.")
        outputs["claudeV_deepseekE"] = row.get("claudeV_deepseekE")
    else:
        main_prompt_val_deepseek = row.get("main_prompt_val_deepseek")
        if main_prompt_val_deepseek and system_prompt_val:
            print("  ▶ Claude on DeepSeek extracted CSV...")
            try:
                outputs["claudeV_deepseekE"] = call_claude(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_deepseek,
                    image_path=image_path
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["claudeV_deepseekE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["claudeV_deepseekE"] = None

    # -------------------------
    # Claude on GPT extraction
    # -------------------------
    if pd.notna(row.get("claudeV_gptE")):
        print("  ⏭ claudeV_gptE already exists. Skipping.")
        outputs["claudeV_gptE"] = row.get("claudeV_gptE")
    else:
        main_prompt_val_gpt = row.get("main_prompt_val_gpt")
        if main_prompt_val_gpt and system_prompt_val:
            print("  ▶ Claude on GPT extracted CSV...")
            try:
                outputs["claudeV_gptE"] = call_claude(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_gpt,
                    image_path=image_path
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["claudeV_gptE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["claudeV_gptE"] = None

    # -------------------------
    # Claude on GPT4o extraction
    # -------------------------
    if pd.notna(row.get("claudeV_gpt4oE")):
        print("  ⏭ claudeV_gpt4oE already exists. Skipping.")
        outputs["claudeV_gpt4oE"] = row.get("claudeV_gpt4oE")
    else:
        main_prompt_val_gpt4o = row.get("main_prompt_val_gpt4o")
        if main_prompt_val_gpt4o and system_prompt_val:
            print("  ▶ Claude on GPT4o extracted CSV...")
            try:
                outputs["claudeV_gpt4oE"] = call_claude(
                    system_prompt=system_prompt_val,
                    main_prompt=main_prompt_val_gpt4o,
                    image_path=image_path
                )
                print("    ✓ Done")
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                outputs["claudeV_gpt4oE"] = None
        else:
            print("    ⏭ Skipped (no prompt)")
            outputs["claudeV_gpt4oE"] = None

    print(f"✅ Row {idx} Claude validation complete")
    return pd.Series(outputs)


In [66]:
for idx, row in input.iterrows():
    outputs = run_claude_validation_on_row(row, idx, figures_path)
    for col, val in outputs.items():
        input.at[idx, col] = val

    # Save every 50 rows
    if idx % 50 == 0:
        input.to_csv(f"{output_data_path}/validation_checkpoint_V2.csv", index=False)

# Final save
input.to_csv(f"{output_data_path}/validation_checkpoint_V2.csv", index=False)


🔹 Row 0 | section_type=FIG
  ▶ Claude on DeepSeek extracted CSV...
    ✓ Done
  ▶ Claude on GPT extracted CSV...
    ✓ Done
  ▶ Claude on GPT4o extracted CSV...
    ✓ Done
✅ Row 0 Claude validation complete


/var/folders/kc/llqm4_p10bxdm821sz789bmh0000gn/T/ipykernel_56114/1128685629.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ERROR: Out of range float values are not JSON compliant' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  input.at[idx, col] = val
/var/folders/kc/llqm4_p10bxdm821sz789bmh0000gn/T/ipykernel_56114/1128685629.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '```
"chemicalName","relationship","geneName","geneSymbol","confidence","agreement","reasoning"
"Sulfur Mustard","decrease abundance","","","3","0","The relationship 'abundance' is a chemical-only relationship and cannot be used between a chemical and gene pair. Based on the image showing decreased cell viability (bar chart shows ~20% viability at 100 μM SM vs ~100% control), this should be 'decrease re


🔹 Row 1 | section_type=FIG
  ▶ Claude on DeepSeek extracted CSV...
    ✓ Done
  ▶ Claude on GPT extracted CSV...
    ✓ Done
  ▶ Claude on GPT4o extracted CSV...
    ✓ Done
✅ Row 1 Claude validation complete

🔹 Row 2 | section_type=FIG
  ▶ Claude on DeepSeek extracted CSV...
    ✓ Done
  ▶ Claude on GPT extracted CSV...
    ✓ Done
  ▶ Claude on GPT4o extracted CSV...
    ✓ Done
✅ Row 2 Claude validation complete

🔹 Row 3 | section_type=FIG
  ▶ Claude on DeepSeek extracted CSV...
    ✓ Done
  ▶ Claude on GPT extracted CSV...
    ✓ Done
  ▶ Claude on GPT4o extracted CSV...
    ✓ Done
✅ Row 3 Claude validation complete

🔹 Row 4 | section_type=FIG
  ▶ Claude on DeepSeek extracted CSV...
    ✓ Done
  ▶ Claude on GPT extracted CSV...
    ✓ Done
  ▶ Claude on GPT4o extracted CSV...
    ✓ Done
✅ Row 4 Claude validation complete

🔹 Row 5 | section_type=FIG
  ▶ Claude on DeepSeek extracted CSV...
    ✓ Done
  ▶ Claude on GPT extracted CSV...
    ✓ Done
  ▶ Claude on GPT4o extracted CSV...
    ✓ 

In [67]:
input.tail()

,pmid,caption,filename,orginal_filename,type,gene_map,chemical_map,system_prompt,main_prompt,section_type,...,deepseekV_gptE,deepseekV_claudeE,gptV_deepseekE,gptV_claudeE,claudeV_deepseekE,claudeV_gptE,deepseekV_gpt4oE,claudeV_gpt4oE,gpt4oV_deepseekE,gpt4oV_claudeE
576,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","```\nchemicalName,relationship,geneName,geneSy...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN
577,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN
578,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN
579,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","```\nchemicalName,relationship,geneName,geneSy...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN
580,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...",NaN,NaN


In [68]:
def run_gpt4o_validation_on_row(row, idx, figures_path):
    """
    Calls GPT on extracted validation prompts for a single row.
    Outputs are saved to new columns: gptV_deepseekE, gptV_claudeE.
    GPT can handle all section types, including FIG/FIG_TABLE.
    If the section is a figure, passes the image path to the model.
    """
    outputs = {}

    section_type = str(row.get("section_type", "")).strip().upper()
    is_fig = section_type in ["FIG", "FIG_TABLE"]

    # Skip if validation outputs already exist
    if pd.notna(row.get("gpt4oV_deepseekE")) and pd.notna(row.get("gpt4oV_claudeE")):
        print(f"Row {idx} ✅ GPT validation outputs already exist. Skipping.")
        return pd.Series(outputs)

    print(f"\n🔹 Row {idx} | section_type={section_type}")

    system_prompt_val = row.get("system_prompt_val")

    # Prepare image path if this is a figure
    image_path = (
        f"{figures_path}/{row['filename']}"
        if is_fig and pd.notna(row.get("filename"))
        else None
    )

    # -------------------------
    # GPT on main_prompt_val_deepseek
    # -------------------------
    main_prompt_val_deepseek = row.get("main_prompt_val_deepseek")
    if main_prompt_val_deepseek and system_prompt_val and not is_fig:
        print("  ▶ GPT4o on DeepSeek extracted CSV...")
        try:
            outputs["gpt4oV_deepseekE"] = call_gpt4o(
                system_prompt=system_prompt_val,
                main_prompt=main_prompt_val_deepseek,
                image_path=image_path
            )
            print("    ✓ Done")
        except Exception as e:
            print(f"    ✗ Failed: {e}")
            outputs["gpt4oV_deepseekE"] = None
    else:
        outputs["gpt4oV_deepseekE"] = None
        print("    ⏭ Skipped (no prompt)")

    # -------------------------
    # GPT on main_prompt_val_claude
    # -------------------------
    main_prompt_val_claude = row.get("main_prompt_val_claude")
    if main_prompt_val_claude and system_prompt_val:
        print("  ▶ GPT4o on Claude extracted CSV...")
        try:
            outputs["gpt4oV_claudeE"] = call_gpt4o(
                system_prompt=system_prompt_val,
                main_prompt=main_prompt_val_claude,
                image_path=image_path
            )
            print("    ✓ Done")
        except Exception as e:
            print(f"    ✗ Failed: {e}")
            outputs["gpt4oV_claudeE"] = None
    else:
        outputs["gpt4oV_claudeE"] = None
        print("    ⏭ Skipped (no prompt)")

    print(f"✅ Row {idx} GPT validation complete")
    return pd.Series(outputs)


In [69]:
for idx, row in input.iterrows():
    outputs = run_gpt4o_validation_on_row(row, idx, figures_path)
    for col, val in outputs.items():
        input.at[idx, col] = val

    # Save every 50 rows
    if idx % 50 == 0:
        input.to_csv(f"{output_data_path}/validation_checkpoint_V2.csv", index=False)

# Final save
input.to_csv(f"{output_data_path}/validation_checkpoint_V2.csv", index=False)


🔹 Row 0 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 0 GPT validation complete


/var/folders/kc/llqm4_p10bxdm821sz789bmh0000gn/T/ipykernel_56114/4109406193.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '```csv
"chemicalName","relationship","geneName","geneSymbol","confidence","agreement","reasoning"
"Sulfur Mustard","decrease viability","-","-","3","1",""
```' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  input.at[idx, col] = val



🔹 Row 1 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 1 GPT validation complete

🔹 Row 2 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 2 GPT validation complete

🔹 Row 3 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 3 GPT validation complete

🔹 Row 4 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 4 GPT validation complete

🔹 Row 5 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 5 GPT validation complete

🔹 Row 6 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 6 GPT validation complete

🔹 Row 7 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 7 GPT validation complete

🔹 Row 8 | section_type=FIG
    ⏭ Skipped (no prompt)
  ▶ GPT4

/var/folders/kc/llqm4_p10bxdm821sz789bmh0000gn/T/ipykernel_56114/4109406193.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '```csv
"explanation","chemicalName","chemicalID","relationship","geneName","geneID","confidence","source","agreement","reasoning"
"LPS synergizes with IFN-gamma to induce MIG expression, cited from other work","LPS","D008070","increase expression","MIG","17329","2","Although it is known that the Toll-Like Receptor-4 (TLR-4) ligand LPS synergizes with IFN-gamma to induce MIG expression in macrophages","0","Relationship should be removed - comes from previous studies/other works"
"LPS induces NF-kappaB, direct finding by authors","LPS","D008070","increase activity","NF-kappaB","18033","3","We determined that the marked synergy between LPS and IFN-gamma on MIG mRNA expression in mouse macrophages is a result of LPS-induced NF-kappaB","1",""
"zymosan synergizes with IFN-gamma to ind


🔹 Row 151 | section_type=DISCUSS
  ▶ GPT4o on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 151 GPT validation complete

🔹 Row 152 | section_type=DISCUSS
  ▶ GPT4o on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 152 GPT validation complete

🔹 Row 153 | section_type=DISCUSS
  ▶ GPT4o on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 153 GPT validation complete

🔹 Row 154 | section_type=DISCUSS
  ▶ GPT4o on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 154 GPT validation complete

🔹 Row 155 | section_type=DISCUSS
  ▶ GPT4o on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 155 GPT validation complete

🔹 Row 156 | section_type=RESULTS
  ▶ GPT4o on DeepSeek extracted CSV...
    ✓ Done
  ▶ GPT4o on Claude extracted CSV...
    ✓ Done
✅ Row 156 GPT validation complete

🔹 R

In [71]:
input.tail()

,pmid,caption,filename,orginal_filename,type,gene_map,chemical_map,system_prompt,main_prompt,section_type,...,deepseekV_gptE,deepseekV_claudeE,gptV_deepseekE,gptV_claudeE,claudeV_deepseekE,claudeV_gptE,deepseekV_gpt4oE,claudeV_gpt4oE,gpt4oV_deepseekE,gpt4oV_claudeE
576,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","```\nchemicalName,relationship,geneName,geneSy...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","```\nchemicalName,relationship,geneName,geneSy..."
577,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```csv\nchemicalName,relationship,geneName,gen...","```csv\nchemicalName,relationship,geneName,gen..."
578,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","```csv\nchemicalName,relationship,geneName,gen..."
579,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","```\nchemicalName,relationship,geneName,geneSy...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","```csv\nchemicalName,relationship,geneName,gen..."
580,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```\nchemicalName,relationship,geneName,geneSy...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","```csv\nchemicalName,relationship,geneName,gen...","```\nchemicalName,relationship,geneName,geneSy..."


In [72]:
def find_csv_blocks(text):
    pattern = r'(?:^[^\n,]+(?:,[^\n,]+)+\n)+(?:^[^\n,]+(?:,[^\n,]+)+$)'
    matches = re.findall(pattern, text, re.MULTILINE)
    return matches

def extract_csv(text):

    # ✅ Handle None / NaN / non-strings safely
    if text is None or pd.isna(text):
        return None

    if not isinstance(text, str):
        text = str(text)

    # STEP 1: Look inside ``` or ```csv blocks
    code_block_pattern = r"```(?:csv)?\s*\n(.*?)```"
    code_blocks = re.findall(code_block_pattern, text, re.DOTALL | re.IGNORECASE)

    csv_blocks = []

    for block in code_blocks:
        if "," in block and len(block.strip().splitlines()) >= 2:
            csv_blocks.append(block.strip())

    if csv_blocks:
        return "\n\n".join(csv_blocks)

    # STEP 2: fallback regex detection
    fallback = find_csv_blocks(text)
    if fallback:
        return "\n\n".join(fallback)

    return None

new_cols = [
    'deepseekV_gptE', 'deepseekV_claudeE',
    'gptV_deepseekE', 'gptV_claudeE',
    'claudeV_deepseekE', 'claudeV_gptE',
    'deepseekV_gpt4oE', 'claudeV_gpt4oE',
    'gpt4oV_deepseekE', 'gpt4oV_claudeE'
]

output_cols = [
    'deepseekV_gptE_output',
    'deepseekV_claudeE_output',
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'deepseekV_gpt4oE_output',
    'claudeV_gpt4oE_output',
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output'
]

for col in new_cols:
    new_col = col + "_output"
    print(f"Processing {col} → {new_col}")

    input[new_col] = input[col].apply(extract_csv)

input

Processing deepseekV_gptE → deepseekV_gptE_output
Processing deepseekV_claudeE → deepseekV_claudeE_output
Processing gptV_deepseekE → gptV_deepseekE_output
Processing gptV_claudeE → gptV_claudeE_output
Processing claudeV_deepseekE → claudeV_deepseekE_output
Processing claudeV_gptE → claudeV_gptE_output
Processing deepseekV_gpt4oE → deepseekV_gpt4oE_output
Processing claudeV_gpt4oE → claudeV_gpt4oE_output
Processing gpt4oV_deepseekE → gpt4oV_deepseekE_output
Processing gpt4oV_claudeE → gpt4oV_claudeE_output


,pmid,caption,filename,orginal_filename,type,gene_map,chemical_map,system_prompt,main_prompt,section_type,...,deepseekV_gptE_output,deepseekV_claudeE_output,gptV_deepseekE_output,gptV_claudeE_output,claudeV_deepseekE_output,claudeV_gptE_output,deepseekV_gpt4oE_output,claudeV_gpt4oE_output,gpt4oV_deepseekE_output,gpt4oV_claudeE_output
0,17524151,Cytotoxicity of SM on SAE and BTE cells . Cell...,17524151_figure1_img1_1471-2121-8-17-1.jpg,1471-2121-8-17-1.jpg,figure,NaN,SM: D009151,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,None,None,None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene..."
1,17524151,Effect of roxithromycin on SM cytotoxicity in ...,17524151_figure2_img1_1471-2121-8-17-2.jpg,1471-2121-8-17-2.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,None,None,None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene..."
2,17524151,Anti-cytotoxicity effect of roxithromycin on S...,17524151_figure3_img1_1471-2121-8-17-3.jpg,1471-2121-8-17-3.jpg,figure,NaN,roxithromycin: D015575\nSM: D009151\nCalcein A...,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,None,None,None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene..."
3,17524151,Inhibition by roxithromycin of cytokine releas...,17524151_figure4_img1_1471-2121-8-17-4.jpg,1471-2121-8-17-4.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,None,None,None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene..."
4,17524151,Inhibition by roxithromycin of cytokine mRNA e...,17524151_figure5_img1_1471-2121-8-17-5.jpg,1471-2121-8-17-5.jpg,figure,NaN,SM: D009151\nroxithromycin: D015575\nRXM: D015575,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,FIG,...,None,None,None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene...",None,"""chemicalName"",""relationship"",""geneName"",""gene..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
576,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...",None,None,"chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,...","chemicalName,relationship,geneName,geneSymbol,..."
577,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,NaN,NaN,You are an AI assistant specialized in extract...,### Extract Specific Relationships Between Che...,SUPPLEMENT,...,"c

In [73]:
input.to_csv(f"{output_data_path}/validation_v1.csv", index=False)

# Refinement

In [4]:
ctd_chemicals = pd.read_csv(f"{ctd_data_path}/CTD_chemicals.csv", skiprows = 27)
ctd_genes = pd.read_csv(f"{ctd_data_path}/CTD_genes.csv", skiprows = 27)

ctd_ixns = pd.read_csv(f"{ctd_data_path}/CTD_chem_gene_ixns.csv", skiprows = 27).drop(labels=0)
ctd_ixns["InteractionActionsList"] = ctd_ixns["InteractionActions"].apply(lambda x: [item.replace("^", " ") for item in x.split("|")])
ctd_ixns["pmcidsList"] = ctd_ixns["PubMedIDs"].apply(lambda x: [str(item) for item in x.split("|")])
ctd_ixns["GeneID"] = ctd_ixns["GeneID"].apply(int)
ctd_ixns["GeneID"] = ctd_ixns["GeneID"].apply(str)
ctd_ixns["ChemicalID"] = ctd_ixns["ChemicalID"].apply(str)

In [5]:
def repair_csv(val_str: str) -> str:
    """
    Repairs CSV strings where a free-text field (e.g. geneName) may contain
    commas, causing row field counts to exceed the header field count.

    Strategy: Instead of hardcoding column positions, this function:
      1. Parses the header to get the expected column count (N).
      2. For each malformed row (too many fields), finds the LONGEST
         contiguous run of 'ambiguous' tokens — i.e. the tokens that,
         when collapsed back into one comma-joined string, produce exactly
         N fields total.
      3. Identifies which column is the free-text one by scanning the header
         for known free-text field names (e.g. 'geneName', 'explanation').
         Falls back to trying every possible merge window if no known field found.

    This means it works correctly whether the schema is:
        chemicalName, chemicalID, relationship, geneName, geneID          (5 cols)
        chemicalName, relationship, geneName, geneSymbol, agreement, reasoning  (6 cols)
        ...or any other schema where exactly one field may contain commas.
    """

    # Known free-text fields that may contain commas, in priority order
    FREE_TEXT_FIELDS = [
        'genename', 'explanation', 'reasoning', 'description',
        'caption', 'text', 'name'
    ]

    # ✅ Extract CSV blocks from markdown code fences
    # If there are multiple ```csv blocks, keep only the last one
    if '```' in val_str:
        blocks = []
        current_block = []
        in_code_block = False
        
        for line in val_str.splitlines():
            stripped = line.strip()
            if stripped.startswith('```'):
                if in_code_block:
                    # End of block
                    if current_block:
                        blocks.append('\n'.join(current_block))
                    current_block = []
                    in_code_block = False
                else:
                    # Start of block
                    in_code_block = True
            elif in_code_block:
                current_block.append(line)
        
        # If there are blocks, use the last one; otherwise use original
        if blocks:
            val_str = blocks[-1].strip()

    lines = val_str.strip().splitlines()
    if not lines:
        return val_str

    # Parse header
    try:
        header_parts = next(csv.reader([lines[0]]))
    except Exception:
        return val_str

    expected = len(header_parts)

    # Find which column index is the free-text field
    free_text_idx = None
    header_lower = [h.strip().lower() for h in header_parts]
    for field in FREE_TEXT_FIELDS:
        if field in header_lower:
            free_text_idx = header_lower.index(field)
            break

    output = io.StringIO()
    writer = csv.writer(output, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(header_parts)

    for line in lines[1:]:
        if not line.strip():
            continue

        # Parse the line respecting existing quotes first
        try:
            parts = next(csv.reader([line]))
        except Exception:
            # If quoted parsing fails, fall back to raw split
            parts = line.split(',')

        if len(parts) == expected:
            writer.writerow(parts)
            continue

        if len(parts) < expected:
            # Under-count — write as-is, let pandas handle/warn
            writer.writerow(parts)
            continue

        # Too many fields — need to merge some tokens
        extra = len(parts) - expected  # how many extra commas exist

        if free_text_idx is not None:
            # We know which field to merge into: collapse `extra+1` tokens
            # starting at free_text_idx into a single value
            merge_count = extra + 1
            prefix = parts[:free_text_idx]
            merged = ','.join(parts[free_text_idx: free_text_idx + merge_count])
            suffix = parts[free_text_idx + merge_count:]
            repaired = prefix + [merged] + suffix
            if len(repaired) == expected:
                writer.writerow(repaired)
                continue

        # Fallback: try every possible merge window and pick the first
        # one that produces exactly `expected` fields
        repaired = None
        for start in range(len(parts) - extra):
            end = start + extra + 1  # merge tokens[start:end]
            candidate = parts[:start] + [','.join(parts[start:end])] + parts[end:]
            if len(candidate) == expected:
                repaired = candidate
                break

        if repaired:
            writer.writerow(repaired)
        else:
            # Can't repair — write raw and let pandas warn
            writer.writerow(parts)

    return output.getvalue()


# ── Rest of the pipeline (unchanged except repair_csv is now called) ──────────

output_cols = [
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output'
]

keep_cols = [
    'explanation',
    'chemicalName',
    'chemicalID',
    'relationship',
    'geneName',
    'geneID',
    'geneSymbol',
    'source',
    '_source_row',
    'agreement',
    'reasoning'
    # 'confidence'
]


def combine_csv_outputs(df, columns):
    all_dfs = {}
    extra_cols = [
        'pmid',
        'caption',
        'original_filename',
        'gene_map',
        'chemical_map',
        'section_type',
        'text',
        'table_xml'
    ]

    for col in columns:
        if col not in df.columns:
            continue

        col_dfs = []
        for idx, val in df[col].items():
            if val is None:
                continue
            if isinstance(val, pd.DataFrame):
                col_dfs.append(val)
                continue

            try:
                val_str = str(val).strip()
            except Exception:
                continue
            if not val_str:
                continue

            # ✅ Repair comma-in-free-text-field issues AND extract from code fences
            val_str = repair_csv(val_str)

            try:
                temp_df = pd.read_csv(
                    io.StringIO(val_str),
                    engine="python",
                    sep=",",
                    quotechar='"',
                    skipinitialspace=True,
                    on_bad_lines="warn"
                )
                temp_df["_source_row"] = idx
                for c in extra_cols:
                    temp_df[c] = df.at[idx, c] if c in df.columns else pd.NA
                col_dfs.append(temp_df)
            except Exception as e:
                print(f"⚠️ Failed parsing row {idx} column {col}: {e}")
                continue

        if col_dfs:
            combined_col_df = pd.concat(col_dfs, ignore_index=True, sort=False)
            for c in keep_cols:
                if c not in combined_col_df.columns:
                    combined_col_df[c] = pd.NA
            final_cols = ['_source_row'] + extra_cols + keep_cols
            combined_col_df = combined_col_df[[c for c in final_cols if c in combined_col_df.columns]]
            all_dfs[col] = combined_col_df
            print(f"done with {col} - total rows = {len(combined_col_df)}")
        else:
            print(f"done with {col} - no valid CSV")

    return all_dfs

## refinement code

In [6]:
def build_ctd_chemical_lookup(ctd_chemicals_df):
    chem_id_to_info = {}
    name_to_chem_id = {}

    # Make column names consistent
    ctd_chemicals_df = ctd_chemicals_df.rename(columns=lambda x: x.strip())

    for _, row in ctd_chemicals_df.iterrows():
        chem_name = str(row.get("# ChemicalName", "") or row.get("Chemical Name", "")).strip()
        chem_id = str(row.get("ChemicalID", "") or row.get("Chemical ID", "")).replace("MESH:", "").upper()
        
        # synonyms
        mesh_syn = str(row.get("MESHSynonyms", "")).split("|") if row.get("MESHSynonyms") else []
        ctd_syn = str(row.get("CTDCuratedSynonyms", "")).split("|") if row.get("CTDCuratedSynonyms") else []
        synonyms = [s.strip() for s in mesh_syn + ctd_syn if s.strip()]

        chem_id_to_info[chem_id] = {"chem_name": chem_name, "synonyms": synonyms}

        if chem_name:
            name_to_chem_id[chem_name.upper()] = chem_id

        for syn in synonyms:
            name_to_chem_id[syn.upper()] = chem_id

    return chem_id_to_info, name_to_chem_id



def normalize_gene_id(x):
    """Convert GeneID to clean string or None"""
    if pd.isna(x):
        return None
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return str(x).strip()

def build_ctd_gene_lookups(ctd_genes_df):
    """
    Build lookup dictionaries from CTD genes DataFrame.
    
    Returns:
      geneid_to_name:   {GeneID -> canonical GeneName}
      geneid_to_symbol:{GeneID -> GeneSymbol}
      name_to_geneid:  {UPPERCASE name/symbol/synonym -> GeneID}
      alt_to_geneid:   {AltGeneID -> GeneID}
    """
    geneid_to_name = {}
    geneid_to_symbol = {}
    name_to_geneid = {}
    alt_to_geneid = {}

    for _, row in ctd_genes_df.iterrows():
        gene_id = normalize_gene_id(row.get("GeneID"))
        if not gene_id:
            continue

        symbol = row.get("# GeneSymbol")
        # print(symbol)
        name = row.get("GeneName")
        synonyms = row.get("Synonyms")
        alt_ids = row.get("AltGeneIDs")

        # Canonical name: prefer GeneName over symbol
        canonical_name = name if isinstance(name, str) and name.strip() else symbol
        if not isinstance(canonical_name, str) or not canonical_name.strip():
            continue

        geneid_to_name[gene_id] = canonical_name
        geneid_to_symbol[gene_id] = symbol.strip() if isinstance(symbol, str) else None

        # Index symbol + name
        for n in [symbol, name]:
            if isinstance(n, str) and n.strip():
                name_to_geneid[n.upper()] = gene_id

        # Index synonyms
        if isinstance(synonyms, str) and synonyms.strip():
            for syn in synonyms.split("|"):
                syn = syn.strip()
                if syn:
                    name_to_geneid[syn.upper()] = gene_id

        # Index alternate GeneIDs
        if isinstance(alt_ids, str) and alt_ids.strip():
            for alt in alt_ids.split("|"):
                alt = normalize_gene_id(alt)
                if alt:
                    alt_to_geneid[alt] = gene_id

    return geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid


def update_chemical(df, chem_id_to_info, name_to_chem_id, verbose=True):
    """
    Update chemicalName and chemicalID efficiently using prebuilt lookup tables.
    Flags:
        chemicalFlag = 0 : found normally
        chemicalFlag = 1 : not found
    """
    df = df.copy()
    df['chemicalFlag'] = 0

    # Normalize chemicalID column
    df['chemicalID'] = df['chemicalID'].apply(
        lambda x: str(x).upper().replace("MESH:", "").strip() if pd.notna(x) else None
    )
    df['chemicalName'] = df['chemicalName'].fillna("").astype(str)

    # Precompute normalized name_to_chem_id dict
    norm_name_to_id = {
        k.upper().replace("-", "").replace(" ", ""): v
        for k, v in name_to_chem_id.items()
    }

    for i, row in df.iterrows():
        chem_orig = row['chemicalName']
        chemicalName = chem_orig.strip()
        chemicalID = row['chemicalID']
        chemical_found = False

        # 1️⃣ Exact match by chemicalID
        if chemicalID and chemicalID in chem_id_to_info:
            df.at[i, 'chemicalName'] = chem_id_to_info[chemicalID]['chem_name']
            if verbose:
                print(f"Row {i} - Exact ID match: {chem_id_to_info[chemicalID]['chem_name']}")
            chemical_found = True
            continue  # already found

        # 2️⃣ Exact match by normalized chemicalName
        query_upper = chemicalName.upper().replace("-", "").replace(" ", "")
        if query_upper in norm_name_to_id:
            matched_id = norm_name_to_id[query_upper]
            df.at[i, 'chemicalID'] = matched_id
            df.at[i, 'chemicalName'] = chem_id_to_info[matched_id]['chem_name']
            if verbose:
                print(f"Row {i} - Exact name match: {chem_id_to_info[matched_id]['chem_name']}")
            chemical_found = True
            continue

        # 3️⃣ Partial match
        for key_norm, cid in norm_name_to_id.items():
            if query_upper and query_upper in key_norm:
                df.at[i, 'chemicalID'] = cid
                df.at[i, 'chemicalName'] = chem_id_to_info[cid]['chem_name']
                chemical_found = True
                if verbose:
                    print(f"Row {i} - Partial match: {chem_id_to_info[cid]['chem_name']}")
                break

        # 4️⃣ If still not found
        if not chemical_found:
            df.at[i, 'chemicalFlag'] = 1
            if not df.at[i, 'chemicalName']:
                df.at[i, 'chemicalName'] = chem_orig
            if verbose:
                print(f"Row {i} - Not found in CTD chemicals")
                print(f"  chemicalName: {chem_orig}")
                print(f"  chemicalID:   {chemicalID}")
                print("-" * 40)

    return df

def update_gene(df, geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid, verbose=True):
    """
    Update geneName, geneID, and geneSymbol in the DataFrame using CTD gene lookups.

    Flags:
        geneFlag = 0 : found normally
        geneFlag = 1 : not found

    Additionally:
        oldGeneID = original geneID before normalization/update
    """
    df = df.copy()
    df["geneFlag"] = 0
    
    # ✅ Store original geneID before any modification
    df["oldGeneID"] = df["geneID"]
    df["oldGeneName"] = df["geneName"]
    df["oldGeneSymbol"] = df["geneSymbol"]

    for i, row in df.iterrows():
        original_geneID = row.get("geneID")
        geneID = normalize_gene_id(original_geneID)
        geneName = row.get("geneName")
        geneSymbol = row.get("geneSymbol", None)

        found = False

        # 1️⃣ Resolve by GeneID
        if geneID:
            if geneID in geneid_to_name:
                geneName = geneid_to_name[geneID]
                geneSymbol = geneid_to_symbol.get(geneID)
                found = True

            elif geneID in alt_to_geneid:
                parent_id = alt_to_geneid[geneID]
                geneID = parent_id
                geneName = geneid_to_name.get(parent_id)
                geneSymbol = geneid_to_symbol.get(parent_id)
                found = True
        # somewhere in here, geneSymbol is getting overwritten to geneName.
        
        # 2️⃣ Flags
        if not found:
            df.at[i, "geneFlag"] = 1
            if verbose:
                print(f"Row {i} not found in CTD genes")
                print(f"  geneName: {geneName}")
                print(f"  geneID:   {geneID}")
                print("-" * 40)
        else:
            df.at[i, "geneFlag"] = 0

        # ✅ Overwrite geneID (while oldGeneID is preserved)
        df.at[i, "geneName"] = geneName
        df.at[i, "geneID"] = geneID
        df.at[i, "geneSymbol"] = geneSymbol

    return df


In [7]:
def update_relationship(df, verbose=True):
    import re

    # Allowed CTD relationships
    relationships = [
        "cotreatment", "binding", "reaction", "activity", "localization", "expression", "abundance", 
        "mutagenesis", "response to chemical", "stability", "splicing", "folding", "transport", 
        "uptake", "import", "secretion", "export", "metabolic processing", "chemical synthesis", 
        "degradation", "acetylation", "acylation", "alkylation", "amination", "carbamoylation", 
        "carboxylation", "cleavage", "ethylation", "glycation", "glycosylation", "N-linked glycosylation", 
        "O-linked glycosylation", "glucuronidation", "hydrolysis", "hydroxylation", "lipidation", 
        "geranoylation", "farnesylation", "myristoylation", "palmitoylation", "prenylation", 
        "methylation", "nitrosation", "nucleotidylation", "oxidation", "phosphorylation", "sulfation", 
        "reduction", "ribosylation", "ADP-ribosylation", "ubiquitination", "sumoylation", 
        "glutathionylation", "polymerization"
    ]
    lower_relationships = {r.lower() for r in relationships}

    # Prefix mapping: canonical form
    prefix_map = {
        'increase': 'increases', 'increases': 'increases', 'increased': 'increases', 'increasing': 'increases',
        'decrease': 'decreases', 'decreases': 'decreases', 'decreased': 'decreases', 'decreasing': 'decreases',
        'affect': 'affects', 'affects': 'affects', 'affected': 'affects', 'affecting': 'affects'
    }

    # Prefix exceptions: always "affects"
    affects_required = ['binding', 'cotreatment']

    # Chemical-chemical only relationships
    chem_chem_only = ['abundance', 'response to chemical', 'chemical synthesis']

    # Inhibitory terms replaced with "decrease"
    inhibit_terms = ['inhibit', 'inhibits', 'inhibited', 'inhibiting', 'inhibition']

    df = df.copy()
    df['relationshipWarningFlag'] = 0
    df['prefixMissingFlag'] = 0  # NEW: initialize flag

    if 'relationship' not in df.columns:
        df['relationship'] = ''

    # Normalize text: lowercase, remove quotes, strip spaces
    rel_series = df['relationship'].astype(str).str.lower().str.replace('"', '').str.strip()
    rel_series = rel_series.apply(lambda x: re.sub(r'\s+', ' ', x))

    # Replace inhibitory terms with "decrease"
    for term in inhibit_terms:
        rel_series = rel_series.str.replace(rf'\b{term}\b', 'decrease', regex=True)

    # Normalize prefix and apply rules
    def normalize(rel):
        words = rel.split()
        prefix = words[0] if words and words[0] in prefix_map else None
        core = ' '.join(words[1:]) if prefix else rel

        # NEW: set prefixMissingFlag if no prefix
        if not prefix:
            df.at[df.index[rel_series[rel_series == rel].index[0]], 'prefixMissingFlag'] = 1

        # Prefix exceptions
        if core in affects_required:
            return f"affects {core}"

        # Apply canonical prefix if prefix exists
        if prefix:
            new_prefix = prefix_map.get(prefix, prefix)
            return f"{new_prefix} {core}" if core else new_prefix

        return core

    rel_series = rel_series.apply(normalize)

    # Flag invalid relationships
    def flag_invalid(rel):
        words = rel.split()
        prefix = words[0] if words and words[0] in prefix_map else None
        core = ' '.join(words[1:]) if prefix else rel

        # chem-chem only relationships always invalid
        if core in chem_chem_only:
            return 1

        # Invalid if core not in allowed relationships
        return 0 if core in lower_relationships else 1

    def flag_prefix(rel):
        words = rel.split()
        prefix = words[0] if words else None
        valid_prefixes = {'increase', 'increases', 'decrease', 'decreases', 'affect', 'affects'}
    
        # Flag if no valid prefix
        if prefix not in valid_prefixes:
            return 1
    
        core = ' '.join(words[1:])
    
        # chem-chem only relationships always invalid
        if core in chem_chem_only:
            return 1
    
        # Invalid if core not in allowed relationships
        return 0 if core in lower_relationships else 1

    df['relationship'] = rel_series
    df['relationshipWarningFlag'] = rel_series.apply(flag_invalid)
    df['relationshipWarningFlag'] = rel_series.apply(flag_prefix)

    if verbose:
        invalid_df = df[df['relationshipWarningFlag'] == 1]
        n_invalid = len(invalid_df)
        print(f"{n_invalid} invalid relationships detected.")
        if n_invalid > 0:
            cols_to_show = ['relationship', 'relationshipWarningFlag', 'prefixMissingFlag']
            for col in ['chemicalID', 'geneID', 'chemicalFlag', 'geneFlag']:
                if col in df.columns:
                    cols_to_show.insert(0, col)
            print(invalid_df[cols_to_show].head(20))

    return df


In [8]:
import re
import numpy as np
import pandas as pd
from typing import Optional


# ---------------------------------------------------------------------------
# Small pure helpers
# ---------------------------------------------------------------------------

def _symbol_numeric_suffix(symbol) -> Optional[str]:
    """
    Extract the portion of a GeneSymbol starting from its first digit.
        FKBP1A -> "1A"
        IL6    -> "6"
        TP53   -> "53"
    Returns None if no digit is found or input is NaN.
    """
    if pd.isna(symbol):
        return None
    m = re.search(r"\d.*", str(symbol))
    return m.group(0) if m else None


def _tokenize(name) -> list:
    """Split a (already-cleaned) name string into lowercase tokens."""
    if pd.isna(name) or not str(name).strip():
        return []
    return str(name).strip().lower().split()


def _remove_parentheses(name) -> str:
    """
    Strip parenthesised substrings (including the parentheses themselves).
        "glyceraldehyde-3-phosphate dehydrogenase (GAPDH)"
        -> "glyceraldehyde-3-phosphate dehydrogenase"
    """
    if pd.isna(name):
        return ""
    return re.sub(r"\s*\(.*?\)", "", str(name)).strip()


def _suffix_in_tokens(suffix: Optional[str], tokens: list) -> bool:
    """
    Return True when *suffix* (e.g. "53") is represented in *tokens*.

    Single-token names  : suffix must appear *anywhere* in the token.
    Multi-token names   : at least one token must *end with* suffix.
    """
    if not suffix:
        return False
    suffix = suffix.lower()
    if len(tokens) == 1:
        return suffix in tokens[0]
    return any(tok.endswith(suffix) for tok in tokens)


def _normalize(s) -> str:
    """Lowercase, strip, coerce NaN -> empty string."""
    if pd.isna(s):
        return ""
    return str(s).strip().lower()


def _alt_count(alt_ids_str) -> int:
    """Count non-empty pipe-separated alternative gene IDs."""
    return len([a for a in str(alt_ids_str).split("|") if a.strip()])


def _symbol_is_missing(symbol) -> bool:
    """Return True if a geneSymbol value should be treated as absent."""
    if pd.isna(symbol):
        return True
    return str(symbol).strip() in ("", "-")


def _strip_dot_suffix(symbol) -> Optional[str]:
    """
    If a symbol contains a dot, return the portion before it.
        BRCA1.2  -> "BRCA1"
        TP53.1   -> "TP53"
        BRCA1    -> None  (no dot present)
    """
    if pd.isna(symbol):
        return None
    s = str(symbol).strip()
    if "." in s:
        return s.split(".")[0]
    return None


# ---------------------------------------------------------------------------
# CTD pre-processing  (call once, reuse across batches)
# ---------------------------------------------------------------------------

def prepare_ctd(ctd_genes: pd.DataFrame) -> dict:
    """
    Build fast lookup structures from the CTD gene table.

    Returns a dict with:
        "df"           : cleaned CTD dataframe with helper columns
        "id_index"     : {gene_id_str          -> list[row_index]}
        "symbol_index" : {normalised_symbol    -> list[row_index]}
        "name_index"   : {normalised_gene_name -> list[row_index]}
        "synonym_index": {normalised_synonym   -> list[row_index]}
    """
    ctd = ctd_genes.copy()
    ctd["GeneID"]       = ctd["GeneID"].astype(str)
    ctd["_symbol_norm"] = ctd["# GeneSymbol"].apply(_normalize)
    ctd["_name_norm"]   = ctd["GeneName"].apply(_normalize)
    ctd["_alt_count"]   = ctd["AltGeneIDs"].fillna("").apply(_alt_count)
    ctd["_syns"]        = ctd["Synonyms"].fillna("").str.lower().str.split("|")

    id_index: dict = {}
    for i, gid in ctd["GeneID"].items():
        if gid:
            id_index.setdefault(gid, []).append(i)

    symbol_index: dict = {}
    for i, sym in ctd["_symbol_norm"].items():
        if sym:
            symbol_index.setdefault(sym, []).append(i)

    name_index: dict = {}
    for i, name in ctd["_name_norm"].items():
        if name:
            name_index.setdefault(name, []).append(i)

    synonym_index: dict = {}
    for i, syns in ctd["_syns"].items():
        for s in syns:
            s = s.strip()
            if s:
                synonym_index.setdefault(s, []).append(i)

    return {
        "df": ctd,
        "id_index": id_index,
        "symbol_index": symbol_index,
        "name_index": name_index,
        "synonym_index": synonym_index,
    }


# ---------------------------------------------------------------------------
# Per-row resolution helpers
# ---------------------------------------------------------------------------

def _resolve_by_id(gene_id: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 1: exact GeneID match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["id_index"].get(gene_id, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_symbol(symbol_norm: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 2: exact normalised GeneSymbol match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["symbol_index"].get(symbol_norm, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_name(name_norm: str, name_tokens: list, ctd_lookup: dict) -> Optional[pd.Series]:
    """
    Priority 3: match via GeneName or Synonyms with alt_count tiebreaking.

    Cases
    -----
    1. Only GeneName hits  : prefer suffix-matching candidates; tiebreak by alt_count
    2. Both hits           : compare total alt_count of each group; pick winner's best
    3. Only Synonym hits   : pick highest alt_count
    """
    if not name_norm:
        return None

    ctd_df        = ctd_lookup["df"]
    name_index    = ctd_lookup["name_index"]
    synonym_index = ctd_lookup["synonym_index"]

    gn_rows  = ctd_df.loc[name_index.get(name_norm, [])]
    syn_rows = ctd_df.loc[synonym_index.get(name_norm, [])]

    # Deduplicate: remove synonym hits already present in gene-name hits
    if not gn_rows.empty and not syn_rows.empty:
        syn_rows = syn_rows[~syn_rows.index.isin(gn_rows.index)]

    # ---- Case 1: only GeneName hits ----
    if not gn_rows.empty and syn_rows.empty:
        suffix_matches = [
            r for _, r in gn_rows.iterrows()
            if _suffix_in_tokens(_symbol_numeric_suffix(r["# GeneSymbol"]), name_tokens)
        ]
        best_df = pd.DataFrame(suffix_matches) if suffix_matches else gn_rows
        return best_df.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 2: both hits ----
    if not gn_rows.empty and not syn_rows.empty:
        if gn_rows["_alt_count"].sum() >= syn_rows["_alt_count"].sum():
            return gn_rows.sort_values("_alt_count", ascending=False).iloc[0]
        else:
            return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 3: only Synonym hits ----
    if not syn_rows.empty:
        return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    return None


# ---------------------------------------------------------------------------
# Dot-symbol resolution helper
# ---------------------------------------------------------------------------

def _apply_dot_check(best: pd.Series, ctd_lookup: dict) -> pd.Series:
    """
    If the resolved symbol contains a dot (e.g. BRCA1.2), strip the suffix
    and re-look up the base symbol in CTD to get the correct GeneID/GeneName.
    Returns the corrected row, or the original row with the dot stripped if
    no CTD entry is found for the base.
    """
    base = _strip_dot_suffix(best["# GeneSymbol"])
    if base is None:
        return best  # no dot, nothing to do

    hit = _resolve_by_symbol(_normalize(base), ctd_lookup)
    if hit is not None:
        return hit

    # No CTD entry for base symbol — strip the dot suffix in-place
    best = best.copy()
    best["# GeneSymbol"] = base
    return best


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------
def update_gene_symbol_(
    df: pd.DataFrame,
    ctd_genes: pd.DataFrame,
    ctd_lookup: Optional[dict] = None,
) -> pd.DataFrame:

    df = df.copy()

    df["oldGeneID"]     = df.get("geneID")
    df["oldGeneName"]   = df.get("geneName")
    df["oldGeneSymbol"] = df.get("geneSymbol")

    # ✅ Treat "-", " -", "- " etc. as missing across key gene columns
    for col in ["geneID", "geneSymbol", "geneName"]:
        if col in df.columns:
            df[col] = df[col].replace(r"^\s*-\s*$", "", regex=True).fillna("")

    if ctd_lookup is None:
        ctd_lookup = prepare_ctd(ctd_genes)

    # Vectorised pre-normalisation
    df["_name_norm"] = (
        df.get("geneName", pd.Series(dtype=str))
        .apply(_remove_parentheses)
        .apply(_normalize)
    )
    df["_symbol_norm"] = (
        df.get("geneSymbol", pd.Series(dtype=str))
        .apply(_normalize)
    )

    flags       = np.full(len(df), 2, dtype=int)
    new_symbols = df["geneSymbol"].copy() if "geneSymbol" in df.columns else pd.Series(index=df.index, dtype=str)
    new_ids     = df["geneID"].astype(str).copy()
    new_names   = df["geneName"].copy()   if "geneName"   in df.columns else pd.Series(index=df.index, dtype=str)

    for pos, (idx, row) in enumerate(df.iterrows()):
        raw_symbol  = row.get("geneSymbol")
        gene_id     = str(row.get("geneID", "")).strip()
        symbol_norm = row["_symbol_norm"]
        name_norm   = row["_name_norm"]
        name_tokens = _tokenize(name_norm)

        # Already valid symbol
        if not _symbol_is_missing(raw_symbol):
            if not gene_id:  # symbol present but ID missing — try to resolve ID from symbol
                best = _resolve_by_symbol(symbol_norm, ctd_lookup)
                if best is not None:
                    best = _apply_dot_check(best, ctd_lookup)
                    new_symbols.iloc[pos] = best["# GeneSymbol"]
                    new_ids.iloc[pos]     = best["GeneID"]
                    new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue
    
        # Priority 1: GeneID
        if gene_id:
            best = _resolve_by_id(gene_id, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 2: GeneSymbol
        if symbol_norm:
            best = _resolve_by_symbol(symbol_norm, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 3a: treat geneName as a symbol
        best = _resolve_by_symbol(name_norm, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue

        # Priority 3b: GeneName / Synonym
        best = _resolve_by_name(name_norm, name_tokens, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0

    df["geneSymbol"]     = new_symbols.values
    df["geneID"]         = new_ids.values
    df["geneName"]       = new_names.values
    df["geneSymbolFlag"] = flags

    df.drop(columns=["_name_norm", "_symbol_norm"], inplace=True)
    return df


## refinement calls

In [9]:
chem_id_to_info, name_to_chem_id = build_ctd_chemical_lookup(ctd_chemicals)
geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid = build_ctd_gene_lookups(ctd_genes)

In [10]:
def refine(df, ctd_chemicals, ctd_genes):
    updated_df = update_chemical(
        df=df, 
        chem_id_to_info=chem_id_to_info, 
        name_to_chem_id=name_to_chem_id,
        verbose=False
    )

    updated_df = update_gene(
        updated_df,
        geneid_to_name,
        geneid_to_symbol,
        name_to_geneid,
        alt_to_geneid,
        verbose=False
    )

    updated_df = update_gene_symbol_(
        updated_df,
        ctd_genes
    )

    # call again to check if gene is in CTD or not
    # updated_df = check_gene_flag(updated_df, geneid_to_name, alt_to_geneid)

    updated_df = update_relationship(updated_df, verbose = False)
    return updated_df

In [11]:
import os

output_cols = [
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',
    
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output'
]

data = pd.read_csv(f"{output_data_path}/validation_v1.csv")
data.columns

combined_df = combine_csv_outputs(data, output_cols)

for col in output_cols:
    # output_file = f'{output_data_path}/refined_V1/{col}.csv'
    output_file = f'{output_data_path}/temp_V1/{col}.csv'

    if os.path.exists(output_file):
        print(f'Skipping {col}, file already exists.')
        continue

    print(f'Starting: {col}')
    temp_df = refine(combined_df[col], ctd_chemicals, ctd_genes)
    temp_df.to_csv(output_file, index=False)
    print(f'Finished: {col}')


done with deepseek_E_output - total rows = 2409
done with gpt_E_output - total rows = 3670
done with claude_E_output - total rows = 3064
done with gpt4o_E_output - total rows = 2629
done with deepseekV_gptE_output - total rows = 1669
done with deepseekV_gpt4oE_output - total rows = 1184
done with deepseekV_claudeE_output - total rows = 1145
done with gptV_deepseekE_output - total rows = 2438
done with gptV_claudeE_output - total rows = 3070
done with claudeV_deepseekE_output - total rows = 2437
done with claudeV_gptE_output - total rows = 3665
done with claudeV_gpt4oE_output - total rows = 2653
done with gpt4oV_deepseekE_output - total rows = 2411
done with gpt4oV_claudeE_output - total rows = 3060
Skipping deepseek_E_output, file already exists.
Skipping gpt_E_output, file already exists.
Skipping claude_E_output, file already exists.
Skipping gpt4o_E_output, file already exists.
Skipping deepseekV_gptE_output, file already exists.
Skipping deepseekV_gpt4oE_output, file already exists

# CTD Match

In [12]:
# ============================================================
# DROP DUPLICATES FUNCTIONS
# ============================================================

def drop_duplicates_(df): # drop duplicates globally with normalized relationship
    pmids = [
        15953866, 17118201, 17524151, 18299717, 19114014, 22110480, 22228119, 22571318,
        22747577, 22808131, 23724058, 24169358, 24403256, 24413757, 24664296, 25351418,
        25361045, 25859307, 25923732, 26039251, 26714306, 28285367, 28593498
    ]
    df = df.copy()
    df = df[df['pmid'].isin(pmids)]
    
    # Normalize relationship (same as in match function)
    def normalize_relationship(s):
        if pd.isna(s):
            return s
        s = str(s).lower().strip()
        s = s.replace("^", " ")
        s = s.replace("increases", "increase")
        s = s.replace("decreases", "decrease")
        s = s.replace("affects", "affect")
        s = " ".join(s.split())
        return s
    
    df['relationship_norm'] = df['relationship'].apply(normalize_relationship)
    
    # Build gene_key (same as in match function)
    df["geneSymbol_clean"] = (
        df["geneSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\..+$", "", regex=True)
    )
    
    def make_gene_key(row):
        sym = row["geneSymbol_clean"]
        if pd.notna(sym) and sym not in ("", "NAN", "NA"):
            return sym
        if pd.notna(row.get("geneID")):
            try:
                return str(int(row["geneID"]))
            except:
                return pd.NA
        return pd.NA
    
    df["gene_key"] = df.apply(make_gene_key, axis=1)
    
    # Section priority
    section_priority = {
        "TABLE": 0,
        "FIG_TABLE": 1,
        "FIG": 2,
        "SUPPLEMENT": 3,
        "TITLE": 4,
        "ABSTRACT": 5,
        "RESULTS": 6,
        "DISCUSS": 7,
        "METHODS": 8,
        "INTRO": 8,
        "CONCL": 8
    }
    
    df['_section_priority'] = df['section_type'].map(section_priority).fillna(100)
    df = df.reset_index(drop=True)
    df['_row_order'] = df.index
    
    # Sort by priority, then original row order
    df = df.sort_values(['_section_priority', '_row_order'], ascending=[True, True])
    
    # Drop duplicates using NORMALIZED MERGE KEYS
    df = df.drop_duplicates(
        subset=['pmid', 'gene_key', 'chemicalID', 'relationship_norm'], 
        keep='first'
    )

    df = df.drop_duplicates(
        subset=['pmid', 'gene_key', 'chemicalID', 'relationship_norm'], 
    )
    
    # KEEP relationship_norm and gene_key for downstream use
    # ONLY drop temporary columns
    df = df.drop(columns=['_section_priority', '_row_order', 'geneSymbol_clean'])
    
    return df


def drop_duplicates_section(df): # drop duplicates within each section with normalized relationship
    pmids = [
        15953866, 17118201, 17524151, 18299717, 19114014, 22110480, 22228119, 22571318,
        22747577, 22808131, 23724058, 24169358, 24403256, 24413757, 24664296, 25351418,
        25361045, 25859307, 25923732, 26039251, 26714306, 28285367, 28593498
    ]
    df = df.copy()
    df = df[df['pmid'].isin(pmids)]
    
    # Consolidate section types
    def consolidate_section(s):
        if pd.isna(s):
            return "OTHER"
        s = str(s).upper()
        if s == "TABLE":
            return "TABLE"
        elif s == "SUPPLEMENT":
            return "SUPPLEMENT"
        elif s in ["FIG", "FIG_TABLE"]:
            return "FIG"
        else:
            return "TEXT"
    
    df['section_consolidated'] = df['section_type'].apply(consolidate_section)
    
    # Normalize relationship (same as in match function)
    def normalize_relationship(s):
        if pd.isna(s):
            return s
        s = str(s).lower().strip()
        s = s.replace("^", " ")
        s = s.replace("increases", "increase")
        s = s.replace("decreases", "decrease")
        s = s.replace("affects", "affect")
        s = " ".join(s.split())
        return s
    
    df['relationship_norm'] = df['relationship'].apply(normalize_relationship)
    
    # Build gene_key (same as in match function)
    df["geneSymbol_clean"] = (
        df["geneSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\..+$", "", regex=True)
    )
    
    def make_gene_key(row):
        sym = row["geneSymbol_clean"]
        if pd.notna(sym) and sym not in ("", "NAN", "NA"):
            return sym
        if pd.notna(row.get("geneID")):
            try:
                return str(int(row["geneID"]))
            except:
                return pd.NA
        return pd.NA
    
    df["gene_key"] = df.apply(make_gene_key, axis=1)
    
    # Section priority
    section_priority = {
        "TABLE": 0,
        "FIG_TABLE": 1,
        "FIG": 2,
        "SUPPLEMENT": 3,
        "TITLE": 4,
        "ABSTRACT": 5,
        "RESULTS": 6,
        "DISCUSS": 7,
        "METHODS": 8,
        "INTRO": 8,
        "CONCL": 8
    }
    
    df['_section_priority'] = df['section_type'].map(section_priority).fillna(100)
    df = df.reset_index(drop=True)
    df['_row_order'] = df.index
    
    # Sort by consolidated section, then priority, then original row order
    df = df.sort_values(['section_consolidated', '_section_priority', '_row_order'], ascending=[True, True, True])
    
    # Drop duplicates WITHIN each consolidated section using NORMALIZED MERGE KEYS
    df = df.drop_duplicates(
        subset=['section_consolidated', 'pmid', 'gene_key', 'chemicalID', 'relationship_norm'], 
        keep='first'
    )
    
    # KEEP relationship_norm, gene_key, and section_consolidated for downstream use
    # ONLY drop temporary columns
    df = df.drop(columns=['_section_priority', '_row_order', 'geneSymbol_clean'])
    
    return df



In [19]:
# ============================================================
# MATCH FUNCTION
# ============================================================

def match_manual_and_df(manual: pd.DataFrame, df: pd.DataFrame):

    manual = manual.copy()
    df = df.copy()

    # --------------------------------------------------
    # 1️⃣ Filter manual rows
    # --------------------------------------------------
    if "Chain?" in manual.columns:
        manual = manual[~manual["Chain?"].fillna(False)].reset_index(drop=True)

    # --------------------------------------------------
    # 2️⃣ Rename manual columns
    # --------------------------------------------------
    manual = manual.rename(columns={
        "PMID": "pmid",
        "Interaction Actions": "relationship",
        "Chemical ID": "chemicalID",
        "Gene Symbol": "geneSymbol",
        "Gene ID": "geneID"
    })

    # --------------------------------------------------
    # 3️⃣ Clean PMID
    # --------------------------------------------------
    def clean_pmid(series):
        return (
            series.astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"[^\d]", "", regex=True)
            .replace("", pd.NA)
            .astype("Int64")
        )

    manual["pmid"] = clean_pmid(manual["pmid"])
    df["pmid"] = clean_pmid(df["pmid"])

    # --------------------------------------------------
    # 4️⃣ Normalize chemicalID
    # --------------------------------------------------
    manual["chemicalID"] = (
        manual["chemicalID"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["chemicalID"] = (
        df["chemicalID"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # --------------------------------------------------
    # 5️⃣ Normalize geneID and geneSymbol
    # --------------------------------------------------
    manual["geneID"] = pd.to_numeric(
        manual["geneID"], errors="coerce"
    ).astype("Int64")

    df["geneID"] = pd.to_numeric(
        df["geneID"], errors="coerce"
    ).astype("Int64")

    manual["geneSymbol"] = (
        manual["geneSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["geneSymbol"] = (
        df["geneSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str.replace(r"\..+$", "", regex=True)
    )

    # --------------------------------------------------
    # 6️⃣ Build gene_key: prefer geneSymbol, fall back to geneID
    # --------------------------------------------------
    def make_gene_key(row):
        sym = row["geneSymbol"]
        if pd.notna(sym) and sym not in ("", "NAN", "NA"):
            return sym
        if pd.notna(row["geneID"]):
            return str(int(row["geneID"]))
        return pd.NA

    manual["gene_key"] = manual.apply(make_gene_key, axis=1)
    manual["gene_key"] = manual["gene_key"].astype("string")

    if 'gene_key' not in df.columns:
        df["gene_key"] = df.apply(make_gene_key, axis=1)
        df["gene_key"] = df["gene_key"].astype("string")
    else:
        df["gene_key"] = df["gene_key"].astype("string")

    # --------------------------------------------------
    # 7️⃣ Normalize relationship
    # --------------------------------------------------
    def normalize_relationship(s):
        if pd.isna(s):
            return s
        s = str(s).lower().strip()
        s = s.replace("^", " ")
        s = s.replace("increases", "increase")
        s = s.replace("decreases", "decrease")
        s = s.replace("affects", "affect")
        s = " ".join(s.split())
        return s

    manual["relationship_norm"] = manual["relationship"].apply(normalize_relationship)

    if 'relationship_norm' not in df.columns:
        df["relationship_norm"] = df["relationship"].apply(normalize_relationship)

    # --------------------------------------------------
    # 7.5️⃣ Consolidate section types in df
    # --------------------------------------------------
    def consolidate_section(s):
        if pd.isna(s):
            return "OTHER"
        s = str(s).upper()
        if s == "TABLE":
            return "TABLE"
        elif s == "SUPPLEMENT":
            return "SUPPLEMENT"
        elif s in ["FIG", "FIG_TABLE"]:
            return "FIG"
        else:
            return "TEXT"

    if "section_type" in df.columns and 'section_consolidated' not in df.columns:
        df['section_consolidated'] = df['section_type'].apply(consolidate_section)
    elif "section_type" not in df.columns and 'section_consolidated' not in df.columns:
        df['section_consolidated'] = "UNKNOWN"

    manual = manual.drop_duplicates(
        subset=["pmid", "chemicalID", "relationship", "geneSymbol"]
    ).reset_index(drop=True)

    # --------------------------------------------------
    # 8️⃣ Deduplicate df using merge keys (if not already done)
    # --------------------------------------------------
    merge_keys = ["pmid", "relationship_norm", "chemicalID", "gene_key"]

    df_deduped = df

    # --------------------------------------------------
    # 9️⃣ Overall merge
    # --------------------------------------------------
    # print('Manual_relationship_distribution')
    # print(manual['relationship_norm'].value_counts())
    # print('-------')
    matched = manual.merge(
        df_deduped,
        on=merge_keys,
        how="inner",
        suffixes=("_manual", "_df")
    )

    print("=" * 60)
    print("OVERALL MATCHING STATISTICS")
    print("=" * 60)
    print(f"Total matched rows: {len(matched)}")
    match_should = matched[matched["Notes"].astype(str).str.strip().str.lower() == "should"].copy()
    print(f"Length of Should data: {len(match_should)}")
    manual_should = manual[manual["Notes"].astype(str).str.strip().str.lower() == "should"].copy()
    print(f"Length of CTD data: {len(manual)-len(manual_should)}")
    print(f"Overall Precision: {len(matched)/len(df)}")
    print(f"Overall Recall: {len(matched)/(len(manual)-len(manual_should))}")

    # --------------------------------------------------
    # 🧬 Gene Symbol Validity Statistics
    # --------------------------------------------------
    valid_gene_symbols = set(
        manual["geneSymbol"]
        .dropna()
        .loc[lambda s: ~s.isin(["", "NAN", "NA"])]
        .unique()
    )

    df_gene_symbols = df["geneSymbol"].astype(str).str.upper().str.strip()
    df_has_valid_symbol = df_gene_symbols.notna() & ~df_gene_symbols.isin(["", "NAN", "NA"])

    df["_gene_symbol_valid"]   = df_has_valid_symbol & df_gene_symbols.isin(valid_gene_symbols)
    df["_gene_symbol_invalid"] = df_has_valid_symbol & ~df_gene_symbols.isin(valid_gene_symbols)

    valid_gene_count   = df["_gene_symbol_valid"].sum()
    invalid_gene_count = df["_gene_symbol_invalid"].sum()
    no_symbol_count    = (~df_has_valid_symbol).sum()

    print(f"\n--- Gene Symbol Validity (df vs. manual) ---")
    print(f"  Valid gene symbols   (found in manual): {valid_gene_count}")
    print(f"  Invalid gene symbols (not in manual):   {invalid_gene_count}")
    print(f"  Rows with no gene symbol:               {no_symbol_count}")
    print(f"  Total df rows:                          {len(df)}")

    # if 'section_consolidated' in df.columns:
    #     print(f"\n  Per-section gene symbol validity:")
    #     section_gene_stats = (
    #         df.groupby('section_consolidated')[['_gene_symbol_valid', '_gene_symbol_invalid']]
    #         .sum()
    #         .rename(columns={'_gene_symbol_valid': 'valid', '_gene_symbol_invalid': 'invalid'})
    #     )
    #     section_gene_stats['total']     = section_gene_stats['valid'] + section_gene_stats['invalid']
    #     section_gene_stats['valid_pct'] = (
    #         section_gene_stats['valid'] / section_gene_stats['total'].replace(0, pd.NA) * 100
    #     ).round(1)
    #     print(section_gene_stats.to_string())

    df.drop(columns=["_gene_symbol_valid", "_gene_symbol_invalid"], inplace=True)

    # --------------------------------------------------
    # 🧪 Chemical ID Validity Statistics
    # --------------------------------------------------
    valid_chemical_ids = set(
        manual["chemicalID"]
        .dropna()
        .loc[lambda s: ~s.isin(["", "NAN", "NA"])]
        .unique()
    )

    df_chem_ids     = df["chemicalID"].astype(str).str.upper().str.strip()
    df_has_valid_chem = df_chem_ids.notna() & ~df_chem_ids.isin(["", "NAN", "NA"])

    df["_chem_valid"]   = df_has_valid_chem & df_chem_ids.isin(valid_chemical_ids)
    df["_chem_invalid"] = df_has_valid_chem & ~df_chem_ids.isin(valid_chemical_ids)

    valid_chem_count   = df["_chem_valid"].sum()
    invalid_chem_count = df["_chem_invalid"].sum()
    no_chem_count      = (~df_has_valid_chem).sum()

    print(f"\n--- Chemical ID Validity (df vs. manual) ---")
    print(f"  Valid chemical IDs   (found in manual): {valid_chem_count}")
    print(f"  Invalid chemical IDs (not in manual):   {invalid_chem_count}")
    print(f"  Rows with no chemical ID:               {no_chem_count}")
    print(f"  Total df rows:                          {len(df)}")

    # if 'section_consolidated' in df.columns:
    #     print(f"\n  Per-section chemical ID validity:")
    #     section_chem_stats = (
    #         df.groupby('section_consolidated')[['_chem_valid', '_chem_invalid']]
    #         .sum()
    #         .rename(columns={'_chem_valid': 'valid', '_chem_invalid': 'invalid'})
    #     )
    #     section_chem_stats['total']     = section_chem_stats['valid'] + section_chem_stats['invalid']
    #     section_chem_stats['valid_pct'] = (
    #         section_chem_stats['valid'] / section_chem_stats['total'].replace(0, pd.NA) * 100
    #     ).round(1)
    #     print(section_chem_stats.to_string())

    df.drop(columns=["_chem_valid", "_chem_invalid"], inplace=True)

    # --------------------------------------------------
    # 🔗 Relationship Error Analysis on Valid Chem-Gene Pairs
    #
    # For df rows whose (pmid, chemicalID, gene_key) triplet exists
    # in manual, we check whether the relationship_norm matches.
    # Mismatches are further classified by decomposing relationship_norm
    # into a prefix  (e.g. "increase", "decrease", "affect") and
    # a type suffix  (e.g. "expression", "activity", "methylation").
    #
    # PREFIX  = the first token of the relationship string
    # TYPE    = everything after the first token (the remainder)
    # --------------------------------------------------

    def split_relationship(rel):
        """Return (prefix, rel_type) for a normalized relationship string."""
        if pd.isna(rel):
            return (pd.NA, pd.NA)
        parts = str(rel).strip().split(" ", 1)
        prefix   = parts[0] if len(parts) >= 1 else pd.NA
        rel_type = parts[1] if len(parts) == 2 else pd.NA
        return (prefix, rel_type)

    # Build a lookup from manual: (pmid, chemicalID, gene_key) -> set of relationship_norm values
    manual_pair_rels = (
        manual.groupby(["pmid", "chemicalID", "gene_key"])["relationship_norm"]
        .apply(set)
        .reset_index()
        .rename(columns={"relationship_norm": "manual_rels"})
    )

    # Join df rows against this lookup to find rows with a valid pair
    df_pair_check = df[["pmid", "chemicalID", "gene_key", "relationship_norm"]].copy()
    df_pair_check = df_pair_check.merge(manual_pair_rels, on=["pmid", "chemicalID", "gene_key"], how="left")

    # Rows where the pair exists in manual at all
    has_valid_pair = df_pair_check["manual_rels"].notna()

    # Among valid pairs, rows where the relationship_norm is an exact match
    exact_match = has_valid_pair & df_pair_check.apply(
        lambda r: r["relationship_norm"] in r["manual_rels"] if pd.notna(r["manual_rels"]) else False,
        axis=1
    )

    # Rows with a valid pair but wrong relationship (any mismatch)
    wrong_rel = has_valid_pair & ~exact_match

    # For the mismatched rows, decompose both df and manual relationships
    mismatch_df = df_pair_check[wrong_rel].copy()
    mismatch_df[["df_prefix", "df_type"]] = pd.DataFrame(
        mismatch_df["relationship_norm"].apply(split_relationship).tolist(),
        index=mismatch_df.index
    )

    # Expand manual_rels (set) so we can compare prefix/type per manual entry
    mismatch_expanded = mismatch_df.explode("manual_rels").copy()
    mismatch_expanded[["manual_prefix", "manual_type"]] = pd.DataFrame(
        mismatch_expanded["manual_rels"].apply(split_relationship).tolist(),
        index=mismatch_expanded.index
    )

    # For each df row, find the "best" manual relationship candidate:
    # prefer ones that share at least prefix OR type (minimise error category).
    # We take the most lenient view per df row — a df row is counted in a
    # category only if *no* manual relationship for that pair escapes that error.
    def classify_mismatch(group):
        """
        Given all manual relationships for one df row (group rows = one manual rel each),
        determine the error category for that df row.

        Priority:
          1. wrong_prefix_only   — prefix differs but type matches for at least one manual rel
          2. wrong_type_only     — type differs but prefix matches for at least one manual rel
          3. wrong_prefix_and_type — neither prefix nor type matches any manual rel
        """
        df_prefix = group["df_prefix"].iloc[0]
        df_type   = group["df_type"].iloc[0]

        prefix_matches = (group["manual_prefix"] == df_prefix).any()
        type_matches   = (group["manual_type"]   == df_type  ).any()

        if prefix_matches and not type_matches:
            return "wrong_type_only"
        elif type_matches and not prefix_matches:
            return "wrong_prefix_only"
        elif not prefix_matches and not type_matches:
            return "wrong_prefix_and_type"
        else:
            # prefix and type both match in some manual rel but not combined — edge case
            return "wrong_prefix_and_type"

    mismatch_categories = (
        mismatch_expanded
        .groupby(mismatch_expanded.index)
        .apply(classify_mismatch)
        .rename("error_category")
    )

    category_counts = mismatch_categories.value_counts()

    wrong_prefix_only       = category_counts.get("wrong_prefix_only",       0)
    wrong_type_only         = category_counts.get("wrong_type_only",          0)
    wrong_prefix_and_type   = category_counts.get("wrong_prefix_and_type",    0)
    wrong_combination       = category_counts.get("wrong_combination",        0)

    print(f"\n--- Relationship Error Analysis on Valid Chem-Gene Pairs ---")
    print(f"  DF rows with a valid (pmid, chemicalID, gene_key) pair: {has_valid_pair.sum()}")
    print(f"  Exact relationship match:                               {exact_match.sum()}")
    print(f"  Wrong relationship (any mismatch):                      {wrong_rel.sum()}")
    print(f"    ├─ Wrong prefix only  (correct type, wrong direction): {wrong_prefix_only}")
    print(f"    ├─ Wrong type only    (correct prefix, wrong type):    {wrong_type_only}")
    print(f"    ├─ Wrong prefix AND type:                              {wrong_prefix_and_type}")
    # if wrong_combination:
    #     print(f"    └─ Ambiguous (components match separately, not jointly): {wrong_combination}")

    # --------------------------------------------------
    # 🔟 Section-specific matching
    # --------------------------------------------------
    section_types = ["TABLE", "SUPPLEMENT", "FIG", "TEXT"]

    print("\n" + "=" * 60)
    print("SECTION-SPECIFIC MATCHING")
    print("=" * 60)

    section_stats = []

    for section in section_types:
        df_section = df_deduped[df_deduped['section_consolidated'] == section]

        matched_section = manual.merge(
            df_section,
            on=merge_keys,
            how="inner",
            suffixes=("_manual", "_df")
        )

        section_stats.append({
            'section': section,
            'df_rows': len(df_section),
            'matched_count': len(matched_section),
            'precision': len(matched_section) / len(df_section) if len(df_section) > 0 else 0,
            'recall': len(matched_section) / (len(manual)-len(manual_should)) if (len(manual)-len(manual_should)) > 0 else 0,
        })

        print(f"\n{section}:")
        print(f"  DF rows in section: {len(df_section)}")
        print(f"  Matched rows: {len(matched_section)}")

    section_df = pd.DataFrame(section_stats)
    print("\n" + "-" * 60)
    print("SUMMARY TABLE:")
    print(section_df.to_string(index=False))
    print("-" * 60)

    # --------------------------------------------------
    # 1️⃣1️⃣ Unmatched rows (overall)
    # --------------------------------------------------
    matched_keys = matched[merge_keys].drop_duplicates(subset=['pmid', 'gene_key', 'chemicalID', 'relationship_norm'])

    unmatched_manual = manual.merge(
        matched_keys,
        on=merge_keys,
        how="left",
        indicator=True
    )
    unmatched_manual = unmatched_manual[
        unmatched_manual["_merge"] == "left_only"
    ].drop(columns="_merge")

    unmatched_df = df.merge(
        matched_keys,
        on=merge_keys,
        how="left",
        indicator=True
    )
    unmatched_df = unmatched_df[
        unmatched_df["_merge"] == "left_only"
    ].drop(columns="_merge")

    print(f"\nUnmatched manual rows: {len(unmatched_manual)}")
    print(f"Unmatched df rows: {len(unmatched_df)}")

    # --------------------------------------------------
    # 1️⃣2️⃣ Partial matches (overall)
    # --------------------------------------------------
    partial_keys = ["pmid", "geneSymbol", "chemicalID"]

    # Include section_consolidated if available
    extra_df_cols = ["relationship_norm", "gene_key"]
    if "section_consolidated" in unmatched_df.columns:
        extra_df_cols.append("section_consolidated")
    
    partial_matches = unmatched_manual.merge(
        unmatched_df[partial_keys + extra_df_cols].drop_duplicates(subset=['pmid', 'gene_key', 'chemicalID']),
        on=partial_keys,
        how="inner",
        suffixes=("_manual", "_df")
    )

    if "section_consolidated" in partial_matches.columns:
        print("\n--- Section distribution of partial matches (DF side) ---")
        print(partial_matches["section_consolidated"].value_counts().to_string())

    if len(partial_matches) > 0:
        print(f"\n⚠️  {len(partial_matches)} rows matched on pmid+geneSymbol+chemicalID but NOT on full merge keys")

        print("\n--- Manual relationship_norm (unmatched, partial key match) ---")
        print(partial_matches["relationship_norm_manual"].value_counts().to_string())

        print("\n--- DF relationship_norm (unmatched, partial key match) ---")
        print(partial_matches["relationship_norm_df"].value_counts().to_string())

        print("\n--- Side-by-side (Manual → DF) ---")
        paired = (
            partial_matches
            .groupby(["relationship_norm_manual", "relationship_norm_df"])
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        print(paired.to_string(index=False))

    else:
        print("\n✅ No partial matches — unmatched rows don't share pmid+geneSymbol+chemicalID")

    print("=" * 60 + "\n")

    return matched, unmatched_manual, unmatched_df, partial_matches

In [20]:
def update_dataframe(sample):
    """
    Updates DataFrame columns and normalizes values without dropping rows.
    
    Args:
        sample: Input DataFrame
        
    Returns:
        DataFrame with updated/normalized columns
    """
    sample = sample.copy()
    
    # Normalize string columns
    sample['source'] = sample['source'].fillna("").astype(str)
    sample['text'] = sample['text'].fillna("").astype(str)
    sample['relationship'] = sample['relationship'].fillna("").astype(str)
    
    # Chemical ID normalization: D000072317 -> D013749
    mask = (
        sample["chemicalID"]
            .astype(str)
            .str.casefold()
            .eq("d000072317")
    )
    sample.loc[mask, "chemicalID"] = "D013749"
    
    # Sort if Unnamed: 0 exists
    if 'Unnamed: 0' in sample.columns:
        sample = sample.sort_values(by=['Unnamed: 0']).reset_index(drop=True)
    else:
        sample = sample.reset_index(drop=True)
    
    return sample


def filter_dataframe(sample, CTDchemicals, CTDgenes):
    """
    Filters DataFrame to remove low-quality/invalid relationships.
    
    Args:
        sample: Input DataFrame (should be output from update_dataframe)
        CTDchemicals: CTD chemicals reference DataFrame
        CTDgenes: CTD genes reference DataFrame
        
    Returns:
        Filtered DataFrame with high-precision relationships
    """
    sample = sample.copy()
    
    print(f"Initial length of data: {len(sample)}")
    
    # ─────────────────────────────────────────────────────────────
    # 1. Filter relationships by specificity
    # ─────────────────────────────────────────────────────────────
    specific_keywords = ["increase", "decrease"]
    generic_keyword = "affect"
        
    def filter_relationships(group):
        rels = group['relationship'].fillna("").astype(str).str.lower()
        
        is_specific = rels.apply(lambda x: any(kw in x for kw in specific_keywords))
        is_generic  = rels.str.contains(generic_keyword, na=False)
        
        specific = group[is_specific]
        generic = group[is_generic & ~is_specific] if specific.empty else pd.DataFrame(columns=group.columns)
        others = group[~is_specific & ~is_generic]
        
        dfs_to_concat = [df for df in [specific, generic, others] if not df.empty]
        return pd.concat(dfs_to_concat, ignore_index=True)
    
    group_cols = ['chemicalID', 'geneSymbol']
    
    sample = (
        sample
        .groupby(group_cols, group_keys=False)[sample.columns]
        .apply(filter_relationships)
        .reset_index(drop=True)
    )

    # ─────────────────────────────────────────────────────────────
    # 2. Drop rows not in CTD
    # ─────────────────────────────────────────────────────────────
    ctd_chem_set = set(CTDchemicals['# ChemicalName'].dropna().str.strip().str.lower())
    ctd_gene_set = set(pd.concat([CTDgenes[col].dropna() for col in ['# GeneSymbol', 'GeneName']])
                       .str.strip().str.lower())
    
    sample['chemical_name_clean'] = sample['chemicalName'].str.strip().str.lower()
    sample['gene_name_clean'] = sample['geneSymbol'].str.strip().str.lower()
    
    chem_mask = sample['chemical_name_clean'].ne('') & sample['chemical_name_clean'].isin(ctd_chem_set)
    gene_mask = sample['gene_name_clean'].ne('') & sample['gene_name_clean'].isin(ctd_gene_set)
    
    sample = sample[chem_mask & gene_mask].reset_index(drop=True)
    sample = sample.drop(columns=['chemical_name_clean', 'gene_name_clean'])
        
    # ─────────────────────────────────────────────────────────────
    # 3. Drop discussion section with citation patterns
    # ─────────────────────────────────────────────────────────────
    bad_discuss_patterns = [
        "previous studies", 
        "reported that", 
        "et al", 
        "previous works", 
        "according to", 
        "authors",
        "author",
    ]

    mask_discuss_bad = (
        sample['section_type'].isin(['DISCUSS']) &
        sample['text']
              .str.lower()
              .str.replace(r'[^\w\s]', ' ', regex=True)
              .str.replace(r'\s+', ' ', regex=True)
              .str.contains('|'.join(bad_discuss_patterns), na=False)
    )
    
    sample = sample[~mask_discuss_bad].reset_index(drop=True)

    # ─────────────────────────────────────────────────────────────
    # 4. Drop rows with invalid/missing entity flags
    # ─────────────────────────────────────────────────────────────
    sample = sample[
        (sample['chemicalFlag'] == 0) &
        (sample['relationshipWarningFlag'] == 0) &
        (sample['prefixMissingFlag'] == 0) &
        (sample['geneSymbolFlag'] == 0)
    ].reset_index(drop=True)

    # ─────────────────────────────────────────────────────────────
    # 5. Drop rows with reference citations [number]
    # ─────────────────────────────────────────────────────────────
    mask_brackets = sample['text'].str.contains(r'\[\d+\]', regex=True, na=False)
    sample = sample[~mask_brackets].reset_index(drop=True)

    # ─────────────────────────────────────────────────────────────
    # 6. Drop uncertain/speculative language in source
    # ─────────────────────────────────────────────────────────────
    pattern = r'\b(?:' \
              r'conceiv\w*|' \
              r'plaus\w*|' \
              r'possibl\w*|' \
              r'might|maybe|' \
              r'known|future|perhaps|potentially|feasible|likely|believable|' \
              r'establish\w*|' \
              r'documented|recognized|reported|' \
              r'upcoming|forthcoming|subsequent|later|' \
              r'other|another|' \
              r'has been|proven|show|previous|previously|' \
              r'partial\w*|' \
              r'small|tiny|little|almost|' \
              r'studied|studies|investigated' \
              r')\b'

    mask_uncertain = sample['source'].str.contains(pattern, case=False, regex=True, na=False)
    sample = sample[~mask_uncertain].reset_index(drop=True)

    # ─────────────────────────────────────────────────────────────
    # 7. Drop unwanted section types
    # ─────────────────────────────────────────────────────────────
    excluded_sections = ['METHODS', 'INTRO', 'CONCL', 'ABBR', 'DISCUSS']
    sample = sample[~sample['section_type'].isin(excluded_sections)]

    # ─────────────────────────────────────────────────────────────
    # 8. Drop negative results in text
    # ─────────────────────────────────────────────────────────────
    negative_patterns = ["does not affect", "no effect", "not alter", "not change", "no change", "significant", "significance"]
    mask_negative = sample['text'].str.lower().str.contains("|".join(negative_patterns), na=False)
    sample = sample[~mask_negative].reset_index(drop=True)

    contradict_patterns = [
        # No effect / lack of effect
        "lack of effect",
        "no effect",
        "without effect",
        "did not affect",
        "does not affect",
        "didn't affect",
        "doesn't affect",
        "did not alter",
        "does not alter",
        "didn't alter",
        "doesn't alter",
        "did not induce",
        "does not induce",
        "didn't induce",
        "doesn't induce",
        "did not change",
        "does not change",
        "didn't change",
        "doesn't change",
        "was not affected",
        "wasn't affected",
        "was not altered",
        "wasn't altered",
        "was not induced",
        "wasn't induced",
        "had no effect",
        "no observable effect",
        
        # Negative results / failed outcomes
        "failed to",
        "unable to",
        "cannot conclude",
        "can't conclude",
        "did not demonstrate",
        "didn't demonstrate",
        "did not observe",
        "didn't observe",
        "was ineffective",
        "no significant change",
        "no significant effect",
        "not significant",
        "insignificant",
        
        # Lack of detection / measurement
        "not detectable",
        "undetectable",
        "could not detect",
        "couldn't detect",
        "cannot detect",
        "can't detect",
        
        # Contradictory phrasing / uncertainty
        "not associated with",
        "no correlation",
        "no relationship",
        "did not correlate",
        "didn't correlate",
        "was not linked",
        "wasn't linked",
        "not linked",
        "no impact",
        "did not impact",
        "didn't impact",
        "no influence",
        "did not influence",
        "didn't influence",
        
        # Misc negative speculation
        "did not support",
        "didn't support",
        "failed to support",
        "was not confirmed",
        "wasn't confirmed",
        "cannot confirm",
        "can't confirm",
        "did not confirm",
        "didn't confirm",
        "not reproducible",
        "irreproducible",
    ]

    mask_contradict = sample['text'].str.lower().str.contains("|".join(contradict_patterns), na=False)
    sample = sample[~mask_contradict].reset_index(drop=True)

    # ─────────────────────────────────────────────────────────────
    # 9. Drop broad/general statements in source
    # ─────────────────────────────────────────────────────────────
    broad_patterns = [
        "generally",
        "commonly",
        "in general",
        "common",
        "general consensus",
        "it is known",
        "widely believed",
        "studies show",
        "extensively studied",
        "previous research",
        "has been reported",
        "according to studies",
        "commonly observed",
        "well established",
        "it is thought",
        "it has been suggested",
        "considered to be",
        "generally accepted",
        "appears to",
        "tends to",
        "is associated with",
        "reported previously",
        "as demonstrated",
        "numerous studies",
        "widely recognized"
    ]
    mask_broad = sample['source'].str.lower().str.contains("|".join(broad_patterns), na=False)
    sample = sample[~mask_broad].reset_index(drop=True)

    # ─────────────────────────────────────────────────────────────
    # 10. Drop rows with missing entity maps (except allowed sections)
    # ─────────────────────────────────────────────────────────────
    allowed_sections = ['FIG', 'FIG_TABLE', 'TABLE', 'SUPPLEMENT']
    
    mask_maps_empty = (
        (sample['chemical_map'].isna() | (sample['chemical_map'] == '')) |
        (sample['gene_map'].isna() | (sample['gene_map'] == ''))
    )
    
    mask_section_not_allowed = ~sample['section_type'].isin(allowed_sections)
    
    sample = sample[~(mask_maps_empty & mask_section_not_allowed)].reset_index(drop=True)

    # ─────────────────────────────────────────────────────────────
    # 12. Drop rows where LLM reasoning flags issues
    # ─────────────────────────────────────────────────────────────
    keywords = ["null", "significant", "remove", "does not affect", "no effect", 
                "not alter", "not change", "no change", "missing", "invalid"]
    pattern = "|".join(keywords)
    
    sample = sample[
        ~sample["reasoning"].astype(str).str.lower().str.contains(pattern, na=False)
    ]

    ##------------------------
    sample['chemical_map'] = sample['chemical_map'].astype(str)
    sample['gene_map'] = sample['gene_map'].astype(str)
    
    excluded_map_sections_chem = ['TABLE', 'SUPPLEMENT']
    excluded_map_sections_gene = ['TABLE', 'SUPPLEMENT', 'FIG', 'FIG_TABLE']
    
    mask_chem_in_map = sample.apply(
        lambda r: r['chemicalID'] == 'D000072317' or str(r['chemicalID']).lower() in r['chemical_map'].lower(), axis=1
    )
    
    mask_gene_in_map = sample.apply(
        lambda r: str(r['geneID']).lower() in r['gene_map'].lower(), axis=1
    )
    
    mask_chem_pass = mask_chem_in_map | sample['section_type'].isin(excluded_map_sections_chem)
    mask_gene_pass = mask_gene_in_map | sample['section_type'].isin(excluded_map_sections_gene)
    
    sample = sample[mask_chem_pass & mask_gene_pass].reset_index(drop=True)

    print(f'Output length of data: {len(sample)}')
    return sample

In [23]:
manual = pd.read_csv('/Users/sc/nlp_4_ctd_code/data_json_clover/manual_annotations/CTD.csv')

output_cols = [
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',
    
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output'
]

save_dir = "/Users/sc/nlp_4_ctd_code/data_json_clover/CLOVER_output_files/"
os.makedirs(save_dir, exist_ok=True)

print('FINAL')

for col in output_cols:
    df = pd.read_csv(f'{output_data_path}/temp_V1/{col}.csv')
    df = drop_duplicates_(df)
    df = update_dataframe(df)
    df = filter_dataframe(df, ctd_chemicals, ctd_genes)
    
    print(f'{col}')
    
    matched, unmatched_manual, unmatched_df, partial_matches = match_manual_and_df(manual, df)
    
    # Save outputs
    matched.to_csv(os.path.join(save_dir, f"{col}_matched.csv"), index=False)
    unmatched_manual.to_csv(os.path.join(save_dir, f"{col}_unmatched_manual.csv"), index=False)
    unmatched_df.to_csv(os.path.join(save_dir, f"{col}_unmatched_df.csv"), index=False)
    partial_matches.to_csv(os.path.join(save_dir, f"{col}_partial_matches.csv"), index=False)

    print()

FINAL
Initial length of data: 1790
Output length of data: 508
deepseek_E_output
OVERALL MATCHING STATISTICS
Total matched rows: 271
Length of Should data: 3
Length of CTD data: 609
Overall Precision: 0.5334645669291339
Overall Recall: 0.44499178981937604

--- Gene Symbol Validity (df vs. manual) ---
  Valid gene symbols   (found in manual): 388
  Invalid gene symbols (not in manual):   120
  Rows with no gene symbol:               0
  Total df rows:                          508

--- Chemical ID Validity (df vs. manual) ---
  Valid chemical IDs   (found in manual): 491
  Invalid chemical IDs (not in manual):   17
  Rows with no chemical ID:               0
  Total df rows:                          508

--- Relationship Error Analysis on Valid Chem-Gene Pairs ---
  DF rows with a valid (pmid, chemicalID, gene_key) pair: 339
  Exact relationship match:                               271
  Wrong relationship (any mismatch):                      68
    ├─ Wrong prefix only  (correct type, wr

In [16]:
manual = pd.read_csv('/Users/sc/nlp_4_ctd_code/data_json_clover/manual_annotations/CTD.csv')

output_cols = [
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',
    
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output'
]

print('SECTION')
for col in output_cols:
    df = pd.read_csv(f'{output_data_path}/temp_v1/{col}.csv')
    df = drop_duplicates_section(df)
    df = update_dataframe(df)
    df = filter_dataframe(df, ctd_chemicals, ctd_genes)
    
    print(f'{col}')
    matched, unmatched_manual, unmatched_df, partial_matches = match_manual_and_df(manual, df)

    print()

SECTION
Initial length of data: 1857
Output length of data: 522
deepseek_E_output
OVERALL MATCHING STATISTICS
Total matched rows: 274
Length of Should data: 4
Length of CTD data: 609
Overall Precision: 0.524904214559387
Overall Recall: 0.44991789819376027

--- Gene Symbol Validity (df vs. manual) ---
  Valid gene symbols   (found in manual): 402
  Invalid gene symbols (not in manual):   120
  Rows with no gene symbol:               0
  Total df rows:                          522

--- Chemical ID Validity (df vs. manual) ---
  Valid chemical IDs   (found in manual): 505
  Invalid chemical IDs (not in manual):   17
  Rows with no chemical ID:               0
  Total df rows:                          522

--- Relationship Error Analysis on Valid Chem-Gene Pairs ---
  DF rows with a valid (pmid, chemicalID, gene_key) pair: 345
  Exact relationship match:                               274
  Wrong relationship (any mismatch):                      71
    ├─ Wrong prefix only  (correct type, w

In [17]:
manual = pd.read_csv('/Users/sc/nlp_4_ctd_code/data_json_clover/manual_annotations/CTD.csv')

output_cols = [
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',
    
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output'
]

print('NO FILTER')
for col in output_cols:
    df = pd.read_csv(f'{output_data_path}/temp_v1/{col}.csv')
    df = drop_duplicates_(df)
    mask = (
        df["chemicalID"]
            .astype(str)
            .str.casefold()
            .eq("d000072317")
    )
    df.loc[mask, "chemicalID"] = "D013749"
    # df = update_dataframe(df)
    # df = filter_dataframe(df, ctd_chemicals, ctd_genes)
    
    print(f'{col}')
    matched, unmatched_manual, unmatched_df, partial_matches = match_manual_and_df(manual, df)

    print()

NO FILTER
deepseek_E_output
OVERALL MATCHING STATISTICS
Total matched rows: 339
Length of Should data: 12
Length of CTD data: 609
Overall Precision: 0.1893854748603352
Overall Recall: 0.5566502463054187

--- Gene Symbol Validity (df vs. manual) ---
  Valid gene symbols   (found in manual): 1161
  Invalid gene symbols (not in manual):   570
  Rows with no gene symbol:               59
  Total df rows:                          1790

--- Chemical ID Validity (df vs. manual) ---
  Valid chemical IDs   (found in manual): 1185
  Invalid chemical IDs (not in manual):   604
  Rows with no chemical ID:               1
  Total df rows:                          1790

--- Relationship Error Analysis on Valid Chem-Gene Pairs ---
  DF rows with a valid (pmid, chemicalID, gene_key) pair: 581
  Exact relationship match:                               339
  Wrong relationship (any mismatch):                      242
    ├─ Wrong prefix only  (correct type, wrong direction): 114
    ├─ Wrong type only   

In [18]:
df = pd.read_csv(f'{output_data_path}/temp_v1/claude_E_output.csv')
df = drop_duplicates_(df)
df = update_dataframe(df)
df = filter_dataframe(df, ctd_chemicals, ctd_genes)

print(f'{col}')
matched, unmatched_manual, unmatched_df, partial_matches = match_manual_and_df(manual, df)

print()
unmatched_manual.to_csv(f'{output_data_path}/temp.csv')

Initial length of data: 2207
Output length of data: 659
gpt4oV_claudeE_output
OVERALL MATCHING STATISTICS
Total matched rows: 437
Length of Should data: 5
Length of CTD data: 609
Overall Precision: 0.6631259484066768
Overall Recall: 0.7175697865353038

--- Gene Symbol Validity (df vs. manual) ---
  Valid gene symbols   (found in manual): 552
  Invalid gene symbols (not in manual):   107
  Rows with no gene symbol:               0
  Total df rows:                          659

--- Chemical ID Validity (df vs. manual) ---
  Valid chemical IDs   (found in manual): 618
  Invalid chemical IDs (not in manual):   41
  Rows with no chemical ID:               0
  Total df rows:                          659

--- Relationship Error Analysis on Valid Chem-Gene Pairs ---
  DF rows with a valid (pmid, chemicalID, gene_key) pair: 473
  Exact relationship match:                               437
  Wrong relationship (any mismatch):                      36
    ├─ Wrong prefix only  (correct type, wrong

# TOKENS

In [15]:
# output_cols = [
#     'deepseek_E_output',
#     'gpt_E_output',
#     'claude_E_output',
#     'gpt4o_E_output',
    
#     'deepseekV_gptE_output',
#     'deepseekV_gpt4oE_output',
#     'deepseekV_claudeE_output',
    
#     'gptV_deepseekE_output',
#     'gptV_claudeE_output',
    
#     'claudeV_deepseekE_output',
#     'claudeV_gptE_output',
#     'claudeV_gpt4oE_output',
    
#     'gpt4oV_deepseekE_output',
#     'gpt4oV_claudeE_output'
# ]

# v1_path = f'{output_data_path}/refined_V1/{col}.csv'
# v2_path = f'{output_data_path}/refined_V2/{col}.csv'
# out_path = f'{output_data_path}/refined_final/{col}.csv'

# for col in output_cols:
#     v1_path = f'{output_data_path}/refined_V1/{col}.csv'
#     v2_path = f'{output_data_path}/refined_V2/{col}.csv'
#     out_path = f'{output_data_path}/refined_final/{col}.csv'
    
#     v1_df = pd.read_csv(v1_path)
#     v2_df = pd.read_csv(v2_path)

#     # Merge confidence from v2 into v1
#     merged_df = v1_df.merge(
#         v2_df[['_source_row', 'chemicalID', 'geneID', 'relationship', 'confidence']],
#         on=['_source_row', 'chemicalID', 'geneID', 'relationship'],
#         how='left'
#     )

#     # Fill missing confidence with 3
#     merged_df['confidence'] = merged_df['confidence'].fillna(3)

#     # Ensure output directory exists
#     os.makedirs(os.path.dirname(out_path), exist_ok=True)

#     # Save
#     merged_df.to_csv(out_path, index=False)

In [16]:
# see how confidence affects scores
# mark table rows as should 
# try to reduce further, confidence may help with image data

In [82]:
import pandas as pd
import tiktoken
import base64
import os

# -------------------------------
# 1️⃣ Load data
# -------------------------------
prompts = pd.read_csv(f'{input_data_path}/inputVal.csv')

# Choose tokenizer
enc = tiktoken.encoding_for_model("gpt-4o")

# -------------------------------
# 2️⃣ Function to count tokens safely
# -------------------------------
def count_tokens(text):
    if pd.isna(text):
        return 0
    return len(enc.encode(str(text)))

# -------------------------------
# 3️⃣ Validation columns
# -------------------------------
val_cols = [
    'main_prompt_val_gpt',
    'main_prompt_val_claude',
    'main_prompt_val_deepseek',
    'main_prompt_val_gpt4o',
]

all_prompt_cols = val_cols + ['main_prompt']

# -------------------------------
# 4️⃣ Normalize section types
# -------------------------------
def normalize_section(section):
    if pd.isna(section):
        return "TEXT"
    section = str(section).upper()
    if "TABLE" in section:
        return "TABLE"
    elif "FIG" in section:
        return "FIG"
    elif "SUPP" in section:
        return "SUPP"
    else:
        return "TEXT"

prompts["section_group"] = prompts["section_type"].apply(normalize_section)

# -------------------------------
# 5️⃣ Function to get base64 length of image
# -------------------------------
image_base_path = f"{figures_path}"  # adjust to your folder

def get_base64_image_str(filename):
    if pd.isna(filename):
        return ""
    path = os.path.join(image_base_path, filename)
    if not os.path.exists(path):
        return ""
    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return encoded

# -------------------------------
# 6️⃣ Count tokens including FIG images
# -------------------------------
# Updated counting including system prompts and FIG_TABLE images
for col in all_prompt_cols:
    if col == "main_prompt":
        # For main_prompt, append FIG base64 AND main system prompt
        def prompt_with_image_and_sys(row):
            text = row[col] if pd.notna(row[col]) else ""
            
            # Append FIG image if applicable
            if row["section_group"] == "FIG":
                # Main FIG image
                if pd.notna(row["filename"]):
                    text += " " + get_base64_image_str(row["filename"])
                
                # FIG_TABLE image (optional column, adjust name if different)
                if "fig_table_filename" in row and pd.notna(row["fig_table_filename"]):
                    text += " " + get_base64_image_str(row["fig_table_filename"])
            
            # Append main system prompt
            if pd.notna(row["system_prompt"]):
                text = str(row["system_prompt"]) + " " + text
            
            return count_tokens(text)
        
        prompts[f"{col}_tokens"] = prompts.apply(prompt_with_image_and_sys, axis=1)
    
    else:
        # Validation prompts: append validation system prompt
        sys_col = "system_prompt_val"
        def val_prompt_with_sys(row):
            text = row[col] if pd.notna(row[col]) else ""
            
            # For FIG prompts, also include FIG_TABLE image if present
            if row["section_group"] == "FIG":
                if pd.notna(row.get("filename")):
                    text += " " + get_base64_image_str(row["filename"])
                if pd.notna(row.get("fig_table_filename")):
                    text += " " + get_base64_image_str(row["fig_table_filename"])
            
            # Append validation system prompt
            if pd.notna(row[sys_col]):
                text = str(row[sys_col]) + " " + text
            
            return count_tokens(text)
        
        prompts[f"{col}_tokens"] = prompts.apply(val_prompt_with_sys, axis=1)

# -------------------------------
# 7️⃣ Average tokens per section group
# -------------------------------
token_cols = [f"{c}_tokens" for c in all_prompt_cols]

section_avg = (
    prompts
    .groupby("section_group")[token_cols]
    .mean()
    .reset_index()
)

print("\nAverage tokens per section (per model):")
print(section_avg)

# -------------------------------
# 8️⃣ Overall validation average per section
# -------------------------------
val_token_cols = [f"{c}_tokens" for c in val_cols]

section_val_overall = (
    prompts
    .groupby("section_group")[val_token_cols]
    .mean()
)

section_val_overall["overall_val_avg_tokens"] = (
    section_val_overall.mean(axis=1)
)

print("\nOverall validation average tokens per section:")
print(section_val_overall[["overall_val_avg_tokens"]])

# -------------------------------
# 9️⃣ Count number of prompts per section group
# -------------------------------
section_counts = prompts["section_group"].value_counts()
print("\nNumber of prompts per section group:")
print(section_counts)


Average tokens per section (per model):
  section_group  main_prompt_val_gpt_tokens  main_prompt_val_claude_tokens  \
0           FIG                68431.628378                   68360.601351   
1          SUPP                 3422.800000                    3420.800000   
2         TABLE                 3572.078947                    2881.842105   
3          TEXT                 2892.907692                    2895.717949   

   main_prompt_val_deepseek_tokens  main_prompt_val_gpt4o_tokens  \
0                     67298.027027                  68324.547297   
1                      3422.800000                   3419.600000   
2                      3018.684211                   3380.473684   
3                      2870.979487                   2840.205128   

   main_prompt_tokens  
0        67941.560811  
1         2913.600000  
2         2690.447368  
3         1575.007692  

Overall validation average tokens per section:
               overall_val_avg_tokens
section_group        

# raw outputs LLM

In [89]:
import os

output_cols = [
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',
    
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output'
]

data = pd.read_csv(f"{output_data_path}/validation_v1.csv")
data.columns

combined_df = combine_csv_outputs(data, output_cols)

for col in output_cols:
    output_file = f'{output_data_path}/temp_V1/raw/{col}.csv'

    if os.path.exists(output_file):
        print(f'Skipping {col}, file already exists.')
        continue

    temp_df = combined_df[col]
    temp_df.to_csv(output_file, index=False)
    print(f'Finished: {col}')

Finished: deepseek_E_output
Finished: gpt_E_output
Finished: claude_E_output
Finished: gpt4o_E_output
Finished: deepseekV_gptE_output
Finished: deepseekV_gpt4oE_output
Finished: deepseekV_claudeE_output
Finished: gptV_deepseekE_output
Finished: gptV_claudeE_output
Finished: claudeV_deepseekE_output
Finished: claudeV_gptE_output
Finished: claudeV_gpt4oE_output
Finished: gpt4oV_deepseekE_output
Finished: gpt4oV_claudeE_output


{'deepseek_E_output':       _source_row      pmid  \
 0             150  15154616   
 1             150  15154616   
 2             150  15154616   
 3             150  15154616   
 4             151  15953866   
 ...           ...       ...   
 2404          580  22747577   
 2405          580  22747577   
 2406          580  22747577   
 2407          580  22747577   
 2408          580  22747577   
 
                                                 caption original_filename  \
 0                                                   NaN               NaN   
 1                                                   NaN               NaN   
 2                                                   NaN               NaN   
 3                                                   NaN               NaN   
 4                                                   NaN               NaN   
 ...                                                 ...               ...   
 2404  Genes with a time-dependent, continuous 

In [95]:
combined_df['deepseek_E_output']

,_source_row,pmid,caption,original_filename,gene_map,chemical_map,section_type,text,table_xml,explanation,chemicalName,chemicalID,relationship,geneName,geneID,geneSymbol,source,_source_row,agreement,reasoning
0,150,15154616,NaN,NaN,"{""MIG"": 17329, ""Toll-Like Receptor-4"": 21898, ...","{""LPS"": ""D008070"", ""CpG"": ""C015772"", ""zymosan""...",ABSTRACT,"Monokine Induced by Interferon- (MIG), a CXC c...",NaN,LPS synergizes with IFN-gamma to induce MIG ex...,LPS,D008070,increase expression,MIG,17329,NaN,the Toll-Like Receptor-4 (TLR-4) ligand LPS sy...,150,<NA>,<NA>
1,150,15154616,NaN,NaN,"{""MIG"": 17329, ""Toll-Like Receptor-4"": 21898, ...","{""LPS"": ""D008070"", ""CpG"": ""C015772"", ""zymosan""...",ABSTRACT,"Monokine Induced by Interferon- (MIG), a CXC c...",NaN,LPS induces NF-kappaB activation,LPS,D008070,increase activity,NF-kappaB,18033,NaN,LPS-induced NF-kappaB,150,<NA>,<NA>
2,150,15154616,NaN,NaN,"{""MIG"": 17329, ""Toll-Like Receptor-4"": 21898, ...","{""LPS"": ""D008070"", ""CpG"": ""C015772"", ""zymosan""...",ABSTRACT,"Monokine Induced by Interferon- (MIG), a CXC c...",NaN,CpG synergizes with IFN-gamma to induce MIG ex...,CpG,C015772,increase expression,MIG,17329,NaN,"other TLR ligands (zymosan, double stranded RN...",150,<NA>,<NA>
3,150,15154616,NaN,NaN,"{""MIG"": 17329, ""Toll-Like Receptor-4"": 21898, ...","{""LPS"": ""D008070"", ""CpG"": ""C015772"", ""zymosan""...",ABSTRACT,"Monokine Induced by Interferon- (MIG), a CXC c...",NaN,zymosan synergizes with IFN-gamma to induce MI...,zymosan,D015054,increase expression,MIG,17329,NaN,"other TLR ligands (zymosan, double stranded RN...",150,<NA>,<NA>
4,151,15953866,NaN,NaN,"{""BMP-2"": 650, ""ALP"": 250, ""osteocalcin"": 632}","{""simvastatin"": ""D019821""}",DISCUSS,Treatment with BMP-2 increases the ALP activit...,NaN,Simvastatin treatment is associated with incre...,which is linked to its bone-forming effects.,simvastatin,D019821,"increase expression,BMP-2",650,NaN,It is generally agreed that the bone forming e...,151,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2404,580,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,SUPPLEMENT,NaN,NaN,NaN,masitinib,NaN,increase expression,thymocyte selection-associated high mobility g...,NaN,TOX,NaN,580,<NA>,<NA>
2405,580,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,SUPPLEMENT,NaN,NaN,NaN,masitinib,NaN,increase expression,"thyroid stimulating hormone, beta",NaN,TSHB,NaN,580,<NA>,<NA>
2406,580,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,SUPPLEMENT,NaN,NaN,NaN,masitinib,NaN,increase expression,tetraspanin 2,NaN,TSPAN2,NaN,580,<NA>,<NA>
2407,580,22747577,"Genes with a time-dependent, continuous increa...",NaN,NaN,NaN,SUPPLEMENT,NaN,NaN,NaN,masitinib,NaN,increase expression,similar to Uroporphyrinogen-III synthase (UROS...,NaN,UROS,NaN,580,<NA>,<NA>


In [11]:
def match_manual_hierarchical(manual: pd.DataFrame, df: pd.DataFrame):
    manual = manual.copy()
    df = df.copy()

    # --------------------------------------------------
    # 1️⃣ Rename manual columns to standard
    # --------------------------------------------------
    manual = manual.rename(columns={
        "PMID": "pmid",
        "Interaction Actions": "relationship",
        "Chemical ID": "chemicalID",
        "Gene Symbol": "geneSymbol",
        "Gene ID": "geneID"
    })

    # --------------------------------------------------
    # 2️⃣ Filter out chains if column exists
    # --------------------------------------------------
    if "Chain?" in manual.columns:
        manual = manual[~manual["Chain?"].fillna(False)].reset_index(drop=True)

    # --------------------------------------------------
    # 3️⃣ Clean PMIDs
    # --------------------------------------------------
    def clean_pmid(series):
        return (
            series.astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"[^\d]", "", regex=True)
            .replace("", pd.NA)
            .astype("Int64")
        )

    manual["pmid"] = clean_pmid(manual["pmid"])
    df["pmid"] = clean_pmid(df["pmid"])

    # --------------------------------------------------
    # 4️⃣ Normalize chemicalID
    # --------------------------------------------------
    manual["chemicalID"] = manual["chemicalID"].astype(str).str.upper().str.strip()
    df["chemicalID"] = df["chemicalID"].astype(str).str.upper().str.strip()

    # --------------------------------------------------
    # 5️⃣ Normalize relationship
    # --------------------------------------------------
    def normalize_relationship(s):
        if pd.isna(s):
            return s
        s = str(s).lower().strip()
        s = s.replace("^", " ")
        s = s.replace("increases", "increase")
        s = s.replace("decreases", "decrease")
        s = s.replace("affects", "affect")
        s = " ".join(s.split())
        return s

    manual["relationship_norm"] = manual["relationship"].apply(normalize_relationship)
    df["relationship_norm"] = df["relationship"].apply(normalize_relationship)

    # --------------------------------------------------
    # 6️⃣ Normalize gene columns
    # --------------------------------------------------
    manual["geneSymbol"] = manual["geneSymbol"].astype(str).str.upper().str.strip()
    df["geneSymbol"] = df["geneSymbol"].astype(str).str.upper().str.strip().str.replace(r"\..+$", "", regex=True)

    manual["geneID"] = pd.to_numeric(manual["geneID"], errors="coerce").astype("Int64")
    df["geneID"] = pd.to_numeric(df["geneID"], errors="coerce").astype("Int64")

    if "geneName" not in manual.columns:
        manual["geneName"] = pd.NA
    if "geneName" not in df.columns:
        df["geneName"] = pd.NA

    # --------------------------------------------------
    # 7️⃣ Hierarchical matching keys
    # geneID > geneSymbol > geneName
    # --------------------------------------------------
    merge_keys_hierarchy = [
        ["pmid", "chemicalID", "relationship_norm", "geneID"],
        ["pmid", "chemicalID", "relationship_norm", "geneSymbol"],
        ["pmid", "chemicalID", "relationship_norm", "geneName"]
    ]

    matched_list = []
    unmatched_manual = manual.copy()

    for keys in merge_keys_hierarchy:
        if len(unmatched_manual) == 0:
            break

        keys_to_use = [k for k in keys if k in df.columns and k in unmatched_manual.columns]
        if len(keys_to_use) < 4:
            continue

        matched_step = unmatched_manual.merge(
            df,
            on=keys_to_use,
            how="inner",
            suffixes=("_manual", "_df")
        )

        if len(matched_step) > 0:
            matched_list.append(matched_step)
            matched_keys = matched_step[keys_to_use].drop_duplicates()
            unmatched_manual = pd.merge(
                unmatched_manual,
                matched_keys,
                on=keys_to_use,
                how="left",
                indicator=True
            )
            unmatched_manual = unmatched_manual[unmatched_manual["_merge"] == "left_only"]
            unmatched_manual = unmatched_manual.drop(columns="_merge")

    if len(matched_list) > 0:
        matched = pd.concat(matched_list, ignore_index=True)
    else:
        matched = pd.DataFrame(columns=manual.columns.tolist() + df.columns.tolist())

    # ==================================================
    # 📊 STATISTICS
    # ==================================================

    print("=" * 60)
    print("HIERARCHICAL MATCHING STATISTICS")
    print("=" * 60)
    print(f"Total matched rows:   {len(matched)}")
    print(f"Unmatched manual rows: {len(unmatched_manual)}")

    # --------------------------------------------------
    # 🔢 Gene Validity — hierarchical: geneID > geneSymbol > geneName
    # Each df row is assessed by the highest-priority field that is
    # populated, and checked against the corresponding manual set.
    # --------------------------------------------------
    valid_gene_ids     = set(manual["geneID"].dropna().unique())
    valid_gene_symbols = set(
        manual["geneSymbol"].dropna()
        .loc[lambda s: ~s.isin(["", "NAN", "NA"])]
        .unique()
    )
    valid_gene_names   = set(
        manual["geneName"].dropna()
        .loc[lambda s: ~s.isin(["", "NAN", "NA"])]
        .unique()
    ) if "geneName" in manual.columns else set()

    def classify_gene(row):
        """
        Returns (matched_by, is_valid) using priority: geneID > geneSymbol > geneName.
        matched_by is the field used; is_valid is whether the value is in the manual set.
        """
        gid = row.get("geneID")
        sym = row.get("geneSymbol")
        nam = row.get("geneName")

        if pd.notna(gid):
            return ("geneID", gid in valid_gene_ids)
        if pd.notna(sym) and str(sym) not in ("", "NAN", "NA"):
            return ("geneSymbol", sym in valid_gene_symbols)
        if pd.notna(nam) and str(nam) not in ("", "NAN", "NA"):
            return ("geneName", nam in valid_gene_names)
        return ("none", False)

    gene_classifications = df.apply(classify_gene, axis=1)
    df["_gene_matched_by"] = gene_classifications.apply(lambda x: x[0])
    df["_gene_valid"]      = gene_classifications.apply(lambda x: x[1])

    total_gene_valid   = df["_gene_valid"].sum()
    total_gene_invalid = (~df["_gene_valid"] & (df["_gene_matched_by"] != "none")).sum()
    total_gene_missing = (df["_gene_matched_by"] == "none").sum()

    print(f"\n--- Gene Validity (hierarchical: geneID > geneSymbol > geneName) ---")
    print(f"  Valid genes   (found in manual): {total_gene_valid}")
    print(f"  Invalid genes (not in manual):   {total_gene_invalid}")
    print(f"  Rows with no gene info:          {total_gene_missing}")
    print(f"  Total df rows:                   {len(df)}")

    print(f"\n  Breakdown by matching field used:")
    for field in ["geneID", "geneSymbol", "geneName", "none"]:
        field_mask  = df["_gene_matched_by"] == field
        field_valid = (df["_gene_valid"] & field_mask).sum()
        field_invalid = (~df["_gene_valid"] & field_mask & (df["_gene_matched_by"] != "none")).sum()
        if field == "none":
            print(f"    {field:<12}: {field_mask.sum()} rows (no gene info)")
        else:
            print(f"    {field:<12}: {field_valid} valid, {field_invalid} invalid")

    df.drop(columns=["_gene_matched_by", "_gene_valid"], inplace=True)

    # --------------------------------------------------
    # 🧪 Chemical ID Validity
    # --------------------------------------------------
    valid_chemical_ids = set(
        manual["chemicalID"].dropna()
        .loc[lambda s: ~s.isin(["", "NAN", "NA"])]
        .unique()
    )

    df_chem          = df["chemicalID"].astype(str).str.upper().str.strip()
    df_has_chem      = df_chem.notna() & ~df_chem.isin(["", "NAN", "NA"])
    df["_chem_valid"]   = df_has_chem & df_chem.isin(valid_chemical_ids)
    df["_chem_invalid"] = df_has_chem & ~df_chem.isin(valid_chemical_ids)

    valid_chem_count   = df["_chem_valid"].sum()
    invalid_chem_count = df["_chem_invalid"].sum()
    no_chem_count      = (~df_has_chem).sum()

    print(f"\n--- Chemical ID Validity (df vs. manual) ---")
    print(f"  Valid chemical IDs   (found in manual): {valid_chem_count}")
    print(f"  Invalid chemical IDs (not in manual):   {invalid_chem_count}")
    print(f"  Rows with no chemical ID:               {no_chem_count}")
    print(f"  Total df rows:                          {len(df)}")

    df.drop(columns=["_chem_valid", "_chem_invalid"], inplace=True)

    # --------------------------------------------------
    # 🔗 Relationship Error Analysis on Valid Chem-Gene Pairs
    #
    # A pair is "valid" if (pmid, chemicalID, gene_identifier) exists
    # in manual using the same geneID > geneSymbol > geneName priority.
    # We then classify relationship mismatches by splitting relationship_norm
    # into PREFIX (first token) and TYPE (remainder).
    # --------------------------------------------------

    def split_relationship(rel):
        """Return (prefix, rel_type) for a normalized relationship string."""
        if pd.isna(rel):
            return (pd.NA, pd.NA)
        parts = str(rel).strip().split(" ", 1)
        prefix   = parts[0] if len(parts) >= 1 else pd.NA
        rel_type = parts[1] if len(parts) == 2 else pd.NA
        return (prefix, rel_type)

    # Build manual pair lookup for each gene identifier level
    # Result: (pmid, chemicalID, <gene_field>) -> set of relationship_norm
    def build_pair_rel_lookup(gene_field):
        return (
            manual[manual[gene_field].notna()]
            .groupby(["pmid", "chemicalID", gene_field])["relationship_norm"]
            .apply(set)
            .reset_index()
            .rename(columns={"relationship_norm": "manual_rels"})
        )

    lookup_geneID     = build_pair_rel_lookup("geneID")
    lookup_geneSymbol = build_pair_rel_lookup("geneSymbol")
    lookup_geneName   = build_pair_rel_lookup("geneName")

    # For each df row, find the manual_rels set using the same priority
    def get_manual_rels(row):
        gid = row.get("geneID")
        sym = row.get("geneSymbol")
        nam = row.get("geneName")

        if pd.notna(gid):
            match = lookup_geneID[
                (lookup_geneID["pmid"] == row["pmid"]) &
                (lookup_geneID["chemicalID"] == row["chemicalID"]) &
                (lookup_geneID["geneID"] == gid)
            ]
            if len(match) > 0:
                return match.iloc[0]["manual_rels"]

        if pd.notna(sym) and str(sym) not in ("", "NAN", "NA"):
            match = lookup_geneSymbol[
                (lookup_geneSymbol["pmid"] == row["pmid"]) &
                (lookup_geneSymbol["chemicalID"] == row["chemicalID"]) &
                (lookup_geneSymbol["geneSymbol"] == sym)
            ]
            if len(match) > 0:
                return match.iloc[0]["manual_rels"]

        if pd.notna(nam) and str(nam) not in ("", "NAN", "NA"):
            match = lookup_geneName[
                (lookup_geneName["pmid"] == row["pmid"]) &
                (lookup_geneName["chemicalID"] == row["chemicalID"]) &
                (lookup_geneName["geneName"] == nam)
            ]
            if len(match) > 0:
                return match.iloc[0]["manual_rels"]

        return None

    df["_manual_rels"] = df.apply(get_manual_rels, axis=1)

    has_valid_pair = df["_manual_rels"].notna()
    exact_match    = has_valid_pair & df.apply(
        lambda r: r["relationship_norm"] in r["_manual_rels"] if r["_manual_rels"] is not None else False,
        axis=1
    )
    wrong_rel = has_valid_pair & ~exact_match

    # Classify mismatches
    mismatch_df = df[wrong_rel].copy()
    mismatch_df[["df_prefix", "df_type"]] = pd.DataFrame(
        mismatch_df["relationship_norm"].apply(split_relationship).tolist(),
        index=mismatch_df.index
    )

    mismatch_expanded = mismatch_df.explode("_manual_rels").copy()
    mismatch_expanded[["manual_prefix", "manual_type"]] = pd.DataFrame(
        mismatch_expanded["_manual_rels"].apply(split_relationship).tolist(),
        index=mismatch_expanded.index
    )

    def classify_mismatch(group):
        """
        For one df row (multiple manual_rels rows after explode), determine
        the least severe error category achievable across all manual candidates.

        Priority (least → most severe):
          wrong_type_only       — prefix matches at least one manual rel
          wrong_prefix_only     — type   matches at least one manual rel
          wrong_prefix_and_type — neither matches any manual rel
        """
        df_prefix = group["df_prefix"].iloc[0]
        df_type   = group["df_type"].iloc[0]

        prefix_matches = (group["manual_prefix"] == df_prefix).any()
        type_matches   = (group["manual_type"]   == df_type  ).any()

        if prefix_matches and not type_matches:
            return "wrong_type_only"
        elif type_matches and not prefix_matches:
            return "wrong_prefix_only"
        elif not prefix_matches and not type_matches:
            return "wrong_prefix_and_type"
        else:
            return "wrong_prefix_and_type"

    if len(mismatch_expanded) > 0:
        mismatch_categories = (
            mismatch_expanded
            .groupby(mismatch_expanded.index)
            .apply(classify_mismatch)
            .rename("error_category")
        )
        category_counts = mismatch_categories.value_counts()
    else:
        category_counts = pd.Series(dtype=int)

    wrong_prefix_only     = category_counts.get("wrong_prefix_only",     0)
    wrong_type_only       = category_counts.get("wrong_type_only",       0)
    wrong_prefix_and_type = category_counts.get("wrong_prefix_and_type", 0)
    wrong_combination     = category_counts.get("wrong_combination",     0)

    print(f"\n--- Relationship Error Analysis on Valid Chem-Gene Pairs ---")
    print(f"  DF rows with a valid (pmid, chemicalID, gene) pair: {has_valid_pair.sum()}")
    print(f"  Exact relationship match:                           {exact_match.sum()}")
    print(f"  Wrong relationship (any mismatch):                  {wrong_rel.sum()}")
    print(f"    ├─ Wrong prefix only  (correct type):             {wrong_prefix_only}")
    print(f"    ├─ Wrong type only    (correct prefix):           {wrong_type_only}")
    print(f"    ├─ Wrong prefix AND type:                         {wrong_prefix_and_type}")
    # if wrong_combination:
    #     print(f"    └─ Ambiguous (components match separately):       {wrong_combination}")

    df.drop(columns=["_manual_rels"], inplace=True)

    print("=" * 60 + "\n")

    return matched, unmatched_manual

In [12]:
manual = pd.read_csv('/Users/sc/nlp_4_ctd_code/data_json_clover/manual_annotations/CTD.csv')

output_cols = [
    'deepseek_E_output',
    'gpt_E_output',
    'claude_E_output',
    'gpt4o_E_output',
    
    'deepseekV_gptE_output',
    'deepseekV_gpt4oE_output',
    'deepseekV_claudeE_output',
    
    'gptV_deepseekE_output',
    'gptV_claudeE_output',
    
    'claudeV_deepseekE_output',
    'claudeV_gptE_output',
    'claudeV_gpt4oE_output',
    
    'gpt4oV_deepseekE_output',
    'gpt4oV_claudeE_output'
]

print('SECTION')
for col in output_cols:
    print(f'{col}')
    df = pd.read_csv(f'{output_data_path}/temp_v1/raw/{col}.csv')

    # Define columns to check for duplicates, using whichever exists
    duplicate_cols = ['pmid', 'chemicalName', 'relationship']
    
    # Choose the first available gene identifier
    for col in ['geneID', 'geneSymbol', 'geneName']:
        if col in df.columns:
            duplicate_cols.append(col)
            break
    
    # Drop duplicates based on these columns
    # print(len(df))
    df = df.drop_duplicates(subset=duplicate_cols)
    # print(len(df))
    
    matched, unmatched_manual = match_manual_hierarchical(manual, df)
    print(f'Precision: {len(matched)/len(df)}')
    print(f'Recall: {len(matched)/609}')

    print()

SECTION
deepseek_E_output
HIERARCHICAL MATCHING STATISTICS
Total matched rows:   110
Unmatched manual rows: 543

--- Gene Validity (hierarchical: geneID > geneSymbol > geneName) ---
  Valid genes   (found in manual): 738
  Invalid genes (not in manual):   879
  Rows with no gene info:          1
  Total df rows:                   1618

  Breakdown by matching field used:
    geneID      : 736 valid, 782 invalid
    geneSymbol  : 2 valid, 0 invalid
    geneName    : 0 valid, 97 invalid
    none        : 1 rows (no gene info)

--- Chemical ID Validity (df vs. manual) ---
  Valid chemical IDs   (found in manual): 731
  Invalid chemical IDs (not in manual):   884
  Rows with no chemical ID:               3
  Total df rows:                          1618

--- Relationship Error Analysis on Valid Chem-Gene Pairs ---
  DF rows with a valid (pmid, chemicalID, gene) pair: 245
  Exact relationship match:                           102
  Wrong relationship (any mismatch):                  143
    ├